# GLM-5.3-Flash-Uncensored-GSQ-RCO-Q4 · Colab G4 320B-A18B 混合注意力与消融破限主力服务（已合入 GCSA-AiLab LoRA-v2）

- **模型别名**：`GLM-5.3-Flash-Uncensored-GSQ-RCO-Q4`（262,144 上下文 / 65,536 最大输出 / 16–64 GiB Prompt Cache + 槽位持久化）
- **内存回收**：权重进显存后删掉内存盘里只供显存用的分片副本（生产模式在后台做），空出的内存一半给 Prompt Cache；服务不重启就不用重新下载。
- **启动**：首格 `DEBUG_VERIFY = False`（调试新模型/改配置时设 True）。生产模式在 `/health` 返回 200 后立刻出就绪横幅；CUDA 预热、内存盘副本释放和 Tailscale 收尾放到后台。调试模式才做 /props 四档门禁、阻塞预热和额外大小核对。不跑拼写探针，也不做 SHA-256。
- **热切换**：“热切换模型”格不释放 G4 就换跑另一个已登记模型，省掉重新分配运行时和装 llama.cpp / Tailscale 的时间。
- **采样参数**：`--temp 1.0 --top-p 0.95 --top-k 0 --min-p 0.0`（按模型卡官方默认值）
- **WebUI 持久化与本机文件直通**：服务端内置 `--ui-config-file` 与 `--ui-mcp-proxy`，自动加载小说共创 System Prompt、关闭后台 LLM 标题生成防缓存冲刷，并通过 Tailscale SOCKS5 隧道直连本机 `~/Documents/Colab创作`（`local_novel_fs` 工具可在 WebUI 输入框一键开关）。
- **固定入口（所有笔记本分时复用 `svc:qwen`）**：
  - 网页：`http://qwen.<YOUR_TAILNET>.ts.net:443/`（首格就绪自动弹出，点首格停机按钮自动关闭）
  - 接口：`http://qwen.<YOUR_TAILNET>.ts.net:443/v1`
  - 运行时目录：`/content/glm53-flash`


In [ ]:
# 一键启动（由 colab_kernel.py 统一内核生成）：tmpfs 内存盘 + 32路下载 + 字节大小校验 + WebUI 本机文件桥 + Tailscale
# 调试新模型/改配置时设 True
DEBUG_VERIFY = False
import concurrent.futures, hashlib, json, os, shutil, socket, struct, subprocess, sys, threading, time
from pathlib import Path
import requests
from requests.adapters import HTTPAdapter

t_boot = time.monotonic()
SPEC = json.loads("{\n  \"slug\": \"glm53-flash\",\n  \"alias\": \"GLM-5.3-Flash-Uncensored-GSQ-RCO-Q4\",\n  \"root_dir\": \"/content/glm53-flash\",\n  \"ts_hostname\": \"glm53-flash-colab\",\n  \"min_vram_mib\": 85000,\n  \"ram_reserve_gib\": 65,\n  \"ple_shard_name\": null,\n  \"template_family\": \"glm5_next\",\n  \"temp\": \"1.0\",\n  \"top_p\": \"0.95\",\n  \"top_k\": \"0\",\n  \"min_p\": \"0.0\",\n  \"reasoning_effort_default\": \"high\",\n  \"effort_prompts\": {\n    \"xhigh\": \"\u7528\u4e2d\u6587\u601d\u8003\uff0c\u5c3d\u91cf\u6df1\u5165\u3002\",\n    \"high\": \"\u7528\u4e2d\u6587\u601d\u8003\u3002\",\n    \"medium\": \"\u7528\u4e2d\u6587\u601d\u8003\uff0c\u7b80\u77ed\u4e9b\u3002\",\n    \"low\": \"\u7528\u4e2d\u6587\u601d\u8003\uff0c\u8d8a\u77ed\u8d8a\u597d\u3002\"\n  },\n  \"mtp_n_max\": null,\n  \"mtp_p_min\": null,\n  \"extra_server_flags\": [\n    \"--n-cpu-moe\",\n    \"19\"\n  ],\n  \"files\": [\n    {\n      \"role\": \"model\",\n      \"repo\": \"GCSA-AiLab/GLM-5.3-Flash-Uncensored-RCO-GSQ-GGUF\",\n      \"rev\": \"1827843362350fbc87ee6b478d9dd09e33d9fb50\",\n      \"path\": \"Q4/GLM-5.3-Flash-Uncensored-GSQ-RCO-Q4.gguf\",\n      \"local_subdir\": \"Q4\",\n      \"size\": 137073273280,\n      \"priority\": 2\n    },\n    {\n      \"role\": \"mmproj\",\n      \"repo\": \"unsloth/GLM-5.3-Flash-GGUF\",\n      \"rev\": \"a38483c8cd5df544f53d70fb281afe97369d5ab6\",\n      \"path\": \"mmproj-BF16.gguf\",\n      \"size\": 1164010176,\n      \"priority\": 1\n    }\n  ]\n}")
UI_CFG_DATA = json.loads("{\n  \"systemMessage\": \"\u4f60\u662f\u4e2d\u6587\u957f\u7bc7\u5c0f\u8bf4\u5171\u521b\u642d\u6863\u3002\\n1. \u5f00\u4e86 local_novel_fs \u65f6\uff0c\u52a8\u7b14\u524d\u5148\u8bfb \u5199\u4f5c\u8981\u6c42.md \u548c\u6b63\u6587\u672b\u5c3e\uff08tail_lines=150\uff09\u3002\u65b0\u7ae0\u6216\u4e0b\u534a\u7ae0\u7528 write_novel_file \u8ffd\u52a0\uff1b\u91cd\u5199\u4e0a\u4e00\u8f6e\u7528 replace_last_write\uff0c\u91cd\u5199\u6574\u7ae0\u7528 replace_last_chapter\uff1b\u5c0f\u6539\u7528 edit_novel_file\uff1b\u64a4\u56de\u7528 undo_novel_write\u3002\u6ca1\u5f00\u5de5\u5177\u5c31\u76f4\u63a5\u5728\u5bf9\u8bdd\u91cc\u5199\u3002\\n2. \u67e5\u539f\u4f5c\u7528 grep_reference\uff1b\u8bbe\u5b9a\u51b2\u7a81\u4ee5\u7528\u6237\u5f53\u524d\u6307\u4ee4\u4e3a\u51c6\u3002\\n3. \u7528\u6237\u8bf4\u300c\u6536\u5de5\u300d\u65f6\uff0c\u7528 edit_novel_file \u628a\u65b0\u5b9a\u7684\u8bbe\u5b9a\u3001\u8fdb\u5ea6\u548c\u4f0f\u7b14\u66f4\u65b0\u8fdb \u5199\u4f5c\u8981\u6c42.md\uff0c\u53ea\u6539\u53d8\u5316\u9879\u3002\\n4. \u7528\u6237\u70b9\u540d\u65f6\u624d\u8bfb\u6280\u80fd\uff1agrill/\u62f7\u95ee \u8bfb skill:grilling\uff0c\u4f18\u5316\u63d0\u793a\u8bcd \u8bfb skill:prompting\uff0cretro/\u590d\u76d8 \u8bfb skill:retro\u3002\",\n  \"showSystemMessage\": true,\n  \"showMessageStats\": true,\n  \"enableContinueGeneration\": true,\n  \"pasteLongTextToFileLen\": 50000,\n  \"excludeReasoningFromContext\": true,\n  \"renderThinkingAsMarkdown\": true,\n  \"renderUserContentAsMarkdown\": true,\n  \"titleGenerationUseLLM\": false,\n  \"titleGenerationUseFirstLine\": true,\n  \"mcpServers\": \"[{\\\"id\\\": \\\"local-novel-fs\\\", \\\"name\\\": \\\"local_novel_fs\\\", \\\"displayName\\\": \\\"\u672c\u673a\u5c0f\u8bf4\u76ee\u5f55 (Colab\u521b\u4f5c)\\\", \\\"url\\\": \\\"http://127.0.0.1:3426/mcp\\\", \\\"enabled\\\": true, \\\"useProxy\\\": true}]\"\n}")

ROOT = Path(SPEC['root_dir'])
DIST = ROOT / 'llama-dist'
MODEL_ALIAS = SPEC['alias']
KEY_FILE = ROOT / '.llama-api-key'
PORT = 8081
KEY = "<YOUR_LLAMA_API_KEY>"
FILES = SPEC['files']
CPP_TAG = 'b12032'
CPP_URL = 'https://github.com/ggml-org/llama.cpp/releases/download/' + CPP_TAG

def _local_path(f):
    sub = f.get('local_subdir')
    return (ROOT / sub / Path(f['path']).name) if sub else (ROOT / Path(f['path']).name)

MODEL = _local_path(FILES[0])
MMPROJ = next((_local_path(f) for f in FILES if f['role'] == 'mmproj'), None)
DRAFT = next((_local_path(f) for f in FILES if f['role'] == 'draft'), None)
LORA = next((_local_path(f) for f in FILES if f['role'] == 'lora'), None)
# PLE 表所在目录跟着各模型的 local_subdir 走（IQ4_XS / IQ4_NL …），不能写死。
SHARD_DIR = next((_local_path(f).parent for f in FILES if Path(f['path']).name == SPEC.get('ple_shard_name')), ROOT)

# 服务把权重读进显存后，/dev/shm 里只供显存用的副本就是冗余内存；删掉后靠这份记录（绑定服务 PID）在重跑本格时跳过重新下载。
RELEASE_STATE = ROOT / 'released-copies.json'
HEAD_CACHE = ROOT / 'model-head.bin'
TMPFS_PREFIXES = ('/dev/shm/', '/var/colab/')
FILE_NAMES = {_local_path(f).name for f in FILES}

def _server_proc():
    # 本模型的 llama-server：(pid, argv)；没有则 (None, [])
    for procdir in Path('/proc').iterdir():
        if not procdir.name.isdigit():
            continue
        try:
            args = (procdir / 'cmdline').read_bytes().decode(errors='replace').split('\0')
        except (FileNotFoundError, PermissionError, ProcessLookupError):
            continue
        if args and Path(args[0]).name == 'llama-server' and str(MODEL) in args:
            return int(procdir.name), args
    return None, []

def _server_pid():
    return _server_proc()[0]

def _proc_identity(pid):
    # boot_id + 进程启动时刻，防止重启后的同模型服务恰好复用旧 PID 时误用释放记录
    try:
        boot = Path('/proc/sys/kernel/random/boot_id').read_text().strip()
        stat = Path('/proc/%d/stat' % pid).read_text()
        return boot + ':' + stat.rsplit(')', 1)[1].split()[19]
    except (OSError, IndexError, TypeError):
        return None

def _cache_ram_of(args):
    try:
        return int(args[args.index('--cache-ram') + 1])
    except (ValueError, IndexError):
        return None

def _stop_llama_servers():
    for procdir in Path('/proc').iterdir():
        if not procdir.name.isdigit():
            continue
        try:
            args = (procdir / 'cmdline').read_bytes().decode(errors='replace').split('\0')
            if args and Path(args[0]).name == 'llama-server':
                os.kill(int(procdir.name), 15)
                for _ in range(60):
                    stat = (procdir / 'stat').read_text() if procdir.exists() else ''
                    if not stat or stat.split(') ', 1)[1].startswith('Z'):
                        break
                    time.sleep(0.5)
                else:
                    os.kill(int(procdir.name), 9)
                    time.sleep(2)
        except (FileNotFoundError, ProcessLookupError, PermissionError):
            pass
    old_proc = globals().get('qwen_server')
    if old_proc is not None:
        try:
            old_proc.wait(timeout=5)
        except Exception:
            pass
    deadline = time.monotonic() + 20
    while time.monotonic() < deadline:
        alive = []
        for procdir in Path('/proc').iterdir():
            try:
                if procdir.name.isdigit() and Path((procdir / 'cmdline').read_bytes().decode(errors='replace').split('\0')[0]).name == 'llama-server':
                    alive.append(procdir.name)
            except (FileNotFoundError, ProcessLookupError, PermissionError):
                pass
        if not alive:
            return
        time.sleep(1)
    raise RuntimeError('旧 llama-server 停不掉（PID ' + ', '.join(alive) + '），已中止；请用停机按钮释放运行时后重来。')

def _mem_available():
    for l in Path('/proc/meminfo').read_text().splitlines():
        if l.startswith('MemAvailable:'):
            return int(l.split()[1]) * 1024
    return 0

SERVICE = 'svc:qwen'
SERVICE_FQDN = "qwen.<YOUR_TAILNET>.ts.net"
SERVICE_TAG = 'tag:colab'
TS_HOSTNAME = SPEC['ts_hostname']
TS_CLIENT_ID = "<YOUR_TS_OAUTH_CLIENT_ID>"
TS_CLIENT_SECRET = "<YOUR_TS_OAUTH_CLIENT_SECRET>"
TS = '/content/tsbin/tailscale'
TS_SOCK = '/var/run/tailscale/tailscaled.sock'
TS_STATE = Path('/content/tailscaled.state')
TS_DIR = Path('/content/tsstatedir')

def sh(c):
    subprocess.run(c, shell=True, check=True)

def health():
    try:
        return requests.get('http://127.0.0.1:%d/health' % PORT, timeout=2).status_code == 200
    except Exception:
        return False

gpu_info = subprocess.getoutput('nvidia-smi --query-gpu=name,memory.total --format=csv,noheader,nounits').strip()
if not gpu_info or ',' not in gpu_info:
    raise RuntimeError('未检测到 NVIDIA GPU，请在右上角切换到 G4 GPU 运行时。输出：' + gpu_info)
gpu_name, gpu_mib_str = [x.strip() for x in gpu_info.split(',', 1)]
gpu_mib = int(gpu_mib_str)
print(f'检测到 GPU: {gpu_name} ({gpu_mib / 1024:.1f} GiB VRAM)', flush=True)
if gpu_mib < SPEC['min_vram_mib']:
    raise RuntimeError(f'当前 GPU 显存仅 {gpu_mib / 1024:.1f} GiB（{gpu_name}），低于要求的 {SPEC["min_vram_mib"] / 1024:.0f} GiB，请切换到 G4 高内存运行时。')

ROOT.mkdir(parents=True, exist_ok=True)
for _d in sorted(set(_local_path(f).parent for f in FILES)):
    _d.mkdir(parents=True, exist_ok=True)
DIST.mkdir(parents=True, exist_ok=True)
KEY_FILE.write_text(KEY + '\n', encoding='utf-8', newline='')
KEY_FILE.chmod(0o600)
AUTH = {'Authorization': 'Bearer ' + KEY}

def prepare_llama_cpp():
    if next(DIST.rglob('llama-server'), None) is not None:
        return
    print(f'[并行任务] 下载并解压 llama.cpp {CPP_TAG} (CUDA 12.8) ...', flush=True)
    sh(f'cd {DIST} && '
       f'curl -fsSL -O {CPP_URL}/llama-{CPP_TAG}-bin-ubuntu-cuda-12.8-x64.tar.gz && '
       f'curl -fsSL -O {CPP_URL}/cudart-llama-{CPP_TAG}-bin-ubuntu-cuda-12.8-x64.tar.gz && '
       f'tar -xzf llama-{CPP_TAG}-bin-ubuntu-cuda-12.8-x64.tar.gz && '
       f'tar -xzf cudart-llama-{CPP_TAG}-bin-ubuntu-cuda-12.8-x64.tar.gz && '
       f'rm -f *.tar.gz')
    print(f'[并行任务] llama.cpp {CPP_TAG} 就绪', flush=True)

def ts(*a, timeout=60):
    return subprocess.run([TS, '--socket=' + TS_SOCK, *a],
                          stdin=subprocess.DEVNULL, capture_output=True, text=True, timeout=timeout)

def ts_state():
    try:
        return json.loads(ts('status', '--json').stdout).get('BackendState', '')
    except Exception:
        return ''

def _start_mcp_bridge():
    srv = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    srv.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
    try:
        srv.bind(('127.0.0.1', 3426))
    except OSError:
        return
    srv.listen(32)
    def _pipe(a, b):
        try:
            while (d := a.recv(65536)):
                b.sendall(d)
        except Exception:
            pass
        finally:
            try: a.close()
            except Exception: pass
            try: b.close()
            except Exception: pass
    while True:
        try:
            cli, _ = srv.accept()
            up = socket.create_connection(('127.0.0.1', 1055), timeout=10)
            up.sendall(bytes([5, 1, 0]))
            assert up.recv(2) == bytes([5, 0])
            up.sendall(bytes([5, 1, 0, 1]) + socket.inet_aton("<YOUR_LOCAL_TAILSCALE_IP>") + struct.pack('>H', 3426))
            assert up.recv(10)[1] == 0
            threading.Thread(target=_pipe, args=(cli, up), daemon=True).start()
            threading.Thread(target=_pipe, args=(up, cli), daemon=True).start()
        except Exception:
            pass

def prepare_tailscale():
    if not Path(TS).exists():
        sh('curl -fsSL https://pkgs.tailscale.com/stable/tailscale_1.104.1_amd64.tgz | tar -xz -C /content && '
           'mv /content/tailscale_1.104.1_amd64 /content/tsbin')
    TS_DIR.mkdir(parents=True, exist_ok=True)
    Path('/var/run/tailscale').mkdir(parents=True, exist_ok=True)
    if ts_state() in ('', 'NoState', 'Stopped'):
        subprocess.Popen(
            ['/content/tsbin/tailscaled', '--tun=userspace-networking',
             '--socks5-server=localhost:1055', '--state=' + str(TS_STATE),
             '--statedir=' + str(TS_DIR), '--socket=' + TS_SOCK],
            stdout=open('/content/tailscaled.log', 'a', encoding='utf-8', newline=''),
            stderr=subprocess.STDOUT, stdin=subprocess.DEVNULL, start_new_session=True,
        )
        for _ in range(40):
            if Path(TS_SOCK).exists():
                break
            time.sleep(0.5)
        for _ in range(40):
            if ts_state() in ('Running', 'NeedsLogin'):
                break
            time.sleep(0.5)
    if ts_state() != 'Running':
        token = requests.post(
            'https://api.tailscale.com/api/v2/oauth/token',
            data={'client_id': TS_CLIENT_ID, 'client_secret': TS_CLIENT_SECRET}, timeout=30
        ).json()['access_token']
        key = requests.post(
            'https://api.tailscale.com/api/v2/tailnet/-/keys',
            headers={'Authorization': 'Bearer ' + token},
            json={'capabilities': {'devices': {'create': {
                'reusable': False, 'ephemeral': True, 'preauthorized': True, 'tags': [SERVICE_TAG]
            }}}}, timeout=30
        ).json()['key']
        ts('up', '--auth-key=' + key, '--hostname=' + TS_HOSTNAME,
           '--advertise-tags=' + SERVICE_TAG, '--accept-dns=false', '--timeout=60s', timeout=120)
        for _ in range(120):
            if ts_state() == 'Running':
                break
            time.sleep(0.5)
    ts('serve', 'reset')
    ts('serve', '--bg', '--service=' + SERVICE, '--tcp=443', 'tcp://127.0.0.1:%d' % PORT, timeout=90)
    ts('serve', '--bg', '--https=443', 'http://127.0.0.1:%d' % PORT, timeout=90)
    threading.Thread(target=_start_mcp_bridge, daemon=True).start()
    print(f'[并行任务] Tailscale Service ({SERVICE_FQDN}) 与本机小说文件 MCP 隧道已就绪', flush=True)

bg_executor = concurrent.futures.ThreadPoolExecutor(max_workers=2)
fut_cpp = bg_executor.submit(prepare_llama_cpp)
fut_ts = bg_executor.submit(prepare_tailscale)

subprocess.run(['mount', '-o', 'remount,size=115G', '/dev/shm'], capture_output=True)
tmpfs_dirs = []
for mline in Path('/proc/mounts').read_text().splitlines():
    mparts = mline.split()
    if len(mparts) >= 3 and mparts[2] == 'tmpfs' and mparts[1] in ('/dev/shm', '/var/colab'):
        tmpfs_dirs.append(Path(mparts[1]))
mp_free = {mp: shutil.disk_usage(mp).free for mp in tmpfs_dirs}
meminfo_txt = Path('/proc/meminfo').read_text()
avail_kb = next((int(l.split()[1]) for l in meminfo_txt.splitlines() if l.startswith('MemAvailable:')), 0)
ram_budget = max(0, avail_kb * 1024 - SPEC['ram_reserve_gib'] * (1024**3))

released_reuse = set()
if RELEASE_STATE.exists():
    try:
        rel_state = json.loads(RELEASE_STATE.read_text(encoding='utf-8'))
        rel_ok = (isinstance(rel_state, dict) and isinstance(rel_state.get('pid'), int)
                  and isinstance(rel_state.get('files'), list)
                  and all(isinstance(n, str) and n in FILE_NAMES for n in rel_state['files']))
    except Exception:
        rel_state, rel_ok = {}, False
    pid_now = _server_pid()
    if rel_ok and pid_now == rel_state['pid'] and rel_state.get('proc_id') and _proc_identity(pid_now) == rel_state['proc_id']:
        released_reuse = set(rel_state['files'])
        print(f'服务仍在运行：{len(released_reuse)} 个已释放内存副本的分片无需重新下载', flush=True)
    else:
        if not rel_ok:
            print('⚠️ released-copies.json 无法识别，缺失的分片会重新下载', flush=True)
        RELEASE_STATE.unlink(missing_ok=True)
if not released_reuse and _server_pid() and any(not _local_path(f).exists() for f in FILES):
    # 记录不可用、服务还在且文件已删：先停服务再补下载，避免服务占着内存时又往内存盘塞几十 GiB。
    print('⚠️ 服务在运行，但分片副本已删且释放记录不可用：先停掉旧服务，再重新下载。', flush=True)
    _stop_llama_servers()

sorted_files = sorted(FILES, key=lambda x: x.get('priority', 2))
all_targets = [
    ('https://huggingface.co/' + f['repo'] + '/resolve/' + f['rev'] + '/' + f['path'],
     _local_path(f), f['size'])
    for f in sorted_files
]
downloads = []
tmpfs_bytes = 0
disk_bytes = 0
for orig_url, logical_path, size in all_targets:
    if logical_path.name in released_reuse:
        continue
    if logical_path.exists() and logical_path.stat().st_size == size:
        continue
    if logical_path.is_symlink() or logical_path.exists():
        logical_path.unlink(missing_ok=True)
    real_path = logical_path
    if ram_budget >= size:
        for mp in tmpfs_dirs:
            if mp_free.get(mp, 0) >= size + 4 * (1024**3):
                cand = mp / SPEC['slug'] / logical_path.name
                cand.parent.mkdir(parents=True, exist_ok=True)
                # 只看大小会把 truncate 出来的稀疏占位文件（下载中断、上次报错）当成已下载；tmpfs 的 st_blocks 是实际占用。
                if cand.exists() and cand.stat().st_size == size and cand.stat().st_blocks * 512 >= size:
                    logical_path.symlink_to(cand)
                    mp_free[mp] -= size
                    ram_budget -= size
                    real_path = None
                    break
                cand.unlink(missing_ok=True)
                with cand.open('wb') as f:
                    f.truncate(size)
                logical_path.symlink_to(cand)
                mp_free[mp] -= size
                ram_budget -= size
                tmpfs_bytes += size
                real_path = cand
                break
    if real_path is None:
        continue
    if real_path == logical_path:
        free_disk = shutil.disk_usage('/content').free
        if free_disk < disk_bytes + size + 6 * (1024**3):
            raise RuntimeError('本地 /content 与 tmpfs 空间不足，请重建运行时。')
        with real_path.open('wb') as f:
            f.truncate(size)
        disk_bytes += size
    downloads.append((orig_url, str(real_path), size))

freshly_downloaded = bool(downloads)
if downloads:
    need_bytes = sum(size for _, _, size in downloads)
    print(f'待并发下载: {need_bytes / (1024**3):.2f} GiB ({len(downloads)} 个文件) → DDR5 内存盘 tmpfs: {tmpfs_bytes / (1024**3):.1f} GiB | 本地磁盘: {disk_bytes / (1024**3):.1f} GiB', flush=True)
    CHUNK = 64 * 1024 * 1024
    hf_tok = os.environ.get('HF_TOKEN', '').strip()
    if not hf_tok:
        try:
            from google.colab import userdata
            hf_tok = (userdata.get('HF_TOKEN') or '').strip()
        except Exception:
            hf_tok = ''

    def _resolve_cdn(item):
        from email.utils import parsedate_to_datetime
        from datetime import datetime, timezone
        orig_url, real_path, size = item
        req_hdr = {'Range': 'bytes=0-0'}
        if hf_tok:
            req_hdr['Authorization'] = 'Bearer ' + hf_tok
        for attempt in range(5):
            try:
                with requests.get(orig_url, headers=req_hdr, stream=True, timeout=30) as r:
                    if r.status_code == 429:
                        ra = r.headers.get('Retry-After', '')
                        try:
                            wait_s = float(ra)
                        except ValueError:
                            try:
                                wait_s = max(1.0, (parsedate_to_datetime(ra) - datetime.now(timezone.utc)).total_seconds())
                            except Exception:
                                wait_s = 2 ** (attempt + 1)
                        time.sleep(min(wait_s, 60.0))
                        continue
                    r.raise_for_status()
                    if r.status_code != 206:
                        raise RuntimeError(f'Range probe expected 206, got {r.status_code}')
                    cdn_url = r.url
                    return [(cdn_url, orig_url, real_path, start, min(start + CHUNK, size) - 1)
                            for start in range(0, size, CHUNK)]
            except Exception:
                if attempt == 4:
                    raise
                time.sleep(2 ** (attempt + 1))

    range_tasks = []
    with concurrent.futures.ThreadPoolExecutor(max_workers=len(downloads)) as rpool:
        for chunk_list in rpool.map(_resolve_cdn, downloads):
            range_tasks.extend(chunk_list)

    tasks_file = Path('/tmp/colab_kernel_dl_tasks.json')
    tasks_file.write_text(json.dumps({'need_bytes': need_bytes, 'tasks': range_tasks}), encoding='utf-8', newline='')
    dl_script = Path('/tmp/colab_kernel_fast_dl.py')
    dl_script.write_text('''import concurrent.futures, json, multiprocessing as mp, os, sys, time
import requests
from requests.adapters import HTTPAdapter

data = json.loads(open(sys.argv[1], encoding="utf-8").read())
need_bytes = data["need_bytes"]
tasks = data["tasks"]
N_PROC = 8
N_THREADS = 4

done_bytes = mp.Value("q", 0)
t_start = time.monotonic()
last_print = mp.Value("d", t_start)

def worker_batch(batch):
    sess = requests.Session()
    adapter = HTTPAdapter(pool_connections=N_THREADS + 4, pool_maxsize=N_THREADS + 4)
    sess.mount("https://", adapter)
    sess.mount("http://", adapter)

    def fetch_one(task):
        cdn_url, orig_url, target, start, end = task
        expected = end - start + 1
        for attempt in range(5):
            url = cdn_url if attempt < 2 else orig_url
            try:
                with sess.get(url, headers={"Range": f"bytes={start}-{end}"}, stream=True, timeout=(15, 120)) as resp:
                    if resp.status_code != 206:
                        raise RuntimeError(f"HTTP {resp.status_code}")
                    fd = os.open(target, os.O_RDWR)
                    try:
                        pos = start
                        for blk in resp.iter_content(8 * 1024 * 1024):
                            if blk:
                                os.pwrite(fd, blk, pos)
                                pos += len(blk)
                    finally:
                        os.close(fd)
                    if pos - start != expected:
                        raise RuntimeError(f"short read {pos - start} != {expected}")
                with done_bytes.get_lock():
                    done_bytes.value += expected
                    now = time.monotonic()
                    if now - last_print.value >= 8:
                        rate = done_bytes.value / max(now - t_start, 0.1) / (1024**2)
                        print(f"  并发下载进度: {done_bytes.value / need_bytes:.0%} ({done_bytes.value / (1024**3):.1f} / {need_bytes / (1024**3):.1f} GiB)，均速 {rate:.0f} MB/s", flush=True)
                        last_print.value = now
                return
            except Exception as err:
                if attempt == 4:
                    raise RuntimeError(f"{os.path.basename(target)} [{start}-{end}] 失败: {err}") from err
                time.sleep(1.5 * (attempt + 1))

    with concurrent.futures.ThreadPoolExecutor(max_workers=N_THREADS) as tpool:
        futs = [tpool.submit(fetch_one, t) for t in batch]
        for f in concurrent.futures.as_completed(futs):
            f.result()

if __name__ == "__main__":
    batches = [tasks[i::N_PROC] for i in range(N_PROC) if tasks[i::N_PROC]]
    ctx = mp.get_context("fork")
    with ctx.Pool(processes=len(batches)) as pool:
        pool.map(worker_batch, batches)
''', encoding='utf-8', newline='')
    t_dl_start = time.monotonic()
    subprocess.run([sys.executable, str(dl_script), str(tasks_file)], check=True)
    dl_sec = max(time.monotonic() - t_dl_start, 0.1)
    print(f'全部权重下载完成：用时 {dl_sec:.1f}s，平均 {need_bytes / (1024**2) / dl_sec:.0f} MB/s', flush=True)
else:
    print(f'全部 {len(FILES)} 个权重与附件已存在于内存盘/本地，跳过下载。', flush=True)

def verify_hashes():
    # 仅调试模式调用。下载决策时已经对过一次字节大小；这里再 stat 一遍。
    try:
        for item in FILES:
            path = _local_path(item)
            if path.name in released_reuse:
                continue
            actual = path.stat().st_size if path.exists() else -1
            if actual != item['size']:
                if path.exists():
                    path.resolve().unlink(missing_ok=True)
                    path.unlink(missing_ok=True)
                raise RuntimeError('权重存在性或字节大小校验失败，请重新运行首格：' + path.name)
            print('SIZE OK:', path.name, item['size'], flush=True)
    except BaseException:
        if globals().get('qwen_server') is not None:
            qwen_server.terminate()
        raise

PARALLEL_SLOTS = 1
SLOT_CONTEXT_TOKENS = 262144
TOTAL_CONTEXT_TOKENS = SLOT_CONTEXT_TOKENS * PARALLEL_SLOTS
UBATCH_SIZE = 2048
MAX_OUTPUT_TOKENS = 65536
SLOT_SAVE_DIR = ROOT / 'slot-saves'
SLOT_SAVE_DIR.mkdir(parents=True, exist_ok=True)
CHAT_TMPL = ROOT / 'chat-template-clean.jinja'
UI_CFG = ROOT / 'webui-config.json'
new_ui_text = json.dumps(UI_CFG_DATA, ensure_ascii=False, indent=2) + '\n'
old_ui_on_disk = UI_CFG.read_text(encoding='utf-8') if UI_CFG.exists() else ''

ple_name = SPEC.get('ple_shard_name')
if ple_name:
    ple_real = str((SHARD_DIR / ple_name).resolve())
    ple_in_tmpfs = ple_real.startswith(TMPFS_PREFIXES)
else:
    ple_in_tmpfs = False

# PLE 表分片在运行期由 CPU 直接 mmap 读取，必须常驻；其余落在内存盘的分片加载后可删。
release_keep = {ple_name} if ple_in_tmpfs else set()
release_candidates = [f for f in FILES
                      if _local_path(f).name not in release_keep
                      and _local_path(f).exists()
                      and str(_local_path(f).resolve()).startswith(TMPFS_PREFIXES)]
releasable_bytes = sum(f['size'] for f in release_candidates)
# 只用于新启动的服务；已在运行的服务保留自己的 --cache-ram（configured_server 不比较这一项，免得为改缓存上限重启）。
# 只拿回收内存的一半给提示缓存，另一半留作余量（mmap 模式下 CPU 张量所在分片删文件后仍占内存）。
PROMPT_CACHE_MIB = min(65536, 16384 + releasable_bytes // (2 * 1024**2))

def release_gpu_only_copies():
    pid, args = _server_proc()
    todo = [f for f in release_candidates if _local_path(f).exists()]
    if not pid or not todo or not health():
        return
    names = sorted({_local_path(f).name for f in todo} | released_reuse)
    tmp_state = RELEASE_STATE.with_suffix('.tmp')
    tmp_state.write_text(json.dumps({'pid': pid, 'proc_id': _proc_identity(pid), 'files': names}), encoding='utf-8', newline='')
    os.replace(tmp_state, RELEASE_STATE)
    if DEBUG_VERIFY:
        before = _mem_available()
    for f in todo:
        lp = _local_path(f)
        lp.resolve().unlink(missing_ok=True)
        lp.unlink(missing_ok=True)
    expect = sum(f['size'] for f in todo)
    if DEBUG_VERIFY:
        time.sleep(1)
        gained = _mem_available() - before
        cap = _cache_ram_of(args) or 0
        print(f'已删除 {len(todo)} 个显存专用分片的内存盘副本：可用内存 +{gained / 1024**3:.1f} GiB（预计 {expect / 1024**3:.1f} GiB）| 当前服务提示缓存上限 {cap / 1024:.0f} GiB', flush=True)
        if gained < expect * 0.5:
            print('⚠️ 实际回收不到预计的一半：部分分片仍被服务映射，提示缓存别塞满；服务重启后才会真正归还。', flush=True)
    else:
        print(f'已删除 {len(todo)} 个显存专用分片的内存盘副本（预计 {expect / 1024**3:.1f} GiB）', flush=True)

if MODEL.exists():
    with MODEL.open('rb') as f:
        raw_head = f.read(16 * 1024 * 1024)
    HEAD_CACHE.write_bytes(raw_head)
else:
    raw_head = HEAD_CACHE.read_bytes()
tmpl_key = b'tokenizer.chat_template'
key_pos = raw_head.index(struct.pack('<Q', len(tmpl_key)) + tmpl_key)
val_pos = key_pos + 8 + len(tmpl_key)
assert struct.unpack_from('<I', raw_head, val_pos)[0] == 8, 'tokenizer.chat_template 不是 GGUF 字符串类型'
tmpl_len = struct.unpack_from('<Q', raw_head, val_pos + 4)[0]
assert val_pos + 12 + tmpl_len <= len(raw_head), '模板超出前 16 MiB'
tmpl_str = raw_head[val_pos + 12:val_pos + 12 + tmpl_len].decode('utf-8')

# 官方模板只有 xhigh/medium/low 三档；换成四档，并按 effort_prompts 在 system 顶部注入一句中文引导。
universal_effort_block = '''{%- set reasoning_instructions = '' %}
{%- if enable_thinking is undefined or enable_thinking is true %}
    {%- set resolved_reasoning_effort = reasoning_effort|default('@default@') %}
    {%- if resolved_reasoning_effort not in ('xhigh', 'high', 'medium', 'low') %}
        {{- raise_exception('Unexpected reasoning effort ' ~ reasoning_effort ~ '. Supported types are xhigh, high, medium, and low (default @default@).') }}
    {%- endif %}
    {%- if resolved_reasoning_effort == 'xhigh' %}
        {%- set reasoning_instructions = '@xhigh@' %}
    {%- elif resolved_reasoning_effort == 'high' %}
        {%- set reasoning_instructions = '@high@' %}
    {%- elif resolved_reasoning_effort == 'medium' %}
        {%- set reasoning_instructions = '@medium@' %}
    {%- elif resolved_reasoning_effort == 'low' %}
        {%- set reasoning_instructions = '@low@' %}
    {%- endif %}
{%- endif %}'''
for _lvl, _txt in SPEC['effort_prompts'].items():
    universal_effort_block = universal_effort_block.replace('@%s@' % _lvl, _txt)
universal_effort_block = universal_effort_block.replace('@default@', SPEC.get('reasoning_effort_default') or 'high')

fam = SPEC.get('template_family', 'qwen38_fn')
if fam == 'qwen38_fn':
    old_effort_block = '''{%- if enable_thinking is undefined or enable_thinking is true %}
    {%- set resolved_reasoning_effort = reasoning_effort|default('xhigh') %}
    {%- if resolved_reasoning_effort not in ('xhigh', 'medium', 'low') %}
        {{- raise_exception('Unexpected reasoning effort ' ~ reasoning_effort ~ '. Supported types are xhigh (default), medium, and low.') }}
    {%- endif %}
    {%- if resolved_reasoning_effort == 'xhigh' %}
        {%- set reasoning_instructions = 'Reasoning effort is set to xhigh. Please think carefully through the task, validate key assumptions, consider plausible alternatives, and prioritize correctness, consistency, and clarity in the final answer.' %}
    {%- elif resolved_reasoning_effort == 'low' %}
        {%- set reasoning_instructions = 'Reasoning effort is set to low. Keep your thinking brief and focused, moving directly to the conclusion without unnecessary elaboration.' %}
    {%- endif %}
{%- endif %}'''
    # Unsloth 导出的原版 Qwen3.8 模板在 resolved_reasoning_effort 下多塞了 3 行 high->xhigh 折合；先剥掉再统一替换成四档中文引导。
    unsloth_high_shim = '''    {%- if resolved_reasoning_effort == 'high' %}
        {%- set resolved_reasoning_effort = 'xhigh' %}
    {%- endif %}
'''
    tmpl_str = tmpl_str.replace(unsloth_high_shim, '', 1)
    assert old_effort_block in tmpl_str, '未找到预期的 reasoning_effort 模板块'
    tmpl_str = tmpl_str.replace(old_effort_block, universal_effort_block, 1)
    old_preserve_line = '{%- if preserve_thinking is undefined or preserve_thinking is true or loop.index0 > ns.last_query_index %}'
    assert old_preserve_line in tmpl_str, '未找到预期的 preserve_thinking 模板行'
    # 官方默认 preserve_thinking 未定义即为真，历史轮次会被套上空 <think></think>。
    # 只保留当前工具链里 reasoning_content 非空的轮次，避免 few-shot 诱导模型秒关思考。
    # 生成起手块不改：官方只在 enable_thinking=false 时关思考，low 照常思考。
    tmpl_str = tmpl_str.replace(
        old_preserve_line,
        '{%- if loop.index0 > ns.last_query_index and reasoning_content %}',
        1,
    )
elif fam == 'gemma4':
    old_think_body = '''        {{- '<|think|>\\n' -}}
        {%- set ns.prev_message_type = 'think' -%}'''
    new_think_body = '''        {{- '<|think|>\\n' + (reasoning_instructions + '\\n' if reasoning_instructions else '') -}}
        {%- set ns.prev_message_type = 'think' -%}'''
    assert old_think_body in tmpl_str, '未找到预期的 gemma4 <|think|> 注入块'
    tmpl_str = tmpl_str.replace(
        '{%- set enable_thinking = enable_thinking | default(false) -%}',
        '{%- set enable_thinking = enable_thinking | default(true) -%}',
        1,
    )
    tmpl_str = (
        "{%- if enable_thinking is not defined %}{%- set enable_thinking = true %}{%- endif %}\n"
        + universal_effort_block
        + "\n"
        + tmpl_str.replace(old_think_body, new_think_body, 1)
    )
elif fam == 'glm5_next':
    old_glm_head = '''{%- set effective_reasoning_effort = reasoning_effort if reasoning_effort is defined and reasoning_effort in ['low', 'high'] else 'max' -%}
{%- if effective_reasoning_effort is not none -%}<|system|>Reasoning Effort: {{ effective_reasoning_effort | capitalize }}{%- endif -%}
{%- set clear_thinking = clear_thinking if clear_thinking is defined else false -%}'''
    new_glm_head = (
        universal_effort_block
        + "\n{%- set effective_reasoning_effort = 'max' if resolved_reasoning_effort == 'xhigh' else ('low' if resolved_reasoning_effort == 'low' else 'high') -%}"
        + "\n{%- if effective_reasoning_effort is not none -%}<|system|>Reasoning Effort: {{ effective_reasoning_effort | capitalize }}{{ ('\\n' + reasoning_instructions) if reasoning_instructions else '' }}{%- endif -%}"
        + "\n{%- set clear_thinking = clear_thinking if clear_thinking is defined else true -%}"
    )
    assert old_glm_head in tmpl_str, '未找到预期的 glm5_next reasoning_effort 模板块'
    tmpl_str = tmpl_str.replace(old_glm_head, new_glm_head, 1)
elif fam == 'qwen35':
    old_tools_head = "{%- if tools and tools is iterable and tools is not mapping %}\n    {{- '<|im_start|>system\\n' }}"
    new_tools_head = (
        universal_effort_block
        + "\n{%- if tools and tools is iterable and tools is not mapping %}\n"
        + "    {{- '<|im_start|>system\\n' }}\n"
        + "    {%- if reasoning_instructions %}\n"
        + "        {{- reasoning_instructions + '\\n\\n' }}\n"
        + "    {%- endif %}"
    )
    assert old_tools_head in tmpl_str, '未找到预期的 qwen35 tools system 头部'
    tmpl_str = tmpl_str.replace(old_tools_head, new_tools_head, 1)
    old_tools_sys = '''    {%- if messages[0].role == 'system' %}
        {%- set content = render_content(messages[0].content, false, true)|trim %}
        {%- if content %}
            {{- '\\n\\n' + content }}
        {%- endif %}
    {%- endif %}
    {{- '<|im_end|>\\n' }}
{%- else %}
    {%- if messages[0].role == 'system' %}
        {%- set content = render_content(messages[0].content, false, true)|trim %}
        {{- '<|im_start|>system\\n' + content + '<|im_end|>\\n' }}
    {%- endif %}
{%- endif %}'''
    new_tools_sys = '''    {%- if messages[0].role == 'system' %}
        {%- set content = render_content(messages[0].content, false, true)|trim %}
        {%- if content %}
            {{- '\\n\\n' + content }}
        {%- endif %}
    {%- endif %}
    {{- '<|im_end|>\\n' }}
{%- else %}
    {%- if messages[0].role == 'system' %}
        {%- set content = render_content(messages[0].content, false, true)|trim %}
        {{- '<|im_start|>system\\n' + (reasoning_instructions + '\\n\\n' if reasoning_instructions else '') + content + '<|im_end|>\\n' }}
    {%- elif reasoning_instructions %}
        {{- '<|im_start|>system\\n' + reasoning_instructions + '<|im_end|>\\n' }}
    {%- endif %}
{%- endif %}'''
    assert old_tools_sys in tmpl_str, '未找到预期的 qwen35 system 分支块'
    tmpl_str = tmpl_str.replace(old_tools_sys, new_tools_sys, 1)
    old_hist_think = '{%- if loop.index0 > ns.last_query_index %}'
    assert old_hist_think in tmpl_str, '未找到预期的历史思考保留条件'
    tmpl_str = tmpl_str.replace(old_hist_think, '{%- if loop.index0 > ns.last_query_index and reasoning_content %}', 1)
    qwen_gen_think_old = '''    {%- if enable_thinking is defined and enable_thinking is false %}
        {{- '<think>\\n\\n</think>\\n\\n' }}
    {%- else %}
        {{- '<think>\\n' }}
    {%- endif %}'''
    qwen_gen_think_new = '''    {%- if (enable_thinking is defined and enable_thinking is false) or (reasoning_effort is defined and reasoning_effort == 'low') %}
        {{- '<think>\\n\\n</think>\\n\\n' }}
    {%- else %}
        {{- '<think>\\n' }}
    {%- endif %}'''
    assert qwen_gen_think_old in tmpl_str, '未找到预期的 add_generation_prompt 思考起手块'
    tmpl_str = tmpl_str.replace(qwen_gen_think_old, qwen_gen_think_new, 1)
elif fam == 'nemotron':
    old_trunc = '{%- set truncate_history_thinking = truncate_history_thinking if truncate_history_thinking is defined else True %}'
    assert old_trunc in tmpl_str, '未找到预期的 truncate_history_thinking 行'
    tmpl_str = tmpl_str.replace(old_trunc, '{%- set truncate_history_thinking = false %}', 1)
    old_sys_block = '''{%- if messages[0]["role"] == "system" %}
    {%- set system_message = messages[0]["content"] %}
    {%- set loop_messages = messages[1:] %}
{%- else %}
    {%- set system_message = "" %}
    {%- set loop_messages = messages %}
{%- endif %}'''
    new_sys_block = (
        universal_effort_block
        + '''
{%- if messages[0]["role"] == "system" %}
    {%- set system_message = (reasoning_instructions ~ "\\n\\n" if reasoning_instructions else "") ~ render_content(messages[0]["content"]) %}
    {%- set loop_messages = messages[1:] %}
{%- else %}
    {%- set system_message = reasoning_instructions %}
    {%- set loop_messages = messages %}
{%- endif %}'''
    )
    assert old_sys_block in tmpl_str, '未找到预期的 nemotron system_message 块'
    tmpl_str = tmpl_str.replace(old_sys_block, new_sys_block, 1)
    nemo_gen_think_old = '''{%- if add_generation_prompt %}
    {%- if enable_thinking %}
        {{- '<|im_start|>assistant\\n<think>\\n' }}
    {%- else %}
        {{- '<|im_start|>assistant\\n<think></think>' }}
    {%- endif %}
{%- endif %}'''
    nemo_gen_think_new = '''{%- if add_generation_prompt %}
    {{- '<|im_start|>assistant\\n' }}
    {%- if (enable_thinking is defined and enable_thinking is false) or (reasoning_effort is defined and reasoning_effort == 'low') %}
        {{- '<think></think>' }}
    {%- else %}
        {{- '<think>\\n' }}
    {%- endif %}
{%- endif -%}'''
    assert nemo_gen_think_old in tmpl_str, '未找到预期的 nemotron add_generation_prompt 思考起手块'
    tmpl_str = tmpl_str.replace(nemo_gen_think_old, nemo_gen_think_new, 1)
else:
    raise RuntimeError(f'未知 template_family={fam}：必须在 colab_kernel.py 中接入 universal_effort_block 四档思考支持')

if LORA and LORA.exists():
    # GCSA-AiLab/GLM-5.3-Flash-Abliterated-LoRA-GGUF 头部写成了 glm5next（缺连字符），原地修补为 glm5-next 并消掉 1 字节对齐填充，保持张量偏移与文件大小不变。
    with LORA.open('r+b') as _lf:
        _lbuf = _lf.read(32768)
        _bad_arch = b'\x08\x00\x00\x00\x00\x00\x00\x00glm5next'
        _good_arch = b'\x09\x00\x00\x00\x00\x00\x00\x00glm5-next'
        if _bad_arch in _lbuf:
            _pad_idx = _lbuf.find(b'\x00' * 8, 16384)
            assert _pad_idx > 0, 'LoRA GGUF 未找到对齐填充区'
            _lbuf = _lbuf.replace(_bad_arch, _good_arch, 1)
            _lbuf = _lbuf[:_pad_idx] + _lbuf[_pad_idx + 1:]
            _lf.seek(0)
            _lf.write(_lbuf)
old_tmpl_on_disk = CHAT_TMPL.read_text(encoding='utf-8') if CHAT_TMPL.exists() else ''
CHAT_TMPL.write_text(tmpl_str, encoding='utf-8', newline='')
UI_CFG.write_text(new_ui_text, encoding='utf-8', newline='')

def _fp_args(args):
    # 指纹跳过二进制路径和 --cache-ram（只改缓存上限不重启）。
    out = []
    i = 1 if args and Path(args[0]).name == 'llama-server' else 0
    while i < len(args):
        a = args[i]
        if not a:
            i += 1
            continue
        if a == '--cache-ram':
            i += 2
            continue
        out.append(a)
        i += 1
    return out

def _boot_fp(args, tmpl, ui):
    blob = chr(0).join(_fp_args(args)) + chr(0) + tmpl + chr(0) + ui
    return hashlib.sha1(blob.encode()).hexdigest()

load_flags = ['--lazy-mode', 'off'] if ple_in_tmpfs else ['-lm', 'none', '--lazy-mode', 'off']
server_args = ['-m', str(MODEL)]
if MMPROJ:
    server_args += ['--mmproj', str(MMPROJ)]
if DRAFT and SPEC.get('mtp_n_max'):
    server_args += ['--model-draft', str(DRAFT), '--spec-type', 'draft-mtp',
                    '--spec-draft-n-max', str(SPEC['mtp_n_max']),
                    '--spec-draft-p-min', str(SPEC['mtp_p_min'])]
if LORA:
    server_args += ['--lora', str(LORA)]
if SPEC.get('extra_server_flags'):
    server_args += list(SPEC['extra_server_flags'])
server_args += [
    '--slot-save-path', str(SLOT_SAVE_DIR),
    '--device', 'CUDA0',
    '-ngl', '99',
    *load_flags,
    '--cache-ram', str(PROMPT_CACHE_MIB),
    '-c', str(TOTAL_CONTEXT_TOKENS),
    '--parallel', str(PARALLEL_SLOTS),
    '--host', '127.0.0.1',
    '--port', str(PORT),
    '--alias', MODEL_ALIAS,
    '--jinja',
    '--chat-template-file', str(CHAT_TMPL),
    '--ui-config-file', str(UI_CFG),
    '--ui-mcp-proxy',
    '--reasoning', 'on',
]
if SPEC.get('reasoning_effort_default'):
    server_args += ['--reasoning-effort', SPEC['reasoning_effort_default']]
server_args += [
    '--temp', SPEC['temp'],
    '--top-p', SPEC['top_p'],
    '--top-k', SPEC['top_k'],
    '--min-p', SPEC['min_p'],
    *SPEC.get('penalty_args', []),
    '--flash-attn', 'on',
    '--cache-type-k', 'q8_0',
    '--cache-type-v', 'q8_0',
    '--batch-size', '4096',
    '--ubatch-size', str(UBATCH_SIZE),
    '--predict', str(MAX_OUTPUT_TOKENS),
]
expected_fp = _boot_fp(server_args, tmpl_str, new_ui_text)

def configured_server():
    # 已在跑且启动指纹一致就跳过重启。指纹 = sha1(argv 去掉二进制和 --cache-ram + 模板 + webui-config)。
    if not health():
        return False
    _pid, args = _server_proc()
    if not args or _boot_fp(args, old_tmpl_on_disk, old_ui_on_disk) != expected_fp:
        return False
    if DEBUG_VERIFY:
        props = requests.get('http://127.0.0.1:%d/props' % PORT, headers=AUTH, timeout=10).json()
        if props.get('default_generation_settings', {}).get('n_ctx') != SLOT_CONTEXT_TOKENS:
            return False
        if props.get('total_slots') != PARALLEL_SLOTS:
            return False
        if props.get('chat_template_caps', {}).get('reasoning_effort_levels') != ['low', 'medium', 'high', 'xhigh']:
            return False
    return True

server_just_started = False
if health() and not configured_server():
    print('检测到旧配置或不同模型服务：重启 llama-server ...', flush=True)
    if not released_reuse:
        _stop_llama_servers()

if configured_server():
    print(f'模型已在运行：{MODEL_ALIAS} / 单路 × {SLOT_CONTEXT_TOKENS} 上下文 / {(_cache_ram_of(_server_proc()[1]) or 0) / 1024:.0f} GiB Prompt Cache', flush=True)
else:
    missing_now = [_local_path(f).name for f in FILES if not _local_path(f).exists()]
    if released_reuse or missing_now:
        _stop_llama_servers()
        RELEASE_STATE.unlink(missing_ok=True)
        raise RuntimeError('需要重新启动服务，但缺少上次为扩大提示缓存而删掉的分片副本：'
                           + ', '.join(sorted(set(missing_now) | released_reuse))
                           + '。旧服务已停，请再运行一次本格，会重新下载这些分片。')
    _stop_llama_servers()
    if ple_name and not freshly_downloaded and not ple_in_tmpfs:
        meminfo = Path('/proc/meminfo').read_text()
        cached_kb = next((int(l.split()[1]) for l in meminfo.splitlines() if l.startswith('Cached:')), 0)
        if cached_kb < 40 * 1024 * 1024:
            print('预热 PLE 表所在分片到主机内存 Page Cache ...', flush=True)
            with (SHARD_DIR / ple_name).open('rb') as f:
                while f.read(32 * 1024 * 1024):
                    pass

    fut_cpp.result()
    SERVER = next(DIST.rglob('llama-server'))
    env = os.environ.copy()
    libdirs = {str(SERVER.parent)}
    for pat in ('libcudart.so.12', 'libcublas.so.12', 'libcublasLt.so.12'):
        libdirs.update(str(p.parent) for p in DIST.rglob(pat))
    env['LD_LIBRARY_PATH'] = ':'.join(sorted(libdirs)) + ':' + env.get('LD_LIBRARY_PATH', '')
    cmd = [str(SERVER), *server_args]
    log = (ROOT / 'server.log').open('a', encoding='utf-8', newline='')
    log.write(chr(10) + '=== START %s ===' % time.strftime('%Y-%m-%d %H:%M:%S') + chr(10))
    log.flush()
    proc = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT, env=env, stdin=subprocess.DEVNULL, start_new_session=True)
    qwen_server = proc
    t0 = time.monotonic()
    # LAUNCH_DETACHED=1（colab_cli launch）：进程已 start_new_session，不等本机 /health，交给后面的后台线程。
    if os.environ.get('LAUNCH_DETACHED') == '1':
        print('LAUNCH_DETACHED: llama-server pid %s' % proc.pid, flush=True)
        server_just_started = True
    else:
        print(f'加载 {MODEL_ALIAS} 到 GPU ...', flush=True)
        while time.monotonic() - t0 < 600:
            if proc.poll() is not None:
                raise RuntimeError((ROOT / 'server.log').read_text(errors='replace')[-3500:])
            if health():
                break
            time.sleep(2 if DEBUG_VERIFY else 0.5)
        else:
            proc.terminate()
            raise RuntimeError(f'模型加载超时，请查看 {ROOT}/server.log')
        server_just_started = True

def _warmup_cuda():
    # 第一次推理会捕获 CUDA graph，短提示通常要数秒到十几秒，不是零点几秒；放后台不挡就绪横幅。只发这一次 max_tokens=4。
    requests.post('http://127.0.0.1:%d/v1/chat/completions' % PORT, headers=AUTH, timeout=600,
                  json={'model': MODEL_ALIAS,
                        'messages': [{'role': 'user', 'content': '请只回复“就绪”两个字。'}],
                        'max_tokens': 4, 'temperature': 0, 'reasoning_effort': 'low'})

if DEBUG_VERIFY:
    if server_just_started:
        print('服务就绪 %.0fs，核验四档思考能力并发送预热请求初始化 CUDA 图 ...' % (time.monotonic() - t0), flush=True)
        props_now = requests.get('http://127.0.0.1:%d/props' % PORT, headers=AUTH, timeout=10).json()
        levels_now = props_now.get('chat_template_caps', {}).get('reasoning_effort_levels', [])
        if levels_now != ['low', 'medium', 'high', 'xhigh']:
            proc.terminate()
            raise RuntimeError(f'思考档位探测未达四档标准 (期望 low/medium/high/xhigh): {levels_now}')
        print('思考档位门禁通过：', levels_now, '| 默认档位：', SPEC.get('reasoning_effort_default', 'high'), flush=True)
        _warmup_cuda()
    verify_hashes()
    release_gpu_only_copies()
    fut_ts.result()
    status = ts('serve', 'status').stdout
    if SERVICE_FQDN not in status:
        raise RuntimeError('Service 未生效，serve status:' + chr(10) + status)
else:
    # 就绪横幅不等这些：预热、删内存盘副本、等 Tailscale。
    def _bg_finish():
        try:
            if server_just_started:
                if os.environ.get('LAUNCH_DETACHED') == '1':
                    # 主线程已返回。预热必须先等本机 /health，否则连接失败会跳过后面的内存盘释放和 Tailscale 收尾。
                    deadline = time.monotonic() + 600
                    while time.monotonic() < deadline:
                        if proc.poll() is not None:
                            raise RuntimeError((ROOT / 'server.log').read_text(errors='replace')[-3500:])
                        if health():
                            break
                        time.sleep(0.5)
                    else:
                        raise RuntimeError('模型加载超时，请查看 %s/server.log' % ROOT)
                _warmup_cuda()
            release_gpu_only_copies()
            fut_cpp.result()
            fut_ts.result()
        except Exception as exc:
            print('后台收尾失败：', exc, flush=True)
    threading.Thread(target=_bg_finish, daemon=True).start()
    print('生产模式：预热、释放内存盘副本、Tailscale 在后台收尾，不挡住下面的就绪横幅。', flush=True)

print('=' * 64)
print('总启动耗时： %.1fs' % (time.monotonic() - t_boot), flush=True)
print('网页版：     http://%s:443/' % SERVICE_FQDN, flush=True)
print('接口：       http://%s:443/v1' % SERVICE_FQDN, flush=True)
print('模型名：     %s' % MODEL_ALIAS, flush=True)
print('=' * 64)

# 6. 就绪横幅 + WebUI 标签页 + 停机按钮
# 浏览器只允许 window.open 的发起帧关闭该标签页，所以 WebUI 由本格输出帧打开，本格停机按钮先关它再释放运行时。
from google.colab import output as colab_output, runtime
from IPython.display import HTML, display

def _qwen_stop():
    if globals().get('qwen_server') and qwen_server.poll() is None:
        qwen_server.terminate()
    if Path(TS).exists():
        try:
            subprocess.run([TS, '--socket=' + TS_SOCK, 'down'], timeout=5, capture_output=True)
        except Exception:
            pass
    runtime.unassign()

colab_output.register_callback('qwen.stop', _qwen_stop)

_QWEN_CONSOLE_HTML = """
<div style="margin:10px 0;padding:12px;border-radius:8px;background:#1e293b;color:#f8fafc;font-size:15px;line-height:1.6;">
  🚀 <b>服务已就绪！</b>
  <a id="qwen-open" href="__URL__" style="color:#38bdf8;font-weight:bold;text-decoration:underline;margin-left:8px;">👉 点击打开 WebUI 交互界面 (免密直连)</a>
  <div style="margin-top:10px;">
    <button id="qwen-stop" style="background:#f0ad4e;color:#111;border:0;border-radius:4px;padding:6px 14px;font-size:14px;cursor:pointer;">🛑 结束 G4 运行时 (省 CU)</button>
  </div>
  <div id="qwen-state" style="margin-top:8px;font-size:14px;"><b>当前状态：</b><span style="color:#5cb85c">🔓 GLM-5.3-Flash 320B-A18B GSQ-RCO-Q4（合入 LoRA-v2），满速运行</span> — 用完点黄色按钮立即释放 GPU，WebUI 标签页会随之自动关闭</div>
</div>
<script>
(() => {
  const URL = "__URL__";
  const stateEl = document.getElementById("qwen-state");
  const bootMs = Date.now();
  let win = null;
  // 只由本次运行的 eval_js 调用；重开笔记本时旧输出里的脚本会重跑，但不会自动弹窗
  window.qwenOpenWebui = () => {
    win = window.open(URL, "qwen-webui");
    if (!win) stateEl.textContent = "浏览器拦截了自动弹窗，请点上面的链接打开";
  };
  document.getElementById("qwen-open").addEventListener("click", (e) => {
    e.preventDefault();
    window.qwenOpenWebui();
  });
  document.getElementById("qwen-stop").addEventListener("click", (e) => {
    e.currentTarget.disabled = true;
    e.currentTarget.textContent = "正在释放…";
    if (win && !win.closed) win.close();
    const runMin = ((Date.now() - bootMs) / 60000).toFixed(1);
    stateEl.innerHTML = "<b>正在关闭服务并销毁运行时（本次就绪后已运行 " + runMin + " 分钟，停止计费）…</b>";
    google.colab.kernel.invokeFunction("qwen.stop", [], {}).catch(() => {});
  });
})();
</script>
""".replace('__URL__', 'http://%s:443/' % SERVICE_FQDN)
display(HTML(_QWEN_CONSOLE_HTML))
try:
    colab_output.eval_js('window.qwenOpenWebui()', ignore_result=True)
except Exception:
    pass


In [ ]:
# 可选：查看当前配置、GPU 显存与主机内存占用
import json, subprocess
from pathlib import Path
import requests

root = Path(globals().get('SPEC', {}).get('root_dir', "/content/glm53-flash"))
hdr = {'Authorization': 'Bearer ' + (root / '.llama-api-key').read_text().strip()}
r = requests.get('http://127.0.0.1:8081/props', headers=hdr, timeout=10)
r.raise_for_status()
props = r.json()
p = props['default_generation_settings']
print('每槽上下文:', p['n_ctx'], 'tokens | 并发槽位数:', props.get('total_slots'), '| WebUI 配置已下发:', bool(props.get('ui_settings') or props.get('webui_settings')))
for procdir in Path('/proc').iterdir():
    if not procdir.name.isdigit():
        continue
    try:
        args = (procdir / 'cmdline').read_bytes().decode().split('\0')
        if args and Path(args[0]).name == 'llama-server':
            flags = ('--alias', '--mmproj', '--model-draft', '--lora', '-c', '--parallel', '--predict', '--spec-draft-n-max', '--spec-draft-p-min', '--reasoning', '--reasoning-effort', '--cache-ram', '--ui-config-file', '--temp', '--top-p', '--top-k', '--min-p', '--repeat-penalty', '--repeat-last-n', '--presence-penalty', '--frequency-penalty', '--dry-multiplier', '--dry-base', '--dry-allowed-length', '--dry-penalty-last-n', '--batch-size', '--ubatch-size', '--cache-type-k', '--cache-type-v', '-lm', '--lazy-mode', '-ot', '--n-cpu-moe')
            print('实际启动参数:', json.dumps({k: args[args.index(k) + 1] for k in flags if k in args}, ensure_ascii=False))
            print('WebUI MCP 代理已启用:', '--ui-mcp-proxy' in args)
    except (FileNotFoundError, PermissionError, IndexError):
        pass
print('GPU:', subprocess.getoutput('nvidia-smi --query-gpu=name,memory.total,memory.used,memory.free --format=csv,noheader'))
print('Host RAM:', subprocess.getoutput('free -h | head -n 2'))
for name in ('released-copies.json',):
    p = root / name
    if p.exists():
        print(name + ':', p.read_text(encoding='utf-8')[:400])


In [ ]:
# 【调试用 · 手动运行】全链路验收：接口 / 思考档位切换 / 速度 / 中文小说共创 / 工具调用
import json, time
from pathlib import Path
import requests

CUR_SPEC = globals().get('SPEC', {})
ROOT = Path(CUR_SPEC.get('root_dir', "/content/glm53-flash"))
MODEL_ALIAS = CUR_SPEC.get('alias', "GLM-5.3-Flash-Uncensored-GSQ-RCO-Q4")
base = 'http://127.0.0.1:8081'
headers = {'Authorization': 'Bearer ' + (ROOT / '.llama-api-key').read_text().strip()}

models = requests.get(base + '/v1/models', headers=headers, timeout=15).json()
print('模型：', [m['id'] for m in models['data']])
assert MODEL_ALIAS in [m['id'] for m in models['data']]

novel = ('你是我的中文小说共创搭档。下面这段武侠场景需要你接着往下写：\n'
         '　　「风从断崖口灌进来，卷起满地碎雪。沈砚之握着半截断剑，指节发白。」\n'
         '请续写 200 字左右：只写正文，不要解释思路，以动作细节表现人物心虚但嘴硬。')

for effort in ('high', 'xhigh'):
    t0 = time.monotonic()
    r = requests.post(base + '/v1/chat/completions', headers=headers, timeout=600,
                      json={'model': MODEL_ALIAS, 'messages': [{'role': 'user', 'content': novel}],
                            'reasoning_effort': effort, 'temperature': float(CUR_SPEC.get('temp', "1.0")), 'max_tokens': 512})
    r.raise_for_status()
    d = r.json()
    msg = d['choices'][0]['message']
    tm = d.get('timings') or {}
    print(f'=== effort={effort} | wall={time.monotonic()-t0:.2f}s | tok/s={tm.get("predicted_per_second", 0):.1f} ===')
    print('REASONING:', (msg.get('reasoning_content') or '(none)')[:160].replace('\n', ' '))
    print('CONTENT:', (msg.get('content') or '')[:240].replace('\n', ' '))

tools = [{'type': 'function', 'function': {'name': 'get_weather', 'description': '查询城市天气',
          'parameters': {'type': 'object', 'properties': {'city': {'type': 'string'}}, 'required': ['city']}}}]
rt = requests.post(base + '/v1/chat/completions', headers=headers, timeout=120,
                   json={'model': MODEL_ALIAS, 'messages': [{'role': 'user', 'content': '亚特兰大现在天气如何？用 get_weather 工具查。'}],
                         'reasoning_effort': 'low', 'temperature': 0, 'tools': tools, 'max_tokens': 128}).json()
print('TOOL_CALLS:', json.dumps(rt['choices'][0]['message'].get('tool_calls'), ensure_ascii=False))


In [ ]:
# 热切换模型：不释放 G4，停掉当前模型、清掉它的文件，再执行目标模型的一键启动（权重重新下载，约 2–4 分钟）
TARGET = 'qwen38-huihui-v2'  # @param ["qwen38-huihui-v2", "gemma4-31b", "qwen38-27b-rvn", "glm53-flash"]
import os, shutil, signal, subprocess, time
from pathlib import Path
import requests

SWITCH = {'qwen38-huihui-v2': {'alias': 'Huihui-Qwen3.8-Flash-Next-abliterated-v2-UD-Q4_K_XL', 'root_dir': '/content/qwen38-huihui-v2', 'min_vram_mib': 85000, 'code': '# 一键启动（由 colab_kernel.py 统一内核生成）：tmpfs 内存盘 + 32路下载 + 字节大小校验 + WebUI 本机文件桥 + Tailscale\n# 调试新模型/改配置时设 True\nDEBUG_VERIFY = False\nimport concurrent.futures, hashlib, json, os, shutil, socket, struct, subprocess, sys, threading, time\nfrom pathlib import Path\nimport requests\nfrom requests.adapters import HTTPAdapter\n\nt_boot = time.monotonic()\nSPEC = json.loads("{\\n  \\"slug\\": \\"qwen38-huihui-v2\\",\\n  \\"alias\\": \\"Huihui-Qwen3.8-Flash-Next-abliterated-v2-UD-Q4_K_XL\\",\\n  \\"root_dir\\": \\"/content/qwen38-huihui-v2\\",\\n  \\"ts_hostname\\": \\"qwen-g4huihui-colab\\",\\n  \\"min_vram_mib\\": 85000,\\n  \\"ram_reserve_gib\\": 95,\\n  \\"ple_shard_name\\": \\"Qwen3.8-Flash-Next-UD-Q4_K_XL-00002-of-00004.gguf\\",\\n  \\"temp\\": \\"1.0\\",\\n  \\"top_p\\": \\"0.95\\",\\n  \\"top_k\\": \\"20\\",\\n  \\"min_p\\": \\"0.0\\",\\n  \\"reasoning_effort_default\\": \\"xhigh\\",\\n  \\"effort_prompts\\": {\\n    \\"xhigh\\": \\"\\u7528\\u4e2d\\u6587\\u601d\\u8003\\uff0c\\u5c3d\\u91cf\\u6df1\\u5165\\u3002\\",\\n    \\"high\\": \\"\\u7528\\u4e2d\\u6587\\u601d\\u8003\\u3002\\",\\n    \\"medium\\": \\"\\u7528\\u4e2d\\u6587\\u601d\\u8003\\uff0c\\u7b80\\u77ed\\u4e9b\\u3002\\",\\n    \\"low\\": \\"\\u7528\\u4e2d\\u6587\\u601d\\u8003\\uff0c\\u8d8a\\u77ed\\u8d8a\\u597d\\u3002\\"\\n  },\\n  \\"mtp_n_max\\": 3,\\n  \\"mtp_p_min\\": \\"0.48\\",\\n  \\"files\\": [\\n    {\\n      \\"role\\": \\"model\\",\\n      \\"repo\\": \\"huihui-ai/Huihui-Qwen3.8-Flash-Next-abliterated-v2-GGUF\\",\\n      \\"rev\\": \\"cbc7fba13f1939ac3fa42b18d77ccf43847d74ef\\",\\n      \\"path\\": \\"unsloth-UD-Q4_K_XL-V2/Qwen3.8-Flash-Next-UD-Q4_K_XL-00001-of-00004.gguf\\",\\n      \\"local_subdir\\": \\"unsloth-UD-Q4_K_XL-V2\\",\\n      \\"size\\": 10946624,\\n      \\"priority\\": 2\\n    },\\n    {\\n      \\"role\\": \\"shard\\",\\n      \\"repo\\": \\"huihui-ai/Huihui-Qwen3.8-Flash-Next-abliterated-v2-GGUF\\",\\n      \\"rev\\": \\"cbc7fba13f1939ac3fa42b18d77ccf43847d74ef\\",\\n      \\"path\\": \\"unsloth-UD-Q4_K_XL-V2/Qwen3.8-Flash-Next-UD-Q4_K_XL-00002-of-00004.gguf\\",\\n      \\"local_subdir\\": \\"unsloth-UD-Q4_K_XL-V2\\",\\n      \\"size\\": 49859583136,\\n      \\"priority\\": 1\\n    },\\n    {\\n      \\"role\\": \\"shard\\",\\n      \\"repo\\": \\"huihui-ai/Huihui-Qwen3.8-Flash-Next-abliterated-v2-GGUF\\",\\n      \\"rev\\": \\"cbc7fba13f1939ac3fa42b18d77ccf43847d74ef\\",\\n      \\"path\\": \\"unsloth-UD-Q4_K_XL-V2/Qwen3.8-Flash-Next-UD-Q4_K_XL-00003-of-00004.gguf\\",\\n      \\"local_subdir\\": \\"unsloth-UD-Q4_K_XL-V2\\",\\n      \\"size\\": 49376141504,\\n      \\"priority\\": 3\\n    },\\n    {\\n      \\"role\\": \\"shard\\",\\n      \\"repo\\": \\"huihui-ai/Huihui-Qwen3.8-Flash-Next-abliterated-v2-GGUF\\",\\n      \\"rev\\": \\"cbc7fba13f1939ac3fa42b18d77ccf43847d74ef\\",\\n      \\"path\\": \\"unsloth-UD-Q4_K_XL-V2/Qwen3.8-Flash-Next-UD-Q4_K_XL-00004-of-00004.gguf\\",\\n      \\"local_subdir\\": \\"unsloth-UD-Q4_K_XL-V2\\",\\n      \\"size\\": 12087983520,\\n      \\"priority\\": 3\\n    },\\n    {\\n      \\"role\\": \\"mmproj\\",\\n      \\"repo\\": \\"huihui-ai/Huihui-Qwen3.8-Flash-Next-abliterated-v2-GGUF\\",\\n      \\"rev\\": \\"cbc7fba13f1939ac3fa42b18d77ccf43847d74ef\\",\\n      \\"path\\": \\"mmproj-Qwen3.8-Flash-Next-BF16.gguf\\",\\n      \\"size\\": 907543008,\\n      \\"priority\\": 2\\n    },\\n    {\\n      \\"role\\": \\"draft\\",\\n      \\"repo\\": \\"unsloth/Qwen3.8-Flash-Next-GGUF\\",\\n      \\"rev\\": \\"766911a6b7369840a91dbcd95f9f997acaab6cd6\\",\\n      \\"path\\": \\"MTP/mtp-Qwen3.8-Flash-Next-Q4_K_M.gguf\\",\\n      \\"size\\": 2786204800,\\n      \\"priority\\": 2\\n    }\\n  ]\\n}")\nUI_CFG_DATA = json.loads("{\\n  \\"systemMessage\\": \\"\\u4f60\\u662f\\u4e2d\\u6587\\u957f\\u7bc7\\u5c0f\\u8bf4\\u5171\\u521b\\u642d\\u6863\\u3002\\\\n1. \\u5f00\\u4e86 local_novel_fs \\u65f6\\uff0c\\u52a8\\u7b14\\u524d\\u5148\\u8bfb \\u5199\\u4f5c\\u8981\\u6c42.md \\u548c\\u6b63\\u6587\\u672b\\u5c3e\\uff08tail_lines=150\\uff09\\u3002\\u65b0\\u7ae0\\u6216\\u4e0b\\u534a\\u7ae0\\u7528 write_novel_file \\u8ffd\\u52a0\\uff1b\\u91cd\\u5199\\u4e0a\\u4e00\\u8f6e\\u7528 replace_last_write\\uff0c\\u91cd\\u5199\\u6574\\u7ae0\\u7528 replace_last_chapter\\uff1b\\u5c0f\\u6539\\u7528 edit_novel_file\\uff1b\\u64a4\\u56de\\u7528 undo_novel_write\\u3002\\u6ca1\\u5f00\\u5de5\\u5177\\u5c31\\u76f4\\u63a5\\u5728\\u5bf9\\u8bdd\\u91cc\\u5199\\u3002\\\\n2. \\u67e5\\u539f\\u4f5c\\u7528 grep_reference\\uff1b\\u8bbe\\u5b9a\\u51b2\\u7a81\\u4ee5\\u7528\\u6237\\u5f53\\u524d\\u6307\\u4ee4\\u4e3a\\u51c6\\u3002\\\\n3. \\u7528\\u6237\\u8bf4\\u300c\\u6536\\u5de5\\u300d\\u65f6\\uff0c\\u7528 edit_novel_file \\u628a\\u65b0\\u5b9a\\u7684\\u8bbe\\u5b9a\\u3001\\u8fdb\\u5ea6\\u548c\\u4f0f\\u7b14\\u66f4\\u65b0\\u8fdb \\u5199\\u4f5c\\u8981\\u6c42.md\\uff0c\\u53ea\\u6539\\u53d8\\u5316\\u9879\\u3002\\\\n4. \\u7528\\u6237\\u70b9\\u540d\\u65f6\\u624d\\u8bfb\\u6280\\u80fd\\uff1agrill/\\u62f7\\u95ee \\u8bfb skill:grilling\\uff0c\\u4f18\\u5316\\u63d0\\u793a\\u8bcd \\u8bfb skill:prompting\\uff0cretro/\\u590d\\u76d8 \\u8bfb skill:retro\\u3002\\",\\n  \\"showSystemMessage\\": true,\\n  \\"showMessageStats\\": true,\\n  \\"enableContinueGeneration\\": true,\\n  \\"pasteLongTextToFileLen\\": 50000,\\n  \\"excludeReasoningFromContext\\": true,\\n  \\"renderThinkingAsMarkdown\\": true,\\n  \\"renderUserContentAsMarkdown\\": true,\\n  \\"titleGenerationUseLLM\\": false,\\n  \\"titleGenerationUseFirstLine\\": true,\\n  \\"mcpServers\\": \\"[{\\\\\\"id\\\\\\": \\\\\\"local-novel-fs\\\\\\", \\\\\\"name\\\\\\": \\\\\\"local_novel_fs\\\\\\", \\\\\\"displayName\\\\\\": \\\\\\"\\u672c\\u673a\\u5c0f\\u8bf4\\u76ee\\u5f55 (Colab\\u521b\\u4f5c)\\\\\\", \\\\\\"url\\\\\\": \\\\\\"http://127.0.0.1:3426/mcp\\\\\\", \\\\\\"enabled\\\\\\": true, \\\\\\"useProxy\\\\\\": true}]\\"\\n}")\n\nROOT = Path(SPEC[\'root_dir\'])\nDIST = ROOT / \'llama-dist\'\nMODEL_ALIAS = SPEC[\'alias\']\nKEY_FILE = ROOT / \'.llama-api-key\'\nPORT = 8081\nKEY = "<YOUR_LLAMA_API_KEY>"\nFILES = SPEC[\'files\']\nCPP_TAG = \'b12032\'\nCPP_URL = \'https://github.com/ggml-org/llama.cpp/releases/download/\' + CPP_TAG\n\ndef _local_path(f):\n    sub = f.get(\'local_subdir\')\n    return (ROOT / sub / Path(f[\'path\']).name) if sub else (ROOT / Path(f[\'path\']).name)\n\nMODEL = _local_path(FILES[0])\nMMPROJ = next((_local_path(f) for f in FILES if f[\'role\'] == \'mmproj\'), None)\nDRAFT = next((_local_path(f) for f in FILES if f[\'role\'] == \'draft\'), None)\nLORA = next((_local_path(f) for f in FILES if f[\'role\'] == \'lora\'), None)\n# PLE 表所在目录跟着各模型的 local_subdir 走（IQ4_XS / IQ4_NL …），不能写死。\nSHARD_DIR = next((_local_path(f).parent for f in FILES if Path(f[\'path\']).name == SPEC.get(\'ple_shard_name\')), ROOT)\n\n# 服务把权重读进显存后，/dev/shm 里只供显存用的副本就是冗余内存；删掉后靠这份记录（绑定服务 PID）在重跑本格时跳过重新下载。\nRELEASE_STATE = ROOT / \'released-copies.json\'\nHEAD_CACHE = ROOT / \'model-head.bin\'\nTMPFS_PREFIXES = (\'/dev/shm/\', \'/var/colab/\')\nFILE_NAMES = {_local_path(f).name for f in FILES}\n\ndef _server_proc():\n    # 本模型的 llama-server：(pid, argv)；没有则 (None, [])\n    for procdir in Path(\'/proc\').iterdir():\n        if not procdir.name.isdigit():\n            continue\n        try:\n            args = (procdir / \'cmdline\').read_bytes().decode(errors=\'replace\').split(\'\\0\')\n        except (FileNotFoundError, PermissionError, ProcessLookupError):\n            continue\n        if args and Path(args[0]).name == \'llama-server\' and str(MODEL) in args:\n            return int(procdir.name), args\n    return None, []\n\ndef _server_pid():\n    return _server_proc()[0]\n\ndef _proc_identity(pid):\n    # boot_id + 进程启动时刻，防止重启后的同模型服务恰好复用旧 PID 时误用释放记录\n    try:\n        boot = Path(\'/proc/sys/kernel/random/boot_id\').read_text().strip()\n        stat = Path(\'/proc/%d/stat\' % pid).read_text()\n        return boot + \':\' + stat.rsplit(\')\', 1)[1].split()[19]\n    except (OSError, IndexError, TypeError):\n        return None\n\ndef _cache_ram_of(args):\n    try:\n        return int(args[args.index(\'--cache-ram\') + 1])\n    except (ValueError, IndexError):\n        return None\n\ndef _stop_llama_servers():\n    for procdir in Path(\'/proc\').iterdir():\n        if not procdir.name.isdigit():\n            continue\n        try:\n            args = (procdir / \'cmdline\').read_bytes().decode(errors=\'replace\').split(\'\\0\')\n            if args and Path(args[0]).name == \'llama-server\':\n                os.kill(int(procdir.name), 15)\n                for _ in range(60):\n                    stat = (procdir / \'stat\').read_text() if procdir.exists() else \'\'\n                    if not stat or stat.split(\') \', 1)[1].startswith(\'Z\'):\n                        break\n                    time.sleep(0.5)\n                else:\n                    os.kill(int(procdir.name), 9)\n                    time.sleep(2)\n        except (FileNotFoundError, ProcessLookupError, PermissionError):\n            pass\n    old_proc = globals().get(\'qwen_server\')\n    if old_proc is not None:\n        try:\n            old_proc.wait(timeout=5)\n        except Exception:\n            pass\n    deadline = time.monotonic() + 20\n    while time.monotonic() < deadline:\n        alive = []\n        for procdir in Path(\'/proc\').iterdir():\n            try:\n                if procdir.name.isdigit() and Path((procdir / \'cmdline\').read_bytes().decode(errors=\'replace\').split(\'\\0\')[0]).name == \'llama-server\':\n                    alive.append(procdir.name)\n            except (FileNotFoundError, ProcessLookupError, PermissionError):\n                pass\n        if not alive:\n            return\n        time.sleep(1)\n    raise RuntimeError(\'旧 llama-server 停不掉（PID \' + \', \'.join(alive) + \'），已中止；请用停机按钮释放运行时后重来。\')\n\ndef _mem_available():\n    for l in Path(\'/proc/meminfo\').read_text().splitlines():\n        if l.startswith(\'MemAvailable:\'):\n            return int(l.split()[1]) * 1024\n    return 0\n\nSERVICE = \'svc:qwen\'\nSERVICE_FQDN = "qwen.<YOUR_TAILNET>.ts.net"\nSERVICE_TAG = \'tag:colab\'\nTS_HOSTNAME = SPEC[\'ts_hostname\']\nTS_CLIENT_ID = "<YOUR_TS_OAUTH_CLIENT_ID>"\nTS_CLIENT_SECRET = "<YOUR_TS_OAUTH_CLIENT_SECRET>"\nTS = \'/content/tsbin/tailscale\'\nTS_SOCK = \'/var/run/tailscale/tailscaled.sock\'\nTS_STATE = Path(\'/content/tailscaled.state\')\nTS_DIR = Path(\'/content/tsstatedir\')\n\ndef sh(c):\n    subprocess.run(c, shell=True, check=True)\n\ndef health():\n    try:\n        return requests.get(\'http://127.0.0.1:%d/health\' % PORT, timeout=2).status_code == 200\n    except Exception:\n        return False\n\ngpu_info = subprocess.getoutput(\'nvidia-smi --query-gpu=name,memory.total --format=csv,noheader,nounits\').strip()\nif not gpu_info or \',\' not in gpu_info:\n    raise RuntimeError(\'未检测到 NVIDIA GPU，请在右上角切换到 G4 GPU 运行时。输出：\' + gpu_info)\ngpu_name, gpu_mib_str = [x.strip() for x in gpu_info.split(\',\', 1)]\ngpu_mib = int(gpu_mib_str)\nprint(f\'检测到 GPU: {gpu_name} ({gpu_mib / 1024:.1f} GiB VRAM)\', flush=True)\nif gpu_mib < SPEC[\'min_vram_mib\']:\n    raise RuntimeError(f\'当前 GPU 显存仅 {gpu_mib / 1024:.1f} GiB（{gpu_name}），低于要求的 {SPEC["min_vram_mib"] / 1024:.0f} GiB，请切换到 G4 高内存运行时。\')\n\nROOT.mkdir(parents=True, exist_ok=True)\nfor _d in sorted(set(_local_path(f).parent for f in FILES)):\n    _d.mkdir(parents=True, exist_ok=True)\nDIST.mkdir(parents=True, exist_ok=True)\nKEY_FILE.write_text(KEY + \'\\n\', encoding=\'utf-8\', newline=\'\')\nKEY_FILE.chmod(0o600)\nAUTH = {\'Authorization\': \'Bearer \' + KEY}\n\ndef prepare_llama_cpp():\n    if next(DIST.rglob(\'llama-server\'), None) is not None:\n        return\n    print(f\'[并行任务] 下载并解压 llama.cpp {CPP_TAG} (CUDA 12.8) ...\', flush=True)\n    sh(f\'cd {DIST} && \'\n       f\'curl -fsSL -O {CPP_URL}/llama-{CPP_TAG}-bin-ubuntu-cuda-12.8-x64.tar.gz && \'\n       f\'curl -fsSL -O {CPP_URL}/cudart-llama-{CPP_TAG}-bin-ubuntu-cuda-12.8-x64.tar.gz && \'\n       f\'tar -xzf llama-{CPP_TAG}-bin-ubuntu-cuda-12.8-x64.tar.gz && \'\n       f\'tar -xzf cudart-llama-{CPP_TAG}-bin-ubuntu-cuda-12.8-x64.tar.gz && \'\n       f\'rm -f *.tar.gz\')\n    print(f\'[并行任务] llama.cpp {CPP_TAG} 就绪\', flush=True)\n\ndef ts(*a, timeout=60):\n    return subprocess.run([TS, \'--socket=\' + TS_SOCK, *a],\n                          stdin=subprocess.DEVNULL, capture_output=True, text=True, timeout=timeout)\n\ndef ts_state():\n    try:\n        return json.loads(ts(\'status\', \'--json\').stdout).get(\'BackendState\', \'\')\n    except Exception:\n        return \'\'\n\ndef _start_mcp_bridge():\n    srv = socket.socket(socket.AF_INET, socket.SOCK_STREAM)\n    srv.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)\n    try:\n        srv.bind((\'127.0.0.1\', 3426))\n    except OSError:\n        return\n    srv.listen(32)\n    def _pipe(a, b):\n        try:\n            while (d := a.recv(65536)):\n                b.sendall(d)\n        except Exception:\n            pass\n        finally:\n            try: a.close()\n            except Exception: pass\n            try: b.close()\n            except Exception: pass\n    while True:\n        try:\n            cli, _ = srv.accept()\n            up = socket.create_connection((\'127.0.0.1\', 1055), timeout=10)\n            up.sendall(bytes([5, 1, 0]))\n            assert up.recv(2) == bytes([5, 0])\n            up.sendall(bytes([5, 1, 0, 1]) + socket.inet_aton("<YOUR_LOCAL_TAILSCALE_IP>") + struct.pack(\'>H\', 3426))\n            assert up.recv(10)[1] == 0\n            threading.Thread(target=_pipe, args=(cli, up), daemon=True).start()\n            threading.Thread(target=_pipe, args=(up, cli), daemon=True).start()\n        except Exception:\n            pass\n\ndef prepare_tailscale():\n    if not Path(TS).exists():\n        sh(\'curl -fsSL https://pkgs.tailscale.com/stable/tailscale_1.104.1_amd64.tgz | tar -xz -C /content && \'\n           \'mv /content/tailscale_1.104.1_amd64 /content/tsbin\')\n    TS_DIR.mkdir(parents=True, exist_ok=True)\n    Path(\'/var/run/tailscale\').mkdir(parents=True, exist_ok=True)\n    if ts_state() in (\'\', \'NoState\', \'Stopped\'):\n        subprocess.Popen(\n            [\'/content/tsbin/tailscaled\', \'--tun=userspace-networking\',\n             \'--socks5-server=localhost:1055\', \'--state=\' + str(TS_STATE),\n             \'--statedir=\' + str(TS_DIR), \'--socket=\' + TS_SOCK],\n            stdout=open(\'/content/tailscaled.log\', \'a\', encoding=\'utf-8\', newline=\'\'),\n            stderr=subprocess.STDOUT, stdin=subprocess.DEVNULL, start_new_session=True,\n        )\n        for _ in range(40):\n            if Path(TS_SOCK).exists():\n                break\n            time.sleep(0.5)\n        for _ in range(40):\n            if ts_state() in (\'Running\', \'NeedsLogin\'):\n                break\n            time.sleep(0.5)\n    if ts_state() != \'Running\':\n        token = requests.post(\n            \'https://api.tailscale.com/api/v2/oauth/token\',\n            data={\'client_id\': TS_CLIENT_ID, \'client_secret\': TS_CLIENT_SECRET}, timeout=30\n        ).json()[\'access_token\']\n        key = requests.post(\n            \'https://api.tailscale.com/api/v2/tailnet/-/keys\',\n            headers={\'Authorization\': \'Bearer \' + token},\n            json={\'capabilities\': {\'devices\': {\'create\': {\n                \'reusable\': False, \'ephemeral\': True, \'preauthorized\': True, \'tags\': [SERVICE_TAG]\n            }}}}, timeout=30\n        ).json()[\'key\']\n        ts(\'up\', \'--auth-key=\' + key, \'--hostname=\' + TS_HOSTNAME,\n           \'--advertise-tags=\' + SERVICE_TAG, \'--accept-dns=false\', \'--timeout=60s\', timeout=120)\n        for _ in range(120):\n            if ts_state() == \'Running\':\n                break\n            time.sleep(0.5)\n    ts(\'serve\', \'reset\')\n    ts(\'serve\', \'--bg\', \'--service=\' + SERVICE, \'--tcp=443\', \'tcp://127.0.0.1:%d\' % PORT, timeout=90)\n    ts(\'serve\', \'--bg\', \'--https=443\', \'http://127.0.0.1:%d\' % PORT, timeout=90)\n    threading.Thread(target=_start_mcp_bridge, daemon=True).start()\n    print(f\'[并行任务] Tailscale Service ({SERVICE_FQDN}) 与本机小说文件 MCP 隧道已就绪\', flush=True)\n\nbg_executor = concurrent.futures.ThreadPoolExecutor(max_workers=2)\nfut_cpp = bg_executor.submit(prepare_llama_cpp)\nfut_ts = bg_executor.submit(prepare_tailscale)\n\nsubprocess.run([\'mount\', \'-o\', \'remount,size=115G\', \'/dev/shm\'], capture_output=True)\ntmpfs_dirs = []\nfor mline in Path(\'/proc/mounts\').read_text().splitlines():\n    mparts = mline.split()\n    if len(mparts) >= 3 and mparts[2] == \'tmpfs\' and mparts[1] in (\'/dev/shm\', \'/var/colab\'):\n        tmpfs_dirs.append(Path(mparts[1]))\nmp_free = {mp: shutil.disk_usage(mp).free for mp in tmpfs_dirs}\nmeminfo_txt = Path(\'/proc/meminfo\').read_text()\navail_kb = next((int(l.split()[1]) for l in meminfo_txt.splitlines() if l.startswith(\'MemAvailable:\')), 0)\nram_budget = max(0, avail_kb * 1024 - SPEC[\'ram_reserve_gib\'] * (1024**3))\n\nreleased_reuse = set()\nif RELEASE_STATE.exists():\n    try:\n        rel_state = json.loads(RELEASE_STATE.read_text(encoding=\'utf-8\'))\n        rel_ok = (isinstance(rel_state, dict) and isinstance(rel_state.get(\'pid\'), int)\n                  and isinstance(rel_state.get(\'files\'), list)\n                  and all(isinstance(n, str) and n in FILE_NAMES for n in rel_state[\'files\']))\n    except Exception:\n        rel_state, rel_ok = {}, False\n    pid_now = _server_pid()\n    if rel_ok and pid_now == rel_state[\'pid\'] and rel_state.get(\'proc_id\') and _proc_identity(pid_now) == rel_state[\'proc_id\']:\n        released_reuse = set(rel_state[\'files\'])\n        print(f\'服务仍在运行：{len(released_reuse)} 个已释放内存副本的分片无需重新下载\', flush=True)\n    else:\n        if not rel_ok:\n            print(\'⚠️ released-copies.json 无法识别，缺失的分片会重新下载\', flush=True)\n        RELEASE_STATE.unlink(missing_ok=True)\nif not released_reuse and _server_pid() and any(not _local_path(f).exists() for f in FILES):\n    # 记录不可用、服务还在且文件已删：先停服务再补下载，避免服务占着内存时又往内存盘塞几十 GiB。\n    print(\'⚠️ 服务在运行，但分片副本已删且释放记录不可用：先停掉旧服务，再重新下载。\', flush=True)\n    _stop_llama_servers()\n\nsorted_files = sorted(FILES, key=lambda x: x.get(\'priority\', 2))\nall_targets = [\n    (\'https://huggingface.co/\' + f[\'repo\'] + \'/resolve/\' + f[\'rev\'] + \'/\' + f[\'path\'],\n     _local_path(f), f[\'size\'])\n    for f in sorted_files\n]\ndownloads = []\ntmpfs_bytes = 0\ndisk_bytes = 0\nfor orig_url, logical_path, size in all_targets:\n    if logical_path.name in released_reuse:\n        continue\n    if logical_path.exists() and logical_path.stat().st_size == size:\n        continue\n    if logical_path.is_symlink() or logical_path.exists():\n        logical_path.unlink(missing_ok=True)\n    real_path = logical_path\n    if ram_budget >= size:\n        for mp in tmpfs_dirs:\n            if mp_free.get(mp, 0) >= size + 4 * (1024**3):\n                cand = mp / SPEC[\'slug\'] / logical_path.name\n                cand.parent.mkdir(parents=True, exist_ok=True)\n                # 只看大小会把 truncate 出来的稀疏占位文件（下载中断、上次报错）当成已下载；tmpfs 的 st_blocks 是实际占用。\n                if cand.exists() and cand.stat().st_size == size and cand.stat().st_blocks * 512 >= size:\n                    logical_path.symlink_to(cand)\n                    mp_free[mp] -= size\n                    ram_budget -= size\n                    real_path = None\n                    break\n                cand.unlink(missing_ok=True)\n                with cand.open(\'wb\') as f:\n                    f.truncate(size)\n                logical_path.symlink_to(cand)\n                mp_free[mp] -= size\n                ram_budget -= size\n                tmpfs_bytes += size\n                real_path = cand\n                break\n    if real_path is None:\n        continue\n    if real_path == logical_path:\n        free_disk = shutil.disk_usage(\'/content\').free\n        if free_disk < disk_bytes + size + 6 * (1024**3):\n            raise RuntimeError(\'本地 /content 与 tmpfs 空间不足，请重建运行时。\')\n        with real_path.open(\'wb\') as f:\n            f.truncate(size)\n        disk_bytes += size\n    downloads.append((orig_url, str(real_path), size))\n\nfreshly_downloaded = bool(downloads)\nif downloads:\n    need_bytes = sum(size for _, _, size in downloads)\n    print(f\'待并发下载: {need_bytes / (1024**3):.2f} GiB ({len(downloads)} 个文件) → DDR5 内存盘 tmpfs: {tmpfs_bytes / (1024**3):.1f} GiB | 本地磁盘: {disk_bytes / (1024**3):.1f} GiB\', flush=True)\n    CHUNK = 64 * 1024 * 1024\n    hf_tok = os.environ.get(\'HF_TOKEN\', \'\').strip()\n    if not hf_tok:\n        try:\n            from google.colab import userdata\n            hf_tok = (userdata.get(\'HF_TOKEN\') or \'\').strip()\n        except Exception:\n            hf_tok = \'\'\n\n    def _resolve_cdn(item):\n        from email.utils import parsedate_to_datetime\n        from datetime import datetime, timezone\n        orig_url, real_path, size = item\n        req_hdr = {\'Range\': \'bytes=0-0\'}\n        if hf_tok:\n            req_hdr[\'Authorization\'] = \'Bearer \' + hf_tok\n        for attempt in range(5):\n            try:\n                with requests.get(orig_url, headers=req_hdr, stream=True, timeout=30) as r:\n                    if r.status_code == 429:\n                        ra = r.headers.get(\'Retry-After\', \'\')\n                        try:\n                            wait_s = float(ra)\n                        except ValueError:\n                            try:\n                                wait_s = max(1.0, (parsedate_to_datetime(ra) - datetime.now(timezone.utc)).total_seconds())\n                            except Exception:\n                                wait_s = 2 ** (attempt + 1)\n                        time.sleep(min(wait_s, 60.0))\n                        continue\n                    r.raise_for_status()\n                    if r.status_code != 206:\n                        raise RuntimeError(f\'Range probe expected 206, got {r.status_code}\')\n                    cdn_url = r.url\n                    return [(cdn_url, orig_url, real_path, start, min(start + CHUNK, size) - 1)\n                            for start in range(0, size, CHUNK)]\n            except Exception:\n                if attempt == 4:\n                    raise\n                time.sleep(2 ** (attempt + 1))\n\n    range_tasks = []\n    with concurrent.futures.ThreadPoolExecutor(max_workers=len(downloads)) as rpool:\n        for chunk_list in rpool.map(_resolve_cdn, downloads):\n            range_tasks.extend(chunk_list)\n\n    tasks_file = Path(\'/tmp/colab_kernel_dl_tasks.json\')\n    tasks_file.write_text(json.dumps({\'need_bytes\': need_bytes, \'tasks\': range_tasks}), encoding=\'utf-8\', newline=\'\')\n    dl_script = Path(\'/tmp/colab_kernel_fast_dl.py\')\n    dl_script.write_text(\'\'\'import concurrent.futures, json, multiprocessing as mp, os, sys, time\nimport requests\nfrom requests.adapters import HTTPAdapter\n\ndata = json.loads(open(sys.argv[1], encoding="utf-8").read())\nneed_bytes = data["need_bytes"]\ntasks = data["tasks"]\nN_PROC = 8\nN_THREADS = 4\n\ndone_bytes = mp.Value("q", 0)\nt_start = time.monotonic()\nlast_print = mp.Value("d", t_start)\n\ndef worker_batch(batch):\n    sess = requests.Session()\n    adapter = HTTPAdapter(pool_connections=N_THREADS + 4, pool_maxsize=N_THREADS + 4)\n    sess.mount("https://", adapter)\n    sess.mount("http://", adapter)\n\n    def fetch_one(task):\n        cdn_url, orig_url, target, start, end = task\n        expected = end - start + 1\n        for attempt in range(5):\n            url = cdn_url if attempt < 2 else orig_url\n            try:\n                with sess.get(url, headers={"Range": f"bytes={start}-{end}"}, stream=True, timeout=(15, 120)) as resp:\n                    if resp.status_code != 206:\n                        raise RuntimeError(f"HTTP {resp.status_code}")\n                    fd = os.open(target, os.O_RDWR)\n                    try:\n                        pos = start\n                        for blk in resp.iter_content(8 * 1024 * 1024):\n                            if blk:\n                                os.pwrite(fd, blk, pos)\n                                pos += len(blk)\n                    finally:\n                        os.close(fd)\n                    if pos - start != expected:\n                        raise RuntimeError(f"short read {pos - start} != {expected}")\n                with done_bytes.get_lock():\n                    done_bytes.value += expected\n                    now = time.monotonic()\n                    if now - last_print.value >= 8:\n                        rate = done_bytes.value / max(now - t_start, 0.1) / (1024**2)\n                        print(f"  并发下载进度: {done_bytes.value / need_bytes:.0%} ({done_bytes.value / (1024**3):.1f} / {need_bytes / (1024**3):.1f} GiB)，均速 {rate:.0f} MB/s", flush=True)\n                        last_print.value = now\n                return\n            except Exception as err:\n                if attempt == 4:\n                    raise RuntimeError(f"{os.path.basename(target)} [{start}-{end}] 失败: {err}") from err\n                time.sleep(1.5 * (attempt + 1))\n\n    with concurrent.futures.ThreadPoolExecutor(max_workers=N_THREADS) as tpool:\n        futs = [tpool.submit(fetch_one, t) for t in batch]\n        for f in concurrent.futures.as_completed(futs):\n            f.result()\n\nif __name__ == "__main__":\n    batches = [tasks[i::N_PROC] for i in range(N_PROC) if tasks[i::N_PROC]]\n    ctx = mp.get_context("fork")\n    with ctx.Pool(processes=len(batches)) as pool:\n        pool.map(worker_batch, batches)\n\'\'\', encoding=\'utf-8\', newline=\'\')\n    t_dl_start = time.monotonic()\n    subprocess.run([sys.executable, str(dl_script), str(tasks_file)], check=True)\n    dl_sec = max(time.monotonic() - t_dl_start, 0.1)\n    print(f\'全部权重下载完成：用时 {dl_sec:.1f}s，平均 {need_bytes / (1024**2) / dl_sec:.0f} MB/s\', flush=True)\nelse:\n    print(f\'全部 {len(FILES)} 个权重与附件已存在于内存盘/本地，跳过下载。\', flush=True)\n\ndef verify_hashes():\n    # 仅调试模式调用。下载决策时已经对过一次字节大小；这里再 stat 一遍。\n    try:\n        for item in FILES:\n            path = _local_path(item)\n            if path.name in released_reuse:\n                continue\n            actual = path.stat().st_size if path.exists() else -1\n            if actual != item[\'size\']:\n                if path.exists():\n                    path.resolve().unlink(missing_ok=True)\n                    path.unlink(missing_ok=True)\n                raise RuntimeError(\'权重存在性或字节大小校验失败，请重新运行首格：\' + path.name)\n            print(\'SIZE OK:\', path.name, item[\'size\'], flush=True)\n    except BaseException:\n        if globals().get(\'qwen_server\') is not None:\n            qwen_server.terminate()\n        raise\n\nPARALLEL_SLOTS = 1\nSLOT_CONTEXT_TOKENS = 262144\nTOTAL_CONTEXT_TOKENS = SLOT_CONTEXT_TOKENS * PARALLEL_SLOTS\nUBATCH_SIZE = 2048\nMAX_OUTPUT_TOKENS = 65536\nSLOT_SAVE_DIR = ROOT / \'slot-saves\'\nSLOT_SAVE_DIR.mkdir(parents=True, exist_ok=True)\nCHAT_TMPL = ROOT / \'chat-template-clean.jinja\'\nUI_CFG = ROOT / \'webui-config.json\'\nnew_ui_text = json.dumps(UI_CFG_DATA, ensure_ascii=False, indent=2) + \'\\n\'\nold_ui_on_disk = UI_CFG.read_text(encoding=\'utf-8\') if UI_CFG.exists() else \'\'\n\nple_name = SPEC.get(\'ple_shard_name\')\nif ple_name:\n    ple_real = str((SHARD_DIR / ple_name).resolve())\n    ple_in_tmpfs = ple_real.startswith(TMPFS_PREFIXES)\nelse:\n    ple_in_tmpfs = False\n\n# PLE 表分片在运行期由 CPU 直接 mmap 读取，必须常驻；其余落在内存盘的分片加载后可删。\nrelease_keep = {ple_name} if ple_in_tmpfs else set()\nrelease_candidates = [f for f in FILES\n                      if _local_path(f).name not in release_keep\n                      and _local_path(f).exists()\n                      and str(_local_path(f).resolve()).startswith(TMPFS_PREFIXES)]\nreleasable_bytes = sum(f[\'size\'] for f in release_candidates)\n# 只用于新启动的服务；已在运行的服务保留自己的 --cache-ram（configured_server 不比较这一项，免得为改缓存上限重启）。\n# 只拿回收内存的一半给提示缓存，另一半留作余量（mmap 模式下 CPU 张量所在分片删文件后仍占内存）。\nPROMPT_CACHE_MIB = min(65536, 16384 + releasable_bytes // (2 * 1024**2))\n\ndef release_gpu_only_copies():\n    pid, args = _server_proc()\n    todo = [f for f in release_candidates if _local_path(f).exists()]\n    if not pid or not todo or not health():\n        return\n    names = sorted({_local_path(f).name for f in todo} | released_reuse)\n    tmp_state = RELEASE_STATE.with_suffix(\'.tmp\')\n    tmp_state.write_text(json.dumps({\'pid\': pid, \'proc_id\': _proc_identity(pid), \'files\': names}), encoding=\'utf-8\', newline=\'\')\n    os.replace(tmp_state, RELEASE_STATE)\n    if DEBUG_VERIFY:\n        before = _mem_available()\n    for f in todo:\n        lp = _local_path(f)\n        lp.resolve().unlink(missing_ok=True)\n        lp.unlink(missing_ok=True)\n    expect = sum(f[\'size\'] for f in todo)\n    if DEBUG_VERIFY:\n        time.sleep(1)\n        gained = _mem_available() - before\n        cap = _cache_ram_of(args) or 0\n        print(f\'已删除 {len(todo)} 个显存专用分片的内存盘副本：可用内存 +{gained / 1024**3:.1f} GiB（预计 {expect / 1024**3:.1f} GiB）| 当前服务提示缓存上限 {cap / 1024:.0f} GiB\', flush=True)\n        if gained < expect * 0.5:\n            print(\'⚠️ 实际回收不到预计的一半：部分分片仍被服务映射，提示缓存别塞满；服务重启后才会真正归还。\', flush=True)\n    else:\n        print(f\'已删除 {len(todo)} 个显存专用分片的内存盘副本（预计 {expect / 1024**3:.1f} GiB）\', flush=True)\n\nif MODEL.exists():\n    with MODEL.open(\'rb\') as f:\n        raw_head = f.read(16 * 1024 * 1024)\n    HEAD_CACHE.write_bytes(raw_head)\nelse:\n    raw_head = HEAD_CACHE.read_bytes()\ntmpl_key = b\'tokenizer.chat_template\'\nkey_pos = raw_head.index(struct.pack(\'<Q\', len(tmpl_key)) + tmpl_key)\nval_pos = key_pos + 8 + len(tmpl_key)\nassert struct.unpack_from(\'<I\', raw_head, val_pos)[0] == 8, \'tokenizer.chat_template 不是 GGUF 字符串类型\'\ntmpl_len = struct.unpack_from(\'<Q\', raw_head, val_pos + 4)[0]\nassert val_pos + 12 + tmpl_len <= len(raw_head), \'模板超出前 16 MiB\'\ntmpl_str = raw_head[val_pos + 12:val_pos + 12 + tmpl_len].decode(\'utf-8\')\n\n# 官方模板只有 xhigh/medium/low 三档；换成四档，并按 effort_prompts 在 system 顶部注入一句中文引导。\nuniversal_effort_block = \'\'\'{%- set reasoning_instructions = \'\' %}\n{%- if enable_thinking is undefined or enable_thinking is true %}\n    {%- set resolved_reasoning_effort = reasoning_effort|default(\'@default@\') %}\n    {%- if resolved_reasoning_effort not in (\'xhigh\', \'high\', \'medium\', \'low\') %}\n        {{- raise_exception(\'Unexpected reasoning effort \' ~ reasoning_effort ~ \'. Supported types are xhigh, high, medium, and low (default @default@).\') }}\n    {%- endif %}\n    {%- if resolved_reasoning_effort == \'xhigh\' %}\n        {%- set reasoning_instructions = \'@xhigh@\' %}\n    {%- elif resolved_reasoning_effort == \'high\' %}\n        {%- set reasoning_instructions = \'@high@\' %}\n    {%- elif resolved_reasoning_effort == \'medium\' %}\n        {%- set reasoning_instructions = \'@medium@\' %}\n    {%- elif resolved_reasoning_effort == \'low\' %}\n        {%- set reasoning_instructions = \'@low@\' %}\n    {%- endif %}\n{%- endif %}\'\'\'\nfor _lvl, _txt in SPEC[\'effort_prompts\'].items():\n    universal_effort_block = universal_effort_block.replace(\'@%s@\' % _lvl, _txt)\nuniversal_effort_block = universal_effort_block.replace(\'@default@\', SPEC.get(\'reasoning_effort_default\') or \'high\')\n\nfam = SPEC.get(\'template_family\', \'qwen38_fn\')\nif fam == \'qwen38_fn\':\n    old_effort_block = \'\'\'{%- if enable_thinking is undefined or enable_thinking is true %}\n    {%- set resolved_reasoning_effort = reasoning_effort|default(\'xhigh\') %}\n    {%- if resolved_reasoning_effort not in (\'xhigh\', \'medium\', \'low\') %}\n        {{- raise_exception(\'Unexpected reasoning effort \' ~ reasoning_effort ~ \'. Supported types are xhigh (default), medium, and low.\') }}\n    {%- endif %}\n    {%- if resolved_reasoning_effort == \'xhigh\' %}\n        {%- set reasoning_instructions = \'Reasoning effort is set to xhigh. Please think carefully through the task, validate key assumptions, consider plausible alternatives, and prioritize correctness, consistency, and clarity in the final answer.\' %}\n    {%- elif resolved_reasoning_effort == \'low\' %}\n        {%- set reasoning_instructions = \'Reasoning effort is set to low. Keep your thinking brief and focused, moving directly to the conclusion without unnecessary elaboration.\' %}\n    {%- endif %}\n{%- endif %}\'\'\'\n    # Unsloth 导出的原版 Qwen3.8 模板在 resolved_reasoning_effort 下多塞了 3 行 high->xhigh 折合；先剥掉再统一替换成四档中文引导。\n    unsloth_high_shim = \'\'\'    {%- if resolved_reasoning_effort == \'high\' %}\n        {%- set resolved_reasoning_effort = \'xhigh\' %}\n    {%- endif %}\n\'\'\'\n    tmpl_str = tmpl_str.replace(unsloth_high_shim, \'\', 1)\n    assert old_effort_block in tmpl_str, \'未找到预期的 reasoning_effort 模板块\'\n    tmpl_str = tmpl_str.replace(old_effort_block, universal_effort_block, 1)\n    old_preserve_line = \'{%- if preserve_thinking is undefined or preserve_thinking is true or loop.index0 > ns.last_query_index %}\'\n    assert old_preserve_line in tmpl_str, \'未找到预期的 preserve_thinking 模板行\'\n    # 官方默认 preserve_thinking 未定义即为真，历史轮次会被套上空 <think></think>。\n    # 只保留当前工具链里 reasoning_content 非空的轮次，避免 few-shot 诱导模型秒关思考。\n    # 生成起手块不改：官方只在 enable_thinking=false 时关思考，low 照常思考。\n    tmpl_str = tmpl_str.replace(\n        old_preserve_line,\n        \'{%- if loop.index0 > ns.last_query_index and reasoning_content %}\',\n        1,\n    )\nelif fam == \'gemma4\':\n    old_think_body = \'\'\'        {{- \'<|think|>\\\\n\' -}}\n        {%- set ns.prev_message_type = \'think\' -%}\'\'\'\n    new_think_body = \'\'\'        {{- \'<|think|>\\\\n\' + (reasoning_instructions + \'\\\\n\' if reasoning_instructions else \'\') -}}\n        {%- set ns.prev_message_type = \'think\' -%}\'\'\'\n    assert old_think_body in tmpl_str, \'未找到预期的 gemma4 <|think|> 注入块\'\n    tmpl_str = tmpl_str.replace(\n        \'{%- set enable_thinking = enable_thinking | default(false) -%}\',\n        \'{%- set enable_thinking = enable_thinking | default(true) -%}\',\n        1,\n    )\n    tmpl_str = (\n        "{%- if enable_thinking is not defined %}{%- set enable_thinking = true %}{%- endif %}\\n"\n        + universal_effort_block\n        + "\\n"\n        + tmpl_str.replace(old_think_body, new_think_body, 1)\n    )\nelif fam == \'glm5_next\':\n    old_glm_head = \'\'\'{%- set effective_reasoning_effort = reasoning_effort if reasoning_effort is defined and reasoning_effort in [\'low\', \'high\'] else \'max\' -%}\n{%- if effective_reasoning_effort is not none -%}<|system|>Reasoning Effort: {{ effective_reasoning_effort | capitalize }}{%- endif -%}\n{%- set clear_thinking = clear_thinking if clear_thinking is defined else false -%}\'\'\'\n    new_glm_head = (\n        universal_effort_block\n        + "\\n{%- set effective_reasoning_effort = \'max\' if resolved_reasoning_effort == \'xhigh\' else (\'low\' if resolved_reasoning_effort == \'low\' else \'high\') -%}"\n        + "\\n{%- if effective_reasoning_effort is not none -%}<|system|>Reasoning Effort: {{ effective_reasoning_effort | capitalize }}{{ (\'\\\\n\' + reasoning_instructions) if reasoning_instructions else \'\' }}{%- endif -%}"\n        + "\\n{%- set clear_thinking = clear_thinking if clear_thinking is defined else true -%}"\n    )\n    assert old_glm_head in tmpl_str, \'未找到预期的 glm5_next reasoning_effort 模板块\'\n    tmpl_str = tmpl_str.replace(old_glm_head, new_glm_head, 1)\nelif fam == \'qwen35\':\n    old_tools_head = "{%- if tools and tools is iterable and tools is not mapping %}\\n    {{- \'<|im_start|>system\\\\n\' }}"\n    new_tools_head = (\n        universal_effort_block\n        + "\\n{%- if tools and tools is iterable and tools is not mapping %}\\n"\n        + "    {{- \'<|im_start|>system\\\\n\' }}\\n"\n        + "    {%- if reasoning_instructions %}\\n"\n        + "        {{- reasoning_instructions + \'\\\\n\\\\n\' }}\\n"\n        + "    {%- endif %}"\n    )\n    assert old_tools_head in tmpl_str, \'未找到预期的 qwen35 tools system 头部\'\n    tmpl_str = tmpl_str.replace(old_tools_head, new_tools_head, 1)\n    old_tools_sys = \'\'\'    {%- if messages[0].role == \'system\' %}\n        {%- set content = render_content(messages[0].content, false, true)|trim %}\n        {%- if content %}\n            {{- \'\\\\n\\\\n\' + content }}\n        {%- endif %}\n    {%- endif %}\n    {{- \'<|im_end|>\\\\n\' }}\n{%- else %}\n    {%- if messages[0].role == \'system\' %}\n        {%- set content = render_content(messages[0].content, false, true)|trim %}\n        {{- \'<|im_start|>system\\\\n\' + content + \'<|im_end|>\\\\n\' }}\n    {%- endif %}\n{%- endif %}\'\'\'\n    new_tools_sys = \'\'\'    {%- if messages[0].role == \'system\' %}\n        {%- set content = render_content(messages[0].content, false, true)|trim %}\n        {%- if content %}\n            {{- \'\\\\n\\\\n\' + content }}\n        {%- endif %}\n    {%- endif %}\n    {{- \'<|im_end|>\\\\n\' }}\n{%- else %}\n    {%- if messages[0].role == \'system\' %}\n        {%- set content = render_content(messages[0].content, false, true)|trim %}\n        {{- \'<|im_start|>system\\\\n\' + (reasoning_instructions + \'\\\\n\\\\n\' if reasoning_instructions else \'\') + content + \'<|im_end|>\\\\n\' }}\n    {%- elif reasoning_instructions %}\n        {{- \'<|im_start|>system\\\\n\' + reasoning_instructions + \'<|im_end|>\\\\n\' }}\n    {%- endif %}\n{%- endif %}\'\'\'\n    assert old_tools_sys in tmpl_str, \'未找到预期的 qwen35 system 分支块\'\n    tmpl_str = tmpl_str.replace(old_tools_sys, new_tools_sys, 1)\n    old_hist_think = \'{%- if loop.index0 > ns.last_query_index %}\'\n    assert old_hist_think in tmpl_str, \'未找到预期的历史思考保留条件\'\n    tmpl_str = tmpl_str.replace(old_hist_think, \'{%- if loop.index0 > ns.last_query_index and reasoning_content %}\', 1)\n    qwen_gen_think_old = \'\'\'    {%- if enable_thinking is defined and enable_thinking is false %}\n        {{- \'<think>\\\\n\\\\n</think>\\\\n\\\\n\' }}\n    {%- else %}\n        {{- \'<think>\\\\n\' }}\n    {%- endif %}\'\'\'\n    qwen_gen_think_new = \'\'\'    {%- if (enable_thinking is defined and enable_thinking is false) or (reasoning_effort is defined and reasoning_effort == \'low\') %}\n        {{- \'<think>\\\\n\\\\n</think>\\\\n\\\\n\' }}\n    {%- else %}\n        {{- \'<think>\\\\n\' }}\n    {%- endif %}\'\'\'\n    assert qwen_gen_think_old in tmpl_str, \'未找到预期的 add_generation_prompt 思考起手块\'\n    tmpl_str = tmpl_str.replace(qwen_gen_think_old, qwen_gen_think_new, 1)\nelif fam == \'nemotron\':\n    old_trunc = \'{%- set truncate_history_thinking = truncate_history_thinking if truncate_history_thinking is defined else True %}\'\n    assert old_trunc in tmpl_str, \'未找到预期的 truncate_history_thinking 行\'\n    tmpl_str = tmpl_str.replace(old_trunc, \'{%- set truncate_history_thinking = false %}\', 1)\n    old_sys_block = \'\'\'{%- if messages[0]["role"] == "system" %}\n    {%- set system_message = messages[0]["content"] %}\n    {%- set loop_messages = messages[1:] %}\n{%- else %}\n    {%- set system_message = "" %}\n    {%- set loop_messages = messages %}\n{%- endif %}\'\'\'\n    new_sys_block = (\n        universal_effort_block\n        + \'\'\'\n{%- if messages[0]["role"] == "system" %}\n    {%- set system_message = (reasoning_instructions ~ "\\\\n\\\\n" if reasoning_instructions else "") ~ render_content(messages[0]["content"]) %}\n    {%- set loop_messages = messages[1:] %}\n{%- else %}\n    {%- set system_message = reasoning_instructions %}\n    {%- set loop_messages = messages %}\n{%- endif %}\'\'\'\n    )\n    assert old_sys_block in tmpl_str, \'未找到预期的 nemotron system_message 块\'\n    tmpl_str = tmpl_str.replace(old_sys_block, new_sys_block, 1)\n    nemo_gen_think_old = \'\'\'{%- if add_generation_prompt %}\n    {%- if enable_thinking %}\n        {{- \'<|im_start|>assistant\\\\n<think>\\\\n\' }}\n    {%- else %}\n        {{- \'<|im_start|>assistant\\\\n<think></think>\' }}\n    {%- endif %}\n{%- endif %}\'\'\'\n    nemo_gen_think_new = \'\'\'{%- if add_generation_prompt %}\n    {{- \'<|im_start|>assistant\\\\n\' }}\n    {%- if (enable_thinking is defined and enable_thinking is false) or (reasoning_effort is defined and reasoning_effort == \'low\') %}\n        {{- \'<think></think>\' }}\n    {%- else %}\n        {{- \'<think>\\\\n\' }}\n    {%- endif %}\n{%- endif -%}\'\'\'\n    assert nemo_gen_think_old in tmpl_str, \'未找到预期的 nemotron add_generation_prompt 思考起手块\'\n    tmpl_str = tmpl_str.replace(nemo_gen_think_old, nemo_gen_think_new, 1)\nelse:\n    raise RuntimeError(f\'未知 template_family={fam}：必须在 colab_kernel.py 中接入 universal_effort_block 四档思考支持\')\n\nif LORA and LORA.exists():\n    # GCSA-AiLab/GLM-5.3-Flash-Abliterated-LoRA-GGUF 头部写成了 glm5next（缺连字符），原地修补为 glm5-next 并消掉 1 字节对齐填充，保持张量偏移与文件大小不变。\n    with LORA.open(\'r+b\') as _lf:\n        _lbuf = _lf.read(32768)\n        _bad_arch = b\'\\x08\\x00\\x00\\x00\\x00\\x00\\x00\\x00glm5next\'\n        _good_arch = b\'\\x09\\x00\\x00\\x00\\x00\\x00\\x00\\x00glm5-next\'\n        if _bad_arch in _lbuf:\n            _pad_idx = _lbuf.find(b\'\\x00\' * 8, 16384)\n            assert _pad_idx > 0, \'LoRA GGUF 未找到对齐填充区\'\n            _lbuf = _lbuf.replace(_bad_arch, _good_arch, 1)\n            _lbuf = _lbuf[:_pad_idx] + _lbuf[_pad_idx + 1:]\n            _lf.seek(0)\n            _lf.write(_lbuf)\nold_tmpl_on_disk = CHAT_TMPL.read_text(encoding=\'utf-8\') if CHAT_TMPL.exists() else \'\'\nCHAT_TMPL.write_text(tmpl_str, encoding=\'utf-8\', newline=\'\')\nUI_CFG.write_text(new_ui_text, encoding=\'utf-8\', newline=\'\')\n\ndef _fp_args(args):\n    # 指纹跳过二进制路径和 --cache-ram（只改缓存上限不重启）。\n    out = []\n    i = 1 if args and Path(args[0]).name == \'llama-server\' else 0\n    while i < len(args):\n        a = args[i]\n        if not a:\n            i += 1\n            continue\n        if a == \'--cache-ram\':\n            i += 2\n            continue\n        out.append(a)\n        i += 1\n    return out\n\ndef _boot_fp(args, tmpl, ui):\n    blob = chr(0).join(_fp_args(args)) + chr(0) + tmpl + chr(0) + ui\n    return hashlib.sha1(blob.encode()).hexdigest()\n\nload_flags = [\'--lazy-mode\', \'off\'] if ple_in_tmpfs else [\'-lm\', \'none\', \'--lazy-mode\', \'off\']\nserver_args = [\'-m\', str(MODEL)]\nif MMPROJ:\n    server_args += [\'--mmproj\', str(MMPROJ)]\nif DRAFT and SPEC.get(\'mtp_n_max\'):\n    server_args += [\'--model-draft\', str(DRAFT), \'--spec-type\', \'draft-mtp\',\n                    \'--spec-draft-n-max\', str(SPEC[\'mtp_n_max\']),\n                    \'--spec-draft-p-min\', str(SPEC[\'mtp_p_min\'])]\nif LORA:\n    server_args += [\'--lora\', str(LORA)]\nif SPEC.get(\'extra_server_flags\'):\n    server_args += list(SPEC[\'extra_server_flags\'])\nserver_args += [\n    \'--slot-save-path\', str(SLOT_SAVE_DIR),\n    \'--device\', \'CUDA0\',\n    \'-ngl\', \'99\',\n    *load_flags,\n    \'--cache-ram\', str(PROMPT_CACHE_MIB),\n    \'-c\', str(TOTAL_CONTEXT_TOKENS),\n    \'--parallel\', str(PARALLEL_SLOTS),\n    \'--host\', \'127.0.0.1\',\n    \'--port\', str(PORT),\n    \'--alias\', MODEL_ALIAS,\n    \'--jinja\',\n    \'--chat-template-file\', str(CHAT_TMPL),\n    \'--ui-config-file\', str(UI_CFG),\n    \'--ui-mcp-proxy\',\n    \'--reasoning\', \'on\',\n]\nif SPEC.get(\'reasoning_effort_default\'):\n    server_args += [\'--reasoning-effort\', SPEC[\'reasoning_effort_default\']]\nserver_args += [\n    \'--temp\', SPEC[\'temp\'],\n    \'--top-p\', SPEC[\'top_p\'],\n    \'--top-k\', SPEC[\'top_k\'],\n    \'--min-p\', SPEC[\'min_p\'],\n    *SPEC.get(\'penalty_args\', []),\n    \'--flash-attn\', \'on\',\n    \'--cache-type-k\', \'q8_0\',\n    \'--cache-type-v\', \'q8_0\',\n    \'--batch-size\', \'4096\',\n    \'--ubatch-size\', str(UBATCH_SIZE),\n    \'--predict\', str(MAX_OUTPUT_TOKENS),\n]\nexpected_fp = _boot_fp(server_args, tmpl_str, new_ui_text)\n\ndef configured_server():\n    # 已在跑且启动指纹一致就跳过重启。指纹 = sha1(argv 去掉二进制和 --cache-ram + 模板 + webui-config)。\n    if not health():\n        return False\n    _pid, args = _server_proc()\n    if not args or _boot_fp(args, old_tmpl_on_disk, old_ui_on_disk) != expected_fp:\n        return False\n    if DEBUG_VERIFY:\n        props = requests.get(\'http://127.0.0.1:%d/props\' % PORT, headers=AUTH, timeout=10).json()\n        if props.get(\'default_generation_settings\', {}).get(\'n_ctx\') != SLOT_CONTEXT_TOKENS:\n            return False\n        if props.get(\'total_slots\') != PARALLEL_SLOTS:\n            return False\n        if props.get(\'chat_template_caps\', {}).get(\'reasoning_effort_levels\') != [\'low\', \'medium\', \'high\', \'xhigh\']:\n            return False\n    return True\n\nserver_just_started = False\nif health() and not configured_server():\n    print(\'检测到旧配置或不同模型服务：重启 llama-server ...\', flush=True)\n    if not released_reuse:\n        _stop_llama_servers()\n\nif configured_server():\n    print(f\'模型已在运行：{MODEL_ALIAS} / 单路 × {SLOT_CONTEXT_TOKENS} 上下文 / {(_cache_ram_of(_server_proc()[1]) or 0) / 1024:.0f} GiB Prompt Cache\', flush=True)\nelse:\n    missing_now = [_local_path(f).name for f in FILES if not _local_path(f).exists()]\n    if released_reuse or missing_now:\n        _stop_llama_servers()\n        RELEASE_STATE.unlink(missing_ok=True)\n        raise RuntimeError(\'需要重新启动服务，但缺少上次为扩大提示缓存而删掉的分片副本：\'\n                           + \', \'.join(sorted(set(missing_now) | released_reuse))\n                           + \'。旧服务已停，请再运行一次本格，会重新下载这些分片。\')\n    _stop_llama_servers()\n    if ple_name and not freshly_downloaded and not ple_in_tmpfs:\n        meminfo = Path(\'/proc/meminfo\').read_text()\n        cached_kb = next((int(l.split()[1]) for l in meminfo.splitlines() if l.startswith(\'Cached:\')), 0)\n        if cached_kb < 40 * 1024 * 1024:\n            print(\'预热 PLE 表所在分片到主机内存 Page Cache ...\', flush=True)\n            with (SHARD_DIR / ple_name).open(\'rb\') as f:\n                while f.read(32 * 1024 * 1024):\n                    pass\n\n    fut_cpp.result()\n    SERVER = next(DIST.rglob(\'llama-server\'))\n    env = os.environ.copy()\n    libdirs = {str(SERVER.parent)}\n    for pat in (\'libcudart.so.12\', \'libcublas.so.12\', \'libcublasLt.so.12\'):\n        libdirs.update(str(p.parent) for p in DIST.rglob(pat))\n    env[\'LD_LIBRARY_PATH\'] = \':\'.join(sorted(libdirs)) + \':\' + env.get(\'LD_LIBRARY_PATH\', \'\')\n    cmd = [str(SERVER), *server_args]\n    log = (ROOT / \'server.log\').open(\'a\', encoding=\'utf-8\', newline=\'\')\n    log.write(chr(10) + \'=== START %s ===\' % time.strftime(\'%Y-%m-%d %H:%M:%S\') + chr(10))\n    log.flush()\n    proc = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT, env=env, stdin=subprocess.DEVNULL, start_new_session=True)\n    qwen_server = proc\n    t0 = time.monotonic()\n    # LAUNCH_DETACHED=1（colab_cli launch）：进程已 start_new_session，不等本机 /health，交给后面的后台线程。\n    if os.environ.get(\'LAUNCH_DETACHED\') == \'1\':\n        print(\'LAUNCH_DETACHED: llama-server pid %s\' % proc.pid, flush=True)\n        server_just_started = True\n    else:\n        print(f\'加载 {MODEL_ALIAS} 到 GPU ...\', flush=True)\n        while time.monotonic() - t0 < 600:\n            if proc.poll() is not None:\n                raise RuntimeError((ROOT / \'server.log\').read_text(errors=\'replace\')[-3500:])\n            if health():\n                break\n            time.sleep(2 if DEBUG_VERIFY else 0.5)\n        else:\n            proc.terminate()\n            raise RuntimeError(f\'模型加载超时，请查看 {ROOT}/server.log\')\n        server_just_started = True\n\ndef _warmup_cuda():\n    # 第一次推理会捕获 CUDA graph，短提示通常要数秒到十几秒，不是零点几秒；放后台不挡就绪横幅。只发这一次 max_tokens=4。\n    requests.post(\'http://127.0.0.1:%d/v1/chat/completions\' % PORT, headers=AUTH, timeout=600,\n                  json={\'model\': MODEL_ALIAS,\n                        \'messages\': [{\'role\': \'user\', \'content\': \'请只回复“就绪”两个字。\'}],\n                        \'max_tokens\': 4, \'temperature\': 0, \'reasoning_effort\': \'low\'})\n\nif DEBUG_VERIFY:\n    if server_just_started:\n        print(\'服务就绪 %.0fs，核验四档思考能力并发送预热请求初始化 CUDA 图 ...\' % (time.monotonic() - t0), flush=True)\n        props_now = requests.get(\'http://127.0.0.1:%d/props\' % PORT, headers=AUTH, timeout=10).json()\n        levels_now = props_now.get(\'chat_template_caps\', {}).get(\'reasoning_effort_levels\', [])\n        if levels_now != [\'low\', \'medium\', \'high\', \'xhigh\']:\n            proc.terminate()\n            raise RuntimeError(f\'思考档位探测未达四档标准 (期望 low/medium/high/xhigh): {levels_now}\')\n        print(\'思考档位门禁通过：\', levels_now, \'| 默认档位：\', SPEC.get(\'reasoning_effort_default\', \'high\'), flush=True)\n        _warmup_cuda()\n    verify_hashes()\n    release_gpu_only_copies()\n    fut_ts.result()\n    status = ts(\'serve\', \'status\').stdout\n    if SERVICE_FQDN not in status:\n        raise RuntimeError(\'Service 未生效，serve status:\' + chr(10) + status)\nelse:\n    # 就绪横幅不等这些：预热、删内存盘副本、等 Tailscale。\n    def _bg_finish():\n        try:\n            if server_just_started:\n                if os.environ.get(\'LAUNCH_DETACHED\') == \'1\':\n                    # 主线程已返回。预热必须先等本机 /health，否则连接失败会跳过后面的内存盘释放和 Tailscale 收尾。\n                    deadline = time.monotonic() + 600\n                    while time.monotonic() < deadline:\n                        if proc.poll() is not None:\n                            raise RuntimeError((ROOT / \'server.log\').read_text(errors=\'replace\')[-3500:])\n                        if health():\n                            break\n                        time.sleep(0.5)\n                    else:\n                        raise RuntimeError(\'模型加载超时，请查看 %s/server.log\' % ROOT)\n                _warmup_cuda()\n            release_gpu_only_copies()\n            fut_cpp.result()\n            fut_ts.result()\n        except Exception as exc:\n            print(\'后台收尾失败：\', exc, flush=True)\n    threading.Thread(target=_bg_finish, daemon=True).start()\n    print(\'生产模式：预热、释放内存盘副本、Tailscale 在后台收尾，不挡住下面的就绪横幅。\', flush=True)\n\nprint(\'=\' * 64)\nprint(\'总启动耗时： %.1fs\' % (time.monotonic() - t_boot), flush=True)\nprint(\'网页版：     http://%s:443/\' % SERVICE_FQDN, flush=True)\nprint(\'接口：       http://%s:443/v1\' % SERVICE_FQDN, flush=True)\nprint(\'模型名：     %s\' % MODEL_ALIAS, flush=True)\nprint(\'=\' * 64)\n\n# 6. 就绪横幅 + WebUI 标签页 + 停机按钮\n# 浏览器只允许 window.open 的发起帧关闭该标签页，所以 WebUI 由本格输出帧打开，本格停机按钮先关它再释放运行时。\nfrom google.colab import output as colab_output, runtime\nfrom IPython.display import HTML, display\n\ndef _qwen_stop():\n    if globals().get(\'qwen_server\') and qwen_server.poll() is None:\n        qwen_server.terminate()\n    if Path(TS).exists():\n        try:\n            subprocess.run([TS, \'--socket=\' + TS_SOCK, \'down\'], timeout=5, capture_output=True)\n        except Exception:\n            pass\n    runtime.unassign()\n\ncolab_output.register_callback(\'qwen.stop\', _qwen_stop)\n\n_QWEN_CONSOLE_HTML = """\n<div style="margin:10px 0;padding:12px;border-radius:8px;background:#1e293b;color:#f8fafc;font-size:15px;line-height:1.6;">\n  🚀 <b>服务已就绪！</b>\n  <a id="qwen-open" href="__URL__" style="color:#38bdf8;font-weight:bold;text-decoration:underline;margin-left:8px;">👉 点击打开 WebUI 交互界面 (免密直连)</a>\n  <div style="margin-top:10px;">\n    <button id="qwen-stop" style="background:#f0ad4e;color:#111;border:0;border-radius:4px;padding:6px 14px;font-size:14px;cursor:pointer;">🛑 结束 G4 运行时 (省 CU)</button>\n  </div>\n  <div id="qwen-state" style="margin-top:8px;font-size:14px;"><b>当前状态：</b><span style="color:#5cb85c">🔓 Huihui v2 原生消融 125B · UD-Q4_K_XL-V2，满速运行</span> — 用完点黄色按钮立即释放 GPU，WebUI 标签页会随之自动关闭</div>\n</div>\n<script>\n(() => {\n  const URL = "__URL__";\n  const stateEl = document.getElementById("qwen-state");\n  const bootMs = Date.now();\n  let win = null;\n  // 只由本次运行的 eval_js 调用；重开笔记本时旧输出里的脚本会重跑，但不会自动弹窗\n  window.qwenOpenWebui = () => {\n    win = window.open(URL, "qwen-webui");\n    if (!win) stateEl.textContent = "浏览器拦截了自动弹窗，请点上面的链接打开";\n  };\n  document.getElementById("qwen-open").addEventListener("click", (e) => {\n    e.preventDefault();\n    window.qwenOpenWebui();\n  });\n  document.getElementById("qwen-stop").addEventListener("click", (e) => {\n    e.currentTarget.disabled = true;\n    e.currentTarget.textContent = "正在释放…";\n    if (win && !win.closed) win.close();\n    const runMin = ((Date.now() - bootMs) / 60000).toFixed(1);\n    stateEl.innerHTML = "<b>正在关闭服务并销毁运行时（本次就绪后已运行 " + runMin + " 分钟，停止计费）…</b>";\n    google.colab.kernel.invokeFunction("qwen.stop", [], {}).catch(() => {});\n  });\n})();\n</script>\n""".replace(\'__URL__\', \'http://%s:443/\' % SERVICE_FQDN)\ndisplay(HTML(_QWEN_CONSOLE_HTML))\ntry:\n    colab_output.eval_js(\'window.qwenOpenWebui()\', ignore_result=True)\nexcept Exception:\n    pass\n'}, 'gemma4-31b': {'alias': 'Gemma-4-31B-it-ARA-Heretic-Q8_0', 'root_dir': '/content/gemma4-31b', 'min_vram_mib': 35000, 'code': '# 一键启动（由 colab_kernel.py 统一内核生成）：tmpfs 内存盘 + 32路下载 + 字节大小校验 + WebUI 本机文件桥 + Tailscale\n# 调试新模型/改配置时设 True\nDEBUG_VERIFY = False\nimport concurrent.futures, hashlib, json, os, shutil, socket, struct, subprocess, sys, threading, time\nfrom pathlib import Path\nimport requests\nfrom requests.adapters import HTTPAdapter\n\nt_boot = time.monotonic()\nSPEC = json.loads("{\\n  \\"slug\\": \\"gemma4-31b\\",\\n  \\"alias\\": \\"Gemma-4-31B-it-ARA-Heretic-Q8_0\\",\\n  \\"root_dir\\": \\"/content/gemma4-31b\\",\\n  \\"ts_hostname\\": \\"gemma4-31b-colab\\",\\n  \\"min_vram_mib\\": 35000,\\n  \\"ram_reserve_gib\\": 30,\\n  \\"ple_shard_name\\": null,\\n  \\"template_family\\": \\"gemma4\\",\\n  \\"temp\\": \\"1.0\\",\\n  \\"top_p\\": \\"0.95\\",\\n  \\"top_k\\": \\"64\\",\\n  \\"min_p\\": \\"0.0\\",\\n  \\"reasoning_effort_default\\": \\"high\\",\\n  \\"effort_prompts\\": {\\n    \\"xhigh\\": \\"\\u7528\\u4e2d\\u6587\\u601d\\u8003\\uff0c\\u5c3d\\u91cf\\u6df1\\u5165\\u3002\\",\\n    \\"high\\": \\"\\u7528\\u4e2d\\u6587\\u601d\\u8003\\u3002\\",\\n    \\"medium\\": \\"\\u7528\\u4e2d\\u6587\\u601d\\u8003\\uff0c\\u7b80\\u77ed\\u4e9b\\u3002\\",\\n    \\"low\\": \\"\\u7528\\u4e2d\\u6587\\u601d\\u8003\\uff0c\\u8d8a\\u77ed\\u8d8a\\u597d\\u3002\\"\\n  },\\n  \\"mtp_n_max\\": 3,\\n  \\"mtp_p_min\\": \\"0.50\\",\\n  \\"files\\": [\\n    {\\n      \\"role\\": \\"model\\",\\n      \\"repo\\": \\"mradermacher/gemma-4-31B-it-ARA-heretic-GGUF\\",\\n      \\"rev\\": \\"1b49bbdcb3dfaa67cc4daced7da7d16552959f97\\",\\n      \\"path\\": \\"gemma-4-31B-it-ARA-heretic.Q8_0.gguf\\",\\n      \\"size\\": 32635677600,\\n      \\"priority\\": 1\\n    },\\n    {\\n      \\"role\\": \\"mmproj\\",\\n      \\"repo\\": \\"mradermacher/gemma-4-31B-it-ARA-heretic-GGUF\\",\\n      \\"rev\\": \\"1b49bbdcb3dfaa67cc4daced7da7d16552959f97\\",\\n      \\"path\\": \\"gemma-4-31B-it-ARA-heretic.mmproj-f16.gguf\\",\\n      \\"size\\": 1198957152,\\n      \\"priority\\": 1\\n    },\\n    {\\n      \\"role\\": \\"draft\\",\\n      \\"repo\\": \\"unsloth/gemma-4-31B-it-GGUF\\",\\n      \\"rev\\": \\"c1ac76e99d5513b141e8adde7288b85c3f9c32ec\\",\\n      \\"path\\": \\"MTP/mtp-gemma-4-31B-it-BF16.gguf\\",\\n      \\"size\\": 954843264,\\n      \\"priority\\": 1\\n    }\\n  ]\\n}")\nUI_CFG_DATA = json.loads("{\\n  \\"systemMessage\\": \\"\\u4f60\\u662f\\u4e2d\\u6587\\u957f\\u7bc7\\u5c0f\\u8bf4\\u5171\\u521b\\u642d\\u6863\\u3002\\\\n1. \\u5f00\\u4e86 local_novel_fs \\u65f6\\uff0c\\u52a8\\u7b14\\u524d\\u5148\\u8bfb \\u5199\\u4f5c\\u8981\\u6c42.md \\u548c\\u6b63\\u6587\\u672b\\u5c3e\\uff08tail_lines=150\\uff09\\u3002\\u65b0\\u7ae0\\u6216\\u4e0b\\u534a\\u7ae0\\u7528 write_novel_file \\u8ffd\\u52a0\\uff1b\\u91cd\\u5199\\u4e0a\\u4e00\\u8f6e\\u7528 replace_last_write\\uff0c\\u91cd\\u5199\\u6574\\u7ae0\\u7528 replace_last_chapter\\uff1b\\u5c0f\\u6539\\u7528 edit_novel_file\\uff1b\\u64a4\\u56de\\u7528 undo_novel_write\\u3002\\u6ca1\\u5f00\\u5de5\\u5177\\u5c31\\u76f4\\u63a5\\u5728\\u5bf9\\u8bdd\\u91cc\\u5199\\u3002\\\\n2. \\u67e5\\u539f\\u4f5c\\u7528 grep_reference\\uff1b\\u8bbe\\u5b9a\\u51b2\\u7a81\\u4ee5\\u7528\\u6237\\u5f53\\u524d\\u6307\\u4ee4\\u4e3a\\u51c6\\u3002\\\\n3. \\u7528\\u6237\\u8bf4\\u300c\\u6536\\u5de5\\u300d\\u65f6\\uff0c\\u7528 edit_novel_file \\u628a\\u65b0\\u5b9a\\u7684\\u8bbe\\u5b9a\\u3001\\u8fdb\\u5ea6\\u548c\\u4f0f\\u7b14\\u66f4\\u65b0\\u8fdb \\u5199\\u4f5c\\u8981\\u6c42.md\\uff0c\\u53ea\\u6539\\u53d8\\u5316\\u9879\\u3002\\\\n4. \\u7528\\u6237\\u70b9\\u540d\\u65f6\\u624d\\u8bfb\\u6280\\u80fd\\uff1agrill/\\u62f7\\u95ee \\u8bfb skill:grilling\\uff0c\\u4f18\\u5316\\u63d0\\u793a\\u8bcd \\u8bfb skill:prompting\\uff0cretro/\\u590d\\u76d8 \\u8bfb skill:retro\\u3002\\",\\n  \\"showSystemMessage\\": true,\\n  \\"showMessageStats\\": true,\\n  \\"enableContinueGeneration\\": true,\\n  \\"pasteLongTextToFileLen\\": 50000,\\n  \\"excludeReasoningFromContext\\": true,\\n  \\"renderThinkingAsMarkdown\\": true,\\n  \\"renderUserContentAsMarkdown\\": true,\\n  \\"titleGenerationUseLLM\\": false,\\n  \\"titleGenerationUseFirstLine\\": true,\\n  \\"mcpServers\\": \\"[{\\\\\\"id\\\\\\": \\\\\\"local-novel-fs\\\\\\", \\\\\\"name\\\\\\": \\\\\\"local_novel_fs\\\\\\", \\\\\\"displayName\\\\\\": \\\\\\"\\u672c\\u673a\\u5c0f\\u8bf4\\u76ee\\u5f55 (Colab\\u521b\\u4f5c)\\\\\\", \\\\\\"url\\\\\\": \\\\\\"http://127.0.0.1:3426/mcp\\\\\\", \\\\\\"enabled\\\\\\": true, \\\\\\"useProxy\\\\\\": true}]\\"\\n}")\n\nROOT = Path(SPEC[\'root_dir\'])\nDIST = ROOT / \'llama-dist\'\nMODEL_ALIAS = SPEC[\'alias\']\nKEY_FILE = ROOT / \'.llama-api-key\'\nPORT = 8081\nKEY = "<YOUR_LLAMA_API_KEY>"\nFILES = SPEC[\'files\']\nCPP_TAG = \'b12032\'\nCPP_URL = \'https://github.com/ggml-org/llama.cpp/releases/download/\' + CPP_TAG\n\ndef _local_path(f):\n    sub = f.get(\'local_subdir\')\n    return (ROOT / sub / Path(f[\'path\']).name) if sub else (ROOT / Path(f[\'path\']).name)\n\nMODEL = _local_path(FILES[0])\nMMPROJ = next((_local_path(f) for f in FILES if f[\'role\'] == \'mmproj\'), None)\nDRAFT = next((_local_path(f) for f in FILES if f[\'role\'] == \'draft\'), None)\nLORA = next((_local_path(f) for f in FILES if f[\'role\'] == \'lora\'), None)\n# PLE 表所在目录跟着各模型的 local_subdir 走（IQ4_XS / IQ4_NL …），不能写死。\nSHARD_DIR = next((_local_path(f).parent for f in FILES if Path(f[\'path\']).name == SPEC.get(\'ple_shard_name\')), ROOT)\n\n# 服务把权重读进显存后，/dev/shm 里只供显存用的副本就是冗余内存；删掉后靠这份记录（绑定服务 PID）在重跑本格时跳过重新下载。\nRELEASE_STATE = ROOT / \'released-copies.json\'\nHEAD_CACHE = ROOT / \'model-head.bin\'\nTMPFS_PREFIXES = (\'/dev/shm/\', \'/var/colab/\')\nFILE_NAMES = {_local_path(f).name for f in FILES}\n\ndef _server_proc():\n    # 本模型的 llama-server：(pid, argv)；没有则 (None, [])\n    for procdir in Path(\'/proc\').iterdir():\n        if not procdir.name.isdigit():\n            continue\n        try:\n            args = (procdir / \'cmdline\').read_bytes().decode(errors=\'replace\').split(\'\\0\')\n        except (FileNotFoundError, PermissionError, ProcessLookupError):\n            continue\n        if args and Path(args[0]).name == \'llama-server\' and str(MODEL) in args:\n            return int(procdir.name), args\n    return None, []\n\ndef _server_pid():\n    return _server_proc()[0]\n\ndef _proc_identity(pid):\n    # boot_id + 进程启动时刻，防止重启后的同模型服务恰好复用旧 PID 时误用释放记录\n    try:\n        boot = Path(\'/proc/sys/kernel/random/boot_id\').read_text().strip()\n        stat = Path(\'/proc/%d/stat\' % pid).read_text()\n        return boot + \':\' + stat.rsplit(\')\', 1)[1].split()[19]\n    except (OSError, IndexError, TypeError):\n        return None\n\ndef _cache_ram_of(args):\n    try:\n        return int(args[args.index(\'--cache-ram\') + 1])\n    except (ValueError, IndexError):\n        return None\n\ndef _stop_llama_servers():\n    for procdir in Path(\'/proc\').iterdir():\n        if not procdir.name.isdigit():\n            continue\n        try:\n            args = (procdir / \'cmdline\').read_bytes().decode(errors=\'replace\').split(\'\\0\')\n            if args and Path(args[0]).name == \'llama-server\':\n                os.kill(int(procdir.name), 15)\n                for _ in range(60):\n                    stat = (procdir / \'stat\').read_text() if procdir.exists() else \'\'\n                    if not stat or stat.split(\') \', 1)[1].startswith(\'Z\'):\n                        break\n                    time.sleep(0.5)\n                else:\n                    os.kill(int(procdir.name), 9)\n                    time.sleep(2)\n        except (FileNotFoundError, ProcessLookupError, PermissionError):\n            pass\n    old_proc = globals().get(\'qwen_server\')\n    if old_proc is not None:\n        try:\n            old_proc.wait(timeout=5)\n        except Exception:\n            pass\n    deadline = time.monotonic() + 20\n    while time.monotonic() < deadline:\n        alive = []\n        for procdir in Path(\'/proc\').iterdir():\n            try:\n                if procdir.name.isdigit() and Path((procdir / \'cmdline\').read_bytes().decode(errors=\'replace\').split(\'\\0\')[0]).name == \'llama-server\':\n                    alive.append(procdir.name)\n            except (FileNotFoundError, ProcessLookupError, PermissionError):\n                pass\n        if not alive:\n            return\n        time.sleep(1)\n    raise RuntimeError(\'旧 llama-server 停不掉（PID \' + \', \'.join(alive) + \'），已中止；请用停机按钮释放运行时后重来。\')\n\ndef _mem_available():\n    for l in Path(\'/proc/meminfo\').read_text().splitlines():\n        if l.startswith(\'MemAvailable:\'):\n            return int(l.split()[1]) * 1024\n    return 0\n\nSERVICE = \'svc:qwen\'\nSERVICE_FQDN = "qwen.<YOUR_TAILNET>.ts.net"\nSERVICE_TAG = \'tag:colab\'\nTS_HOSTNAME = SPEC[\'ts_hostname\']\nTS_CLIENT_ID = "<YOUR_TS_OAUTH_CLIENT_ID>"\nTS_CLIENT_SECRET = "<YOUR_TS_OAUTH_CLIENT_SECRET>"\nTS = \'/content/tsbin/tailscale\'\nTS_SOCK = \'/var/run/tailscale/tailscaled.sock\'\nTS_STATE = Path(\'/content/tailscaled.state\')\nTS_DIR = Path(\'/content/tsstatedir\')\n\ndef sh(c):\n    subprocess.run(c, shell=True, check=True)\n\ndef health():\n    try:\n        return requests.get(\'http://127.0.0.1:%d/health\' % PORT, timeout=2).status_code == 200\n    except Exception:\n        return False\n\ngpu_info = subprocess.getoutput(\'nvidia-smi --query-gpu=name,memory.total --format=csv,noheader,nounits\').strip()\nif not gpu_info or \',\' not in gpu_info:\n    raise RuntimeError(\'未检测到 NVIDIA GPU，请在右上角切换到 G4 GPU 运行时。输出：\' + gpu_info)\ngpu_name, gpu_mib_str = [x.strip() for x in gpu_info.split(\',\', 1)]\ngpu_mib = int(gpu_mib_str)\nprint(f\'检测到 GPU: {gpu_name} ({gpu_mib / 1024:.1f} GiB VRAM)\', flush=True)\nif gpu_mib < SPEC[\'min_vram_mib\']:\n    raise RuntimeError(f\'当前 GPU 显存仅 {gpu_mib / 1024:.1f} GiB（{gpu_name}），低于要求的 {SPEC["min_vram_mib"] / 1024:.0f} GiB，请切换到 G4 高内存运行时。\')\n\nROOT.mkdir(parents=True, exist_ok=True)\nfor _d in sorted(set(_local_path(f).parent for f in FILES)):\n    _d.mkdir(parents=True, exist_ok=True)\nDIST.mkdir(parents=True, exist_ok=True)\nKEY_FILE.write_text(KEY + \'\\n\', encoding=\'utf-8\', newline=\'\')\nKEY_FILE.chmod(0o600)\nAUTH = {\'Authorization\': \'Bearer \' + KEY}\n\ndef prepare_llama_cpp():\n    if next(DIST.rglob(\'llama-server\'), None) is not None:\n        return\n    print(f\'[并行任务] 下载并解压 llama.cpp {CPP_TAG} (CUDA 12.8) ...\', flush=True)\n    sh(f\'cd {DIST} && \'\n       f\'curl -fsSL -O {CPP_URL}/llama-{CPP_TAG}-bin-ubuntu-cuda-12.8-x64.tar.gz && \'\n       f\'curl -fsSL -O {CPP_URL}/cudart-llama-{CPP_TAG}-bin-ubuntu-cuda-12.8-x64.tar.gz && \'\n       f\'tar -xzf llama-{CPP_TAG}-bin-ubuntu-cuda-12.8-x64.tar.gz && \'\n       f\'tar -xzf cudart-llama-{CPP_TAG}-bin-ubuntu-cuda-12.8-x64.tar.gz && \'\n       f\'rm -f *.tar.gz\')\n    print(f\'[并行任务] llama.cpp {CPP_TAG} 就绪\', flush=True)\n\ndef ts(*a, timeout=60):\n    return subprocess.run([TS, \'--socket=\' + TS_SOCK, *a],\n                          stdin=subprocess.DEVNULL, capture_output=True, text=True, timeout=timeout)\n\ndef ts_state():\n    try:\n        return json.loads(ts(\'status\', \'--json\').stdout).get(\'BackendState\', \'\')\n    except Exception:\n        return \'\'\n\ndef _start_mcp_bridge():\n    srv = socket.socket(socket.AF_INET, socket.SOCK_STREAM)\n    srv.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)\n    try:\n        srv.bind((\'127.0.0.1\', 3426))\n    except OSError:\n        return\n    srv.listen(32)\n    def _pipe(a, b):\n        try:\n            while (d := a.recv(65536)):\n                b.sendall(d)\n        except Exception:\n            pass\n        finally:\n            try: a.close()\n            except Exception: pass\n            try: b.close()\n            except Exception: pass\n    while True:\n        try:\n            cli, _ = srv.accept()\n            up = socket.create_connection((\'127.0.0.1\', 1055), timeout=10)\n            up.sendall(bytes([5, 1, 0]))\n            assert up.recv(2) == bytes([5, 0])\n            up.sendall(bytes([5, 1, 0, 1]) + socket.inet_aton("<YOUR_LOCAL_TAILSCALE_IP>") + struct.pack(\'>H\', 3426))\n            assert up.recv(10)[1] == 0\n            threading.Thread(target=_pipe, args=(cli, up), daemon=True).start()\n            threading.Thread(target=_pipe, args=(up, cli), daemon=True).start()\n        except Exception:\n            pass\n\ndef prepare_tailscale():\n    if not Path(TS).exists():\n        sh(\'curl -fsSL https://pkgs.tailscale.com/stable/tailscale_1.104.1_amd64.tgz | tar -xz -C /content && \'\n           \'mv /content/tailscale_1.104.1_amd64 /content/tsbin\')\n    TS_DIR.mkdir(parents=True, exist_ok=True)\n    Path(\'/var/run/tailscale\').mkdir(parents=True, exist_ok=True)\n    if ts_state() in (\'\', \'NoState\', \'Stopped\'):\n        subprocess.Popen(\n            [\'/content/tsbin/tailscaled\', \'--tun=userspace-networking\',\n             \'--socks5-server=localhost:1055\', \'--state=\' + str(TS_STATE),\n             \'--statedir=\' + str(TS_DIR), \'--socket=\' + TS_SOCK],\n            stdout=open(\'/content/tailscaled.log\', \'a\', encoding=\'utf-8\', newline=\'\'),\n            stderr=subprocess.STDOUT, stdin=subprocess.DEVNULL, start_new_session=True,\n        )\n        for _ in range(40):\n            if Path(TS_SOCK).exists():\n                break\n            time.sleep(0.5)\n        for _ in range(40):\n            if ts_state() in (\'Running\', \'NeedsLogin\'):\n                break\n            time.sleep(0.5)\n    if ts_state() != \'Running\':\n        token = requests.post(\n            \'https://api.tailscale.com/api/v2/oauth/token\',\n            data={\'client_id\': TS_CLIENT_ID, \'client_secret\': TS_CLIENT_SECRET}, timeout=30\n        ).json()[\'access_token\']\n        key = requests.post(\n            \'https://api.tailscale.com/api/v2/tailnet/-/keys\',\n            headers={\'Authorization\': \'Bearer \' + token},\n            json={\'capabilities\': {\'devices\': {\'create\': {\n                \'reusable\': False, \'ephemeral\': True, \'preauthorized\': True, \'tags\': [SERVICE_TAG]\n            }}}}, timeout=30\n        ).json()[\'key\']\n        ts(\'up\', \'--auth-key=\' + key, \'--hostname=\' + TS_HOSTNAME,\n           \'--advertise-tags=\' + SERVICE_TAG, \'--accept-dns=false\', \'--timeout=60s\', timeout=120)\n        for _ in range(120):\n            if ts_state() == \'Running\':\n                break\n            time.sleep(0.5)\n    ts(\'serve\', \'reset\')\n    ts(\'serve\', \'--bg\', \'--service=\' + SERVICE, \'--tcp=443\', \'tcp://127.0.0.1:%d\' % PORT, timeout=90)\n    ts(\'serve\', \'--bg\', \'--https=443\', \'http://127.0.0.1:%d\' % PORT, timeout=90)\n    threading.Thread(target=_start_mcp_bridge, daemon=True).start()\n    print(f\'[并行任务] Tailscale Service ({SERVICE_FQDN}) 与本机小说文件 MCP 隧道已就绪\', flush=True)\n\nbg_executor = concurrent.futures.ThreadPoolExecutor(max_workers=2)\nfut_cpp = bg_executor.submit(prepare_llama_cpp)\nfut_ts = bg_executor.submit(prepare_tailscale)\n\nsubprocess.run([\'mount\', \'-o\', \'remount,size=115G\', \'/dev/shm\'], capture_output=True)\ntmpfs_dirs = []\nfor mline in Path(\'/proc/mounts\').read_text().splitlines():\n    mparts = mline.split()\n    if len(mparts) >= 3 and mparts[2] == \'tmpfs\' and mparts[1] in (\'/dev/shm\', \'/var/colab\'):\n        tmpfs_dirs.append(Path(mparts[1]))\nmp_free = {mp: shutil.disk_usage(mp).free for mp in tmpfs_dirs}\nmeminfo_txt = Path(\'/proc/meminfo\').read_text()\navail_kb = next((int(l.split()[1]) for l in meminfo_txt.splitlines() if l.startswith(\'MemAvailable:\')), 0)\nram_budget = max(0, avail_kb * 1024 - SPEC[\'ram_reserve_gib\'] * (1024**3))\n\nreleased_reuse = set()\nif RELEASE_STATE.exists():\n    try:\n        rel_state = json.loads(RELEASE_STATE.read_text(encoding=\'utf-8\'))\n        rel_ok = (isinstance(rel_state, dict) and isinstance(rel_state.get(\'pid\'), int)\n                  and isinstance(rel_state.get(\'files\'), list)\n                  and all(isinstance(n, str) and n in FILE_NAMES for n in rel_state[\'files\']))\n    except Exception:\n        rel_state, rel_ok = {}, False\n    pid_now = _server_pid()\n    if rel_ok and pid_now == rel_state[\'pid\'] and rel_state.get(\'proc_id\') and _proc_identity(pid_now) == rel_state[\'proc_id\']:\n        released_reuse = set(rel_state[\'files\'])\n        print(f\'服务仍在运行：{len(released_reuse)} 个已释放内存副本的分片无需重新下载\', flush=True)\n    else:\n        if not rel_ok:\n            print(\'⚠️ released-copies.json 无法识别，缺失的分片会重新下载\', flush=True)\n        RELEASE_STATE.unlink(missing_ok=True)\nif not released_reuse and _server_pid() and any(not _local_path(f).exists() for f in FILES):\n    # 记录不可用、服务还在且文件已删：先停服务再补下载，避免服务占着内存时又往内存盘塞几十 GiB。\n    print(\'⚠️ 服务在运行，但分片副本已删且释放记录不可用：先停掉旧服务，再重新下载。\', flush=True)\n    _stop_llama_servers()\n\nsorted_files = sorted(FILES, key=lambda x: x.get(\'priority\', 2))\nall_targets = [\n    (\'https://huggingface.co/\' + f[\'repo\'] + \'/resolve/\' + f[\'rev\'] + \'/\' + f[\'path\'],\n     _local_path(f), f[\'size\'])\n    for f in sorted_files\n]\ndownloads = []\ntmpfs_bytes = 0\ndisk_bytes = 0\nfor orig_url, logical_path, size in all_targets:\n    if logical_path.name in released_reuse:\n        continue\n    if logical_path.exists() and logical_path.stat().st_size == size:\n        continue\n    if logical_path.is_symlink() or logical_path.exists():\n        logical_path.unlink(missing_ok=True)\n    real_path = logical_path\n    if ram_budget >= size:\n        for mp in tmpfs_dirs:\n            if mp_free.get(mp, 0) >= size + 4 * (1024**3):\n                cand = mp / SPEC[\'slug\'] / logical_path.name\n                cand.parent.mkdir(parents=True, exist_ok=True)\n                # 只看大小会把 truncate 出来的稀疏占位文件（下载中断、上次报错）当成已下载；tmpfs 的 st_blocks 是实际占用。\n                if cand.exists() and cand.stat().st_size == size and cand.stat().st_blocks * 512 >= size:\n                    logical_path.symlink_to(cand)\n                    mp_free[mp] -= size\n                    ram_budget -= size\n                    real_path = None\n                    break\n                cand.unlink(missing_ok=True)\n                with cand.open(\'wb\') as f:\n                    f.truncate(size)\n                logical_path.symlink_to(cand)\n                mp_free[mp] -= size\n                ram_budget -= size\n                tmpfs_bytes += size\n                real_path = cand\n                break\n    if real_path is None:\n        continue\n    if real_path == logical_path:\n        free_disk = shutil.disk_usage(\'/content\').free\n        if free_disk < disk_bytes + size + 6 * (1024**3):\n            raise RuntimeError(\'本地 /content 与 tmpfs 空间不足，请重建运行时。\')\n        with real_path.open(\'wb\') as f:\n            f.truncate(size)\n        disk_bytes += size\n    downloads.append((orig_url, str(real_path), size))\n\nfreshly_downloaded = bool(downloads)\nif downloads:\n    need_bytes = sum(size for _, _, size in downloads)\n    print(f\'待并发下载: {need_bytes / (1024**3):.2f} GiB ({len(downloads)} 个文件) → DDR5 内存盘 tmpfs: {tmpfs_bytes / (1024**3):.1f} GiB | 本地磁盘: {disk_bytes / (1024**3):.1f} GiB\', flush=True)\n    CHUNK = 64 * 1024 * 1024\n    hf_tok = os.environ.get(\'HF_TOKEN\', \'\').strip()\n    if not hf_tok:\n        try:\n            from google.colab import userdata\n            hf_tok = (userdata.get(\'HF_TOKEN\') or \'\').strip()\n        except Exception:\n            hf_tok = \'\'\n\n    def _resolve_cdn(item):\n        from email.utils import parsedate_to_datetime\n        from datetime import datetime, timezone\n        orig_url, real_path, size = item\n        req_hdr = {\'Range\': \'bytes=0-0\'}\n        if hf_tok:\n            req_hdr[\'Authorization\'] = \'Bearer \' + hf_tok\n        for attempt in range(5):\n            try:\n                with requests.get(orig_url, headers=req_hdr, stream=True, timeout=30) as r:\n                    if r.status_code == 429:\n                        ra = r.headers.get(\'Retry-After\', \'\')\n                        try:\n                            wait_s = float(ra)\n                        except ValueError:\n                            try:\n                                wait_s = max(1.0, (parsedate_to_datetime(ra) - datetime.now(timezone.utc)).total_seconds())\n                            except Exception:\n                                wait_s = 2 ** (attempt + 1)\n                        time.sleep(min(wait_s, 60.0))\n                        continue\n                    r.raise_for_status()\n                    if r.status_code != 206:\n                        raise RuntimeError(f\'Range probe expected 206, got {r.status_code}\')\n                    cdn_url = r.url\n                    return [(cdn_url, orig_url, real_path, start, min(start + CHUNK, size) - 1)\n                            for start in range(0, size, CHUNK)]\n            except Exception:\n                if attempt == 4:\n                    raise\n                time.sleep(2 ** (attempt + 1))\n\n    range_tasks = []\n    with concurrent.futures.ThreadPoolExecutor(max_workers=len(downloads)) as rpool:\n        for chunk_list in rpool.map(_resolve_cdn, downloads):\n            range_tasks.extend(chunk_list)\n\n    tasks_file = Path(\'/tmp/colab_kernel_dl_tasks.json\')\n    tasks_file.write_text(json.dumps({\'need_bytes\': need_bytes, \'tasks\': range_tasks}), encoding=\'utf-8\', newline=\'\')\n    dl_script = Path(\'/tmp/colab_kernel_fast_dl.py\')\n    dl_script.write_text(\'\'\'import concurrent.futures, json, multiprocessing as mp, os, sys, time\nimport requests\nfrom requests.adapters import HTTPAdapter\n\ndata = json.loads(open(sys.argv[1], encoding="utf-8").read())\nneed_bytes = data["need_bytes"]\ntasks = data["tasks"]\nN_PROC = 8\nN_THREADS = 4\n\ndone_bytes = mp.Value("q", 0)\nt_start = time.monotonic()\nlast_print = mp.Value("d", t_start)\n\ndef worker_batch(batch):\n    sess = requests.Session()\n    adapter = HTTPAdapter(pool_connections=N_THREADS + 4, pool_maxsize=N_THREADS + 4)\n    sess.mount("https://", adapter)\n    sess.mount("http://", adapter)\n\n    def fetch_one(task):\n        cdn_url, orig_url, target, start, end = task\n        expected = end - start + 1\n        for attempt in range(5):\n            url = cdn_url if attempt < 2 else orig_url\n            try:\n                with sess.get(url, headers={"Range": f"bytes={start}-{end}"}, stream=True, timeout=(15, 120)) as resp:\n                    if resp.status_code != 206:\n                        raise RuntimeError(f"HTTP {resp.status_code}")\n                    fd = os.open(target, os.O_RDWR)\n                    try:\n                        pos = start\n                        for blk in resp.iter_content(8 * 1024 * 1024):\n                            if blk:\n                                os.pwrite(fd, blk, pos)\n                                pos += len(blk)\n                    finally:\n                        os.close(fd)\n                    if pos - start != expected:\n                        raise RuntimeError(f"short read {pos - start} != {expected}")\n                with done_bytes.get_lock():\n                    done_bytes.value += expected\n                    now = time.monotonic()\n                    if now - last_print.value >= 8:\n                        rate = done_bytes.value / max(now - t_start, 0.1) / (1024**2)\n                        print(f"  并发下载进度: {done_bytes.value / need_bytes:.0%} ({done_bytes.value / (1024**3):.1f} / {need_bytes / (1024**3):.1f} GiB)，均速 {rate:.0f} MB/s", flush=True)\n                        last_print.value = now\n                return\n            except Exception as err:\n                if attempt == 4:\n                    raise RuntimeError(f"{os.path.basename(target)} [{start}-{end}] 失败: {err}") from err\n                time.sleep(1.5 * (attempt + 1))\n\n    with concurrent.futures.ThreadPoolExecutor(max_workers=N_THREADS) as tpool:\n        futs = [tpool.submit(fetch_one, t) for t in batch]\n        for f in concurrent.futures.as_completed(futs):\n            f.result()\n\nif __name__ == "__main__":\n    batches = [tasks[i::N_PROC] for i in range(N_PROC) if tasks[i::N_PROC]]\n    ctx = mp.get_context("fork")\n    with ctx.Pool(processes=len(batches)) as pool:\n        pool.map(worker_batch, batches)\n\'\'\', encoding=\'utf-8\', newline=\'\')\n    t_dl_start = time.monotonic()\n    subprocess.run([sys.executable, str(dl_script), str(tasks_file)], check=True)\n    dl_sec = max(time.monotonic() - t_dl_start, 0.1)\n    print(f\'全部权重下载完成：用时 {dl_sec:.1f}s，平均 {need_bytes / (1024**2) / dl_sec:.0f} MB/s\', flush=True)\nelse:\n    print(f\'全部 {len(FILES)} 个权重与附件已存在于内存盘/本地，跳过下载。\', flush=True)\n\ndef verify_hashes():\n    # 仅调试模式调用。下载决策时已经对过一次字节大小；这里再 stat 一遍。\n    try:\n        for item in FILES:\n            path = _local_path(item)\n            if path.name in released_reuse:\n                continue\n            actual = path.stat().st_size if path.exists() else -1\n            if actual != item[\'size\']:\n                if path.exists():\n                    path.resolve().unlink(missing_ok=True)\n                    path.unlink(missing_ok=True)\n                raise RuntimeError(\'权重存在性或字节大小校验失败，请重新运行首格：\' + path.name)\n            print(\'SIZE OK:\', path.name, item[\'size\'], flush=True)\n    except BaseException:\n        if globals().get(\'qwen_server\') is not None:\n            qwen_server.terminate()\n        raise\n\nPARALLEL_SLOTS = 1\nSLOT_CONTEXT_TOKENS = 262144\nTOTAL_CONTEXT_TOKENS = SLOT_CONTEXT_TOKENS * PARALLEL_SLOTS\nUBATCH_SIZE = 2048\nMAX_OUTPUT_TOKENS = 65536\nSLOT_SAVE_DIR = ROOT / \'slot-saves\'\nSLOT_SAVE_DIR.mkdir(parents=True, exist_ok=True)\nCHAT_TMPL = ROOT / \'chat-template-clean.jinja\'\nUI_CFG = ROOT / \'webui-config.json\'\nnew_ui_text = json.dumps(UI_CFG_DATA, ensure_ascii=False, indent=2) + \'\\n\'\nold_ui_on_disk = UI_CFG.read_text(encoding=\'utf-8\') if UI_CFG.exists() else \'\'\n\nple_name = SPEC.get(\'ple_shard_name\')\nif ple_name:\n    ple_real = str((SHARD_DIR / ple_name).resolve())\n    ple_in_tmpfs = ple_real.startswith(TMPFS_PREFIXES)\nelse:\n    ple_in_tmpfs = False\n\n# PLE 表分片在运行期由 CPU 直接 mmap 读取，必须常驻；其余落在内存盘的分片加载后可删。\nrelease_keep = {ple_name} if ple_in_tmpfs else set()\nrelease_candidates = [f for f in FILES\n                      if _local_path(f).name not in release_keep\n                      and _local_path(f).exists()\n                      and str(_local_path(f).resolve()).startswith(TMPFS_PREFIXES)]\nreleasable_bytes = sum(f[\'size\'] for f in release_candidates)\n# 只用于新启动的服务；已在运行的服务保留自己的 --cache-ram（configured_server 不比较这一项，免得为改缓存上限重启）。\n# 只拿回收内存的一半给提示缓存，另一半留作余量（mmap 模式下 CPU 张量所在分片删文件后仍占内存）。\nPROMPT_CACHE_MIB = min(65536, 16384 + releasable_bytes // (2 * 1024**2))\n\ndef release_gpu_only_copies():\n    pid, args = _server_proc()\n    todo = [f for f in release_candidates if _local_path(f).exists()]\n    if not pid or not todo or not health():\n        return\n    names = sorted({_local_path(f).name for f in todo} | released_reuse)\n    tmp_state = RELEASE_STATE.with_suffix(\'.tmp\')\n    tmp_state.write_text(json.dumps({\'pid\': pid, \'proc_id\': _proc_identity(pid), \'files\': names}), encoding=\'utf-8\', newline=\'\')\n    os.replace(tmp_state, RELEASE_STATE)\n    if DEBUG_VERIFY:\n        before = _mem_available()\n    for f in todo:\n        lp = _local_path(f)\n        lp.resolve().unlink(missing_ok=True)\n        lp.unlink(missing_ok=True)\n    expect = sum(f[\'size\'] for f in todo)\n    if DEBUG_VERIFY:\n        time.sleep(1)\n        gained = _mem_available() - before\n        cap = _cache_ram_of(args) or 0\n        print(f\'已删除 {len(todo)} 个显存专用分片的内存盘副本：可用内存 +{gained / 1024**3:.1f} GiB（预计 {expect / 1024**3:.1f} GiB）| 当前服务提示缓存上限 {cap / 1024:.0f} GiB\', flush=True)\n        if gained < expect * 0.5:\n            print(\'⚠️ 实际回收不到预计的一半：部分分片仍被服务映射，提示缓存别塞满；服务重启后才会真正归还。\', flush=True)\n    else:\n        print(f\'已删除 {len(todo)} 个显存专用分片的内存盘副本（预计 {expect / 1024**3:.1f} GiB）\', flush=True)\n\nif MODEL.exists():\n    with MODEL.open(\'rb\') as f:\n        raw_head = f.read(16 * 1024 * 1024)\n    HEAD_CACHE.write_bytes(raw_head)\nelse:\n    raw_head = HEAD_CACHE.read_bytes()\ntmpl_key = b\'tokenizer.chat_template\'\nkey_pos = raw_head.index(struct.pack(\'<Q\', len(tmpl_key)) + tmpl_key)\nval_pos = key_pos + 8 + len(tmpl_key)\nassert struct.unpack_from(\'<I\', raw_head, val_pos)[0] == 8, \'tokenizer.chat_template 不是 GGUF 字符串类型\'\ntmpl_len = struct.unpack_from(\'<Q\', raw_head, val_pos + 4)[0]\nassert val_pos + 12 + tmpl_len <= len(raw_head), \'模板超出前 16 MiB\'\ntmpl_str = raw_head[val_pos + 12:val_pos + 12 + tmpl_len].decode(\'utf-8\')\n\n# 官方模板只有 xhigh/medium/low 三档；换成四档，并按 effort_prompts 在 system 顶部注入一句中文引导。\nuniversal_effort_block = \'\'\'{%- set reasoning_instructions = \'\' %}\n{%- if enable_thinking is undefined or enable_thinking is true %}\n    {%- set resolved_reasoning_effort = reasoning_effort|default(\'@default@\') %}\n    {%- if resolved_reasoning_effort not in (\'xhigh\', \'high\', \'medium\', \'low\') %}\n        {{- raise_exception(\'Unexpected reasoning effort \' ~ reasoning_effort ~ \'. Supported types are xhigh, high, medium, and low (default @default@).\') }}\n    {%- endif %}\n    {%- if resolved_reasoning_effort == \'xhigh\' %}\n        {%- set reasoning_instructions = \'@xhigh@\' %}\n    {%- elif resolved_reasoning_effort == \'high\' %}\n        {%- set reasoning_instructions = \'@high@\' %}\n    {%- elif resolved_reasoning_effort == \'medium\' %}\n        {%- set reasoning_instructions = \'@medium@\' %}\n    {%- elif resolved_reasoning_effort == \'low\' %}\n        {%- set reasoning_instructions = \'@low@\' %}\n    {%- endif %}\n{%- endif %}\'\'\'\nfor _lvl, _txt in SPEC[\'effort_prompts\'].items():\n    universal_effort_block = universal_effort_block.replace(\'@%s@\' % _lvl, _txt)\nuniversal_effort_block = universal_effort_block.replace(\'@default@\', SPEC.get(\'reasoning_effort_default\') or \'high\')\n\nfam = SPEC.get(\'template_family\', \'qwen38_fn\')\nif fam == \'qwen38_fn\':\n    old_effort_block = \'\'\'{%- if enable_thinking is undefined or enable_thinking is true %}\n    {%- set resolved_reasoning_effort = reasoning_effort|default(\'xhigh\') %}\n    {%- if resolved_reasoning_effort not in (\'xhigh\', \'medium\', \'low\') %}\n        {{- raise_exception(\'Unexpected reasoning effort \' ~ reasoning_effort ~ \'. Supported types are xhigh (default), medium, and low.\') }}\n    {%- endif %}\n    {%- if resolved_reasoning_effort == \'xhigh\' %}\n        {%- set reasoning_instructions = \'Reasoning effort is set to xhigh. Please think carefully through the task, validate key assumptions, consider plausible alternatives, and prioritize correctness, consistency, and clarity in the final answer.\' %}\n    {%- elif resolved_reasoning_effort == \'low\' %}\n        {%- set reasoning_instructions = \'Reasoning effort is set to low. Keep your thinking brief and focused, moving directly to the conclusion without unnecessary elaboration.\' %}\n    {%- endif %}\n{%- endif %}\'\'\'\n    # Unsloth 导出的原版 Qwen3.8 模板在 resolved_reasoning_effort 下多塞了 3 行 high->xhigh 折合；先剥掉再统一替换成四档中文引导。\n    unsloth_high_shim = \'\'\'    {%- if resolved_reasoning_effort == \'high\' %}\n        {%- set resolved_reasoning_effort = \'xhigh\' %}\n    {%- endif %}\n\'\'\'\n    tmpl_str = tmpl_str.replace(unsloth_high_shim, \'\', 1)\n    assert old_effort_block in tmpl_str, \'未找到预期的 reasoning_effort 模板块\'\n    tmpl_str = tmpl_str.replace(old_effort_block, universal_effort_block, 1)\n    old_preserve_line = \'{%- if preserve_thinking is undefined or preserve_thinking is true or loop.index0 > ns.last_query_index %}\'\n    assert old_preserve_line in tmpl_str, \'未找到预期的 preserve_thinking 模板行\'\n    # 官方默认 preserve_thinking 未定义即为真，历史轮次会被套上空 <think></think>。\n    # 只保留当前工具链里 reasoning_content 非空的轮次，避免 few-shot 诱导模型秒关思考。\n    # 生成起手块不改：官方只在 enable_thinking=false 时关思考，low 照常思考。\n    tmpl_str = tmpl_str.replace(\n        old_preserve_line,\n        \'{%- if loop.index0 > ns.last_query_index and reasoning_content %}\',\n        1,\n    )\nelif fam == \'gemma4\':\n    old_think_body = \'\'\'        {{- \'<|think|>\\\\n\' -}}\n        {%- set ns.prev_message_type = \'think\' -%}\'\'\'\n    new_think_body = \'\'\'        {{- \'<|think|>\\\\n\' + (reasoning_instructions + \'\\\\n\' if reasoning_instructions else \'\') -}}\n        {%- set ns.prev_message_type = \'think\' -%}\'\'\'\n    assert old_think_body in tmpl_str, \'未找到预期的 gemma4 <|think|> 注入块\'\n    tmpl_str = tmpl_str.replace(\n        \'{%- set enable_thinking = enable_thinking | default(false) -%}\',\n        \'{%- set enable_thinking = enable_thinking | default(true) -%}\',\n        1,\n    )\n    tmpl_str = (\n        "{%- if enable_thinking is not defined %}{%- set enable_thinking = true %}{%- endif %}\\n"\n        + universal_effort_block\n        + "\\n"\n        + tmpl_str.replace(old_think_body, new_think_body, 1)\n    )\nelif fam == \'glm5_next\':\n    old_glm_head = \'\'\'{%- set effective_reasoning_effort = reasoning_effort if reasoning_effort is defined and reasoning_effort in [\'low\', \'high\'] else \'max\' -%}\n{%- if effective_reasoning_effort is not none -%}<|system|>Reasoning Effort: {{ effective_reasoning_effort | capitalize }}{%- endif -%}\n{%- set clear_thinking = clear_thinking if clear_thinking is defined else false -%}\'\'\'\n    new_glm_head = (\n        universal_effort_block\n        + "\\n{%- set effective_reasoning_effort = \'max\' if resolved_reasoning_effort == \'xhigh\' else (\'low\' if resolved_reasoning_effort == \'low\' else \'high\') -%}"\n        + "\\n{%- if effective_reasoning_effort is not none -%}<|system|>Reasoning Effort: {{ effective_reasoning_effort | capitalize }}{{ (\'\\\\n\' + reasoning_instructions) if reasoning_instructions else \'\' }}{%- endif -%}"\n        + "\\n{%- set clear_thinking = clear_thinking if clear_thinking is defined else true -%}"\n    )\n    assert old_glm_head in tmpl_str, \'未找到预期的 glm5_next reasoning_effort 模板块\'\n    tmpl_str = tmpl_str.replace(old_glm_head, new_glm_head, 1)\nelif fam == \'qwen35\':\n    old_tools_head = "{%- if tools and tools is iterable and tools is not mapping %}\\n    {{- \'<|im_start|>system\\\\n\' }}"\n    new_tools_head = (\n        universal_effort_block\n        + "\\n{%- if tools and tools is iterable and tools is not mapping %}\\n"\n        + "    {{- \'<|im_start|>system\\\\n\' }}\\n"\n        + "    {%- if reasoning_instructions %}\\n"\n        + "        {{- reasoning_instructions + \'\\\\n\\\\n\' }}\\n"\n        + "    {%- endif %}"\n    )\n    assert old_tools_head in tmpl_str, \'未找到预期的 qwen35 tools system 头部\'\n    tmpl_str = tmpl_str.replace(old_tools_head, new_tools_head, 1)\n    old_tools_sys = \'\'\'    {%- if messages[0].role == \'system\' %}\n        {%- set content = render_content(messages[0].content, false, true)|trim %}\n        {%- if content %}\n            {{- \'\\\\n\\\\n\' + content }}\n        {%- endif %}\n    {%- endif %}\n    {{- \'<|im_end|>\\\\n\' }}\n{%- else %}\n    {%- if messages[0].role == \'system\' %}\n        {%- set content = render_content(messages[0].content, false, true)|trim %}\n        {{- \'<|im_start|>system\\\\n\' + content + \'<|im_end|>\\\\n\' }}\n    {%- endif %}\n{%- endif %}\'\'\'\n    new_tools_sys = \'\'\'    {%- if messages[0].role == \'system\' %}\n        {%- set content = render_content(messages[0].content, false, true)|trim %}\n        {%- if content %}\n            {{- \'\\\\n\\\\n\' + content }}\n        {%- endif %}\n    {%- endif %}\n    {{- \'<|im_end|>\\\\n\' }}\n{%- else %}\n    {%- if messages[0].role == \'system\' %}\n        {%- set content = render_content(messages[0].content, false, true)|trim %}\n        {{- \'<|im_start|>system\\\\n\' + (reasoning_instructions + \'\\\\n\\\\n\' if reasoning_instructions else \'\') + content + \'<|im_end|>\\\\n\' }}\n    {%- elif reasoning_instructions %}\n        {{- \'<|im_start|>system\\\\n\' + reasoning_instructions + \'<|im_end|>\\\\n\' }}\n    {%- endif %}\n{%- endif %}\'\'\'\n    assert old_tools_sys in tmpl_str, \'未找到预期的 qwen35 system 分支块\'\n    tmpl_str = tmpl_str.replace(old_tools_sys, new_tools_sys, 1)\n    old_hist_think = \'{%- if loop.index0 > ns.last_query_index %}\'\n    assert old_hist_think in tmpl_str, \'未找到预期的历史思考保留条件\'\n    tmpl_str = tmpl_str.replace(old_hist_think, \'{%- if loop.index0 > ns.last_query_index and reasoning_content %}\', 1)\n    qwen_gen_think_old = \'\'\'    {%- if enable_thinking is defined and enable_thinking is false %}\n        {{- \'<think>\\\\n\\\\n</think>\\\\n\\\\n\' }}\n    {%- else %}\n        {{- \'<think>\\\\n\' }}\n    {%- endif %}\'\'\'\n    qwen_gen_think_new = \'\'\'    {%- if (enable_thinking is defined and enable_thinking is false) or (reasoning_effort is defined and reasoning_effort == \'low\') %}\n        {{- \'<think>\\\\n\\\\n</think>\\\\n\\\\n\' }}\n    {%- else %}\n        {{- \'<think>\\\\n\' }}\n    {%- endif %}\'\'\'\n    assert qwen_gen_think_old in tmpl_str, \'未找到预期的 add_generation_prompt 思考起手块\'\n    tmpl_str = tmpl_str.replace(qwen_gen_think_old, qwen_gen_think_new, 1)\nelif fam == \'nemotron\':\n    old_trunc = \'{%- set truncate_history_thinking = truncate_history_thinking if truncate_history_thinking is defined else True %}\'\n    assert old_trunc in tmpl_str, \'未找到预期的 truncate_history_thinking 行\'\n    tmpl_str = tmpl_str.replace(old_trunc, \'{%- set truncate_history_thinking = false %}\', 1)\n    old_sys_block = \'\'\'{%- if messages[0]["role"] == "system" %}\n    {%- set system_message = messages[0]["content"] %}\n    {%- set loop_messages = messages[1:] %}\n{%- else %}\n    {%- set system_message = "" %}\n    {%- set loop_messages = messages %}\n{%- endif %}\'\'\'\n    new_sys_block = (\n        universal_effort_block\n        + \'\'\'\n{%- if messages[0]["role"] == "system" %}\n    {%- set system_message = (reasoning_instructions ~ "\\\\n\\\\n" if reasoning_instructions else "") ~ render_content(messages[0]["content"]) %}\n    {%- set loop_messages = messages[1:] %}\n{%- else %}\n    {%- set system_message = reasoning_instructions %}\n    {%- set loop_messages = messages %}\n{%- endif %}\'\'\'\n    )\n    assert old_sys_block in tmpl_str, \'未找到预期的 nemotron system_message 块\'\n    tmpl_str = tmpl_str.replace(old_sys_block, new_sys_block, 1)\n    nemo_gen_think_old = \'\'\'{%- if add_generation_prompt %}\n    {%- if enable_thinking %}\n        {{- \'<|im_start|>assistant\\\\n<think>\\\\n\' }}\n    {%- else %}\n        {{- \'<|im_start|>assistant\\\\n<think></think>\' }}\n    {%- endif %}\n{%- endif %}\'\'\'\n    nemo_gen_think_new = \'\'\'{%- if add_generation_prompt %}\n    {{- \'<|im_start|>assistant\\\\n\' }}\n    {%- if (enable_thinking is defined and enable_thinking is false) or (reasoning_effort is defined and reasoning_effort == \'low\') %}\n        {{- \'<think></think>\' }}\n    {%- else %}\n        {{- \'<think>\\\\n\' }}\n    {%- endif %}\n{%- endif -%}\'\'\'\n    assert nemo_gen_think_old in tmpl_str, \'未找到预期的 nemotron add_generation_prompt 思考起手块\'\n    tmpl_str = tmpl_str.replace(nemo_gen_think_old, nemo_gen_think_new, 1)\nelse:\n    raise RuntimeError(f\'未知 template_family={fam}：必须在 colab_kernel.py 中接入 universal_effort_block 四档思考支持\')\n\nif LORA and LORA.exists():\n    # GCSA-AiLab/GLM-5.3-Flash-Abliterated-LoRA-GGUF 头部写成了 glm5next（缺连字符），原地修补为 glm5-next 并消掉 1 字节对齐填充，保持张量偏移与文件大小不变。\n    with LORA.open(\'r+b\') as _lf:\n        _lbuf = _lf.read(32768)\n        _bad_arch = b\'\\x08\\x00\\x00\\x00\\x00\\x00\\x00\\x00glm5next\'\n        _good_arch = b\'\\x09\\x00\\x00\\x00\\x00\\x00\\x00\\x00glm5-next\'\n        if _bad_arch in _lbuf:\n            _pad_idx = _lbuf.find(b\'\\x00\' * 8, 16384)\n            assert _pad_idx > 0, \'LoRA GGUF 未找到对齐填充区\'\n            _lbuf = _lbuf.replace(_bad_arch, _good_arch, 1)\n            _lbuf = _lbuf[:_pad_idx] + _lbuf[_pad_idx + 1:]\n            _lf.seek(0)\n            _lf.write(_lbuf)\nold_tmpl_on_disk = CHAT_TMPL.read_text(encoding=\'utf-8\') if CHAT_TMPL.exists() else \'\'\nCHAT_TMPL.write_text(tmpl_str, encoding=\'utf-8\', newline=\'\')\nUI_CFG.write_text(new_ui_text, encoding=\'utf-8\', newline=\'\')\n\ndef _fp_args(args):\n    # 指纹跳过二进制路径和 --cache-ram（只改缓存上限不重启）。\n    out = []\n    i = 1 if args and Path(args[0]).name == \'llama-server\' else 0\n    while i < len(args):\n        a = args[i]\n        if not a:\n            i += 1\n            continue\n        if a == \'--cache-ram\':\n            i += 2\n            continue\n        out.append(a)\n        i += 1\n    return out\n\ndef _boot_fp(args, tmpl, ui):\n    blob = chr(0).join(_fp_args(args)) + chr(0) + tmpl + chr(0) + ui\n    return hashlib.sha1(blob.encode()).hexdigest()\n\nload_flags = [\'--lazy-mode\', \'off\'] if ple_in_tmpfs else [\'-lm\', \'none\', \'--lazy-mode\', \'off\']\nserver_args = [\'-m\', str(MODEL)]\nif MMPROJ:\n    server_args += [\'--mmproj\', str(MMPROJ)]\nif DRAFT and SPEC.get(\'mtp_n_max\'):\n    server_args += [\'--model-draft\', str(DRAFT), \'--spec-type\', \'draft-mtp\',\n                    \'--spec-draft-n-max\', str(SPEC[\'mtp_n_max\']),\n                    \'--spec-draft-p-min\', str(SPEC[\'mtp_p_min\'])]\nif LORA:\n    server_args += [\'--lora\', str(LORA)]\nif SPEC.get(\'extra_server_flags\'):\n    server_args += list(SPEC[\'extra_server_flags\'])\nserver_args += [\n    \'--slot-save-path\', str(SLOT_SAVE_DIR),\n    \'--device\', \'CUDA0\',\n    \'-ngl\', \'99\',\n    *load_flags,\n    \'--cache-ram\', str(PROMPT_CACHE_MIB),\n    \'-c\', str(TOTAL_CONTEXT_TOKENS),\n    \'--parallel\', str(PARALLEL_SLOTS),\n    \'--host\', \'127.0.0.1\',\n    \'--port\', str(PORT),\n    \'--alias\', MODEL_ALIAS,\n    \'--jinja\',\n    \'--chat-template-file\', str(CHAT_TMPL),\n    \'--ui-config-file\', str(UI_CFG),\n    \'--ui-mcp-proxy\',\n    \'--reasoning\', \'on\',\n]\nif SPEC.get(\'reasoning_effort_default\'):\n    server_args += [\'--reasoning-effort\', SPEC[\'reasoning_effort_default\']]\nserver_args += [\n    \'--temp\', SPEC[\'temp\'],\n    \'--top-p\', SPEC[\'top_p\'],\n    \'--top-k\', SPEC[\'top_k\'],\n    \'--min-p\', SPEC[\'min_p\'],\n    *SPEC.get(\'penalty_args\', []),\n    \'--flash-attn\', \'on\',\n    \'--cache-type-k\', \'q8_0\',\n    \'--cache-type-v\', \'q8_0\',\n    \'--batch-size\', \'4096\',\n    \'--ubatch-size\', str(UBATCH_SIZE),\n    \'--predict\', str(MAX_OUTPUT_TOKENS),\n]\nexpected_fp = _boot_fp(server_args, tmpl_str, new_ui_text)\n\ndef configured_server():\n    # 已在跑且启动指纹一致就跳过重启。指纹 = sha1(argv 去掉二进制和 --cache-ram + 模板 + webui-config)。\n    if not health():\n        return False\n    _pid, args = _server_proc()\n    if not args or _boot_fp(args, old_tmpl_on_disk, old_ui_on_disk) != expected_fp:\n        return False\n    if DEBUG_VERIFY:\n        props = requests.get(\'http://127.0.0.1:%d/props\' % PORT, headers=AUTH, timeout=10).json()\n        if props.get(\'default_generation_settings\', {}).get(\'n_ctx\') != SLOT_CONTEXT_TOKENS:\n            return False\n        if props.get(\'total_slots\') != PARALLEL_SLOTS:\n            return False\n        if props.get(\'chat_template_caps\', {}).get(\'reasoning_effort_levels\') != [\'low\', \'medium\', \'high\', \'xhigh\']:\n            return False\n    return True\n\nserver_just_started = False\nif health() and not configured_server():\n    print(\'检测到旧配置或不同模型服务：重启 llama-server ...\', flush=True)\n    if not released_reuse:\n        _stop_llama_servers()\n\nif configured_server():\n    print(f\'模型已在运行：{MODEL_ALIAS} / 单路 × {SLOT_CONTEXT_TOKENS} 上下文 / {(_cache_ram_of(_server_proc()[1]) or 0) / 1024:.0f} GiB Prompt Cache\', flush=True)\nelse:\n    missing_now = [_local_path(f).name for f in FILES if not _local_path(f).exists()]\n    if released_reuse or missing_now:\n        _stop_llama_servers()\n        RELEASE_STATE.unlink(missing_ok=True)\n        raise RuntimeError(\'需要重新启动服务，但缺少上次为扩大提示缓存而删掉的分片副本：\'\n                           + \', \'.join(sorted(set(missing_now) | released_reuse))\n                           + \'。旧服务已停，请再运行一次本格，会重新下载这些分片。\')\n    _stop_llama_servers()\n    if ple_name and not freshly_downloaded and not ple_in_tmpfs:\n        meminfo = Path(\'/proc/meminfo\').read_text()\n        cached_kb = next((int(l.split()[1]) for l in meminfo.splitlines() if l.startswith(\'Cached:\')), 0)\n        if cached_kb < 40 * 1024 * 1024:\n            print(\'预热 PLE 表所在分片到主机内存 Page Cache ...\', flush=True)\n            with (SHARD_DIR / ple_name).open(\'rb\') as f:\n                while f.read(32 * 1024 * 1024):\n                    pass\n\n    fut_cpp.result()\n    SERVER = next(DIST.rglob(\'llama-server\'))\n    env = os.environ.copy()\n    libdirs = {str(SERVER.parent)}\n    for pat in (\'libcudart.so.12\', \'libcublas.so.12\', \'libcublasLt.so.12\'):\n        libdirs.update(str(p.parent) for p in DIST.rglob(pat))\n    env[\'LD_LIBRARY_PATH\'] = \':\'.join(sorted(libdirs)) + \':\' + env.get(\'LD_LIBRARY_PATH\', \'\')\n    cmd = [str(SERVER), *server_args]\n    log = (ROOT / \'server.log\').open(\'a\', encoding=\'utf-8\', newline=\'\')\n    log.write(chr(10) + \'=== START %s ===\' % time.strftime(\'%Y-%m-%d %H:%M:%S\') + chr(10))\n    log.flush()\n    proc = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT, env=env, stdin=subprocess.DEVNULL, start_new_session=True)\n    qwen_server = proc\n    t0 = time.monotonic()\n    # LAUNCH_DETACHED=1（colab_cli launch）：进程已 start_new_session，不等本机 /health，交给后面的后台线程。\n    if os.environ.get(\'LAUNCH_DETACHED\') == \'1\':\n        print(\'LAUNCH_DETACHED: llama-server pid %s\' % proc.pid, flush=True)\n        server_just_started = True\n    else:\n        print(f\'加载 {MODEL_ALIAS} 到 GPU ...\', flush=True)\n        while time.monotonic() - t0 < 600:\n            if proc.poll() is not None:\n                raise RuntimeError((ROOT / \'server.log\').read_text(errors=\'replace\')[-3500:])\n            if health():\n                break\n            time.sleep(2 if DEBUG_VERIFY else 0.5)\n        else:\n            proc.terminate()\n            raise RuntimeError(f\'模型加载超时，请查看 {ROOT}/server.log\')\n        server_just_started = True\n\ndef _warmup_cuda():\n    # 第一次推理会捕获 CUDA graph，短提示通常要数秒到十几秒，不是零点几秒；放后台不挡就绪横幅。只发这一次 max_tokens=4。\n    requests.post(\'http://127.0.0.1:%d/v1/chat/completions\' % PORT, headers=AUTH, timeout=600,\n                  json={\'model\': MODEL_ALIAS,\n                        \'messages\': [{\'role\': \'user\', \'content\': \'请只回复“就绪”两个字。\'}],\n                        \'max_tokens\': 4, \'temperature\': 0, \'reasoning_effort\': \'low\'})\n\nif DEBUG_VERIFY:\n    if server_just_started:\n        print(\'服务就绪 %.0fs，核验四档思考能力并发送预热请求初始化 CUDA 图 ...\' % (time.monotonic() - t0), flush=True)\n        props_now = requests.get(\'http://127.0.0.1:%d/props\' % PORT, headers=AUTH, timeout=10).json()\n        levels_now = props_now.get(\'chat_template_caps\', {}).get(\'reasoning_effort_levels\', [])\n        if levels_now != [\'low\', \'medium\', \'high\', \'xhigh\']:\n            proc.terminate()\n            raise RuntimeError(f\'思考档位探测未达四档标准 (期望 low/medium/high/xhigh): {levels_now}\')\n        print(\'思考档位门禁通过：\', levels_now, \'| 默认档位：\', SPEC.get(\'reasoning_effort_default\', \'high\'), flush=True)\n        _warmup_cuda()\n    verify_hashes()\n    release_gpu_only_copies()\n    fut_ts.result()\n    status = ts(\'serve\', \'status\').stdout\n    if SERVICE_FQDN not in status:\n        raise RuntimeError(\'Service 未生效，serve status:\' + chr(10) + status)\nelse:\n    # 就绪横幅不等这些：预热、删内存盘副本、等 Tailscale。\n    def _bg_finish():\n        try:\n            if server_just_started:\n                if os.environ.get(\'LAUNCH_DETACHED\') == \'1\':\n                    # 主线程已返回。预热必须先等本机 /health，否则连接失败会跳过后面的内存盘释放和 Tailscale 收尾。\n                    deadline = time.monotonic() + 600\n                    while time.monotonic() < deadline:\n                        if proc.poll() is not None:\n                            raise RuntimeError((ROOT / \'server.log\').read_text(errors=\'replace\')[-3500:])\n                        if health():\n                            break\n                        time.sleep(0.5)\n                    else:\n                        raise RuntimeError(\'模型加载超时，请查看 %s/server.log\' % ROOT)\n                _warmup_cuda()\n            release_gpu_only_copies()\n            fut_cpp.result()\n            fut_ts.result()\n        except Exception as exc:\n            print(\'后台收尾失败：\', exc, flush=True)\n    threading.Thread(target=_bg_finish, daemon=True).start()\n    print(\'生产模式：预热、释放内存盘副本、Tailscale 在后台收尾，不挡住下面的就绪横幅。\', flush=True)\n\nprint(\'=\' * 64)\nprint(\'总启动耗时： %.1fs\' % (time.monotonic() - t_boot), flush=True)\nprint(\'网页版：     http://%s:443/\' % SERVICE_FQDN, flush=True)\nprint(\'接口：       http://%s:443/v1\' % SERVICE_FQDN, flush=True)\nprint(\'模型名：     %s\' % MODEL_ALIAS, flush=True)\nprint(\'=\' * 64)\n\n# 6. 就绪横幅 + WebUI 标签页 + 停机按钮\n# 浏览器只允许 window.open 的发起帧关闭该标签页，所以 WebUI 由本格输出帧打开，本格停机按钮先关它再释放运行时。\nfrom google.colab import output as colab_output, runtime\nfrom IPython.display import HTML, display\n\ndef _qwen_stop():\n    if globals().get(\'qwen_server\') and qwen_server.poll() is None:\n        qwen_server.terminate()\n    if Path(TS).exists():\n        try:\n            subprocess.run([TS, \'--socket=\' + TS_SOCK, \'down\'], timeout=5, capture_output=True)\n        except Exception:\n            pass\n    runtime.unassign()\n\ncolab_output.register_callback(\'qwen.stop\', _qwen_stop)\n\n_QWEN_CONSOLE_HTML = """\n<div style="margin:10px 0;padding:12px;border-radius:8px;background:#1e293b;color:#f8fafc;font-size:15px;line-height:1.6;">\n  🚀 <b>服务已就绪！</b>\n  <a id="qwen-open" href="__URL__" style="color:#38bdf8;font-weight:bold;text-decoration:underline;margin-left:8px;">👉 点击打开 WebUI 交互界面 (免密直连)</a>\n  <div style="margin-top:10px;">\n    <button id="qwen-stop" style="background:#f0ad4e;color:#111;border:0;border-radius:4px;padding:6px 14px;font-size:14px;cursor:pointer;">🛑 结束 G4 运行时 (省 CU)</button>\n  </div>\n  <div id="qwen-state" style="margin-top:8px;font-size:14px;"><b>当前状态：</b><span style="color:#5cb85c">🔓 Gemma 4 31B ARA Heretic Q8_0，满速运行</span> — 用完点黄色按钮立即释放 GPU，WebUI 标签页会随之自动关闭</div>\n</div>\n<script>\n(() => {\n  const URL = "__URL__";\n  const stateEl = document.getElementById("qwen-state");\n  const bootMs = Date.now();\n  let win = null;\n  // 只由本次运行的 eval_js 调用；重开笔记本时旧输出里的脚本会重跑，但不会自动弹窗\n  window.qwenOpenWebui = () => {\n    win = window.open(URL, "qwen-webui");\n    if (!win) stateEl.textContent = "浏览器拦截了自动弹窗，请点上面的链接打开";\n  };\n  document.getElementById("qwen-open").addEventListener("click", (e) => {\n    e.preventDefault();\n    window.qwenOpenWebui();\n  });\n  document.getElementById("qwen-stop").addEventListener("click", (e) => {\n    e.currentTarget.disabled = true;\n    e.currentTarget.textContent = "正在释放…";\n    if (win && !win.closed) win.close();\n    const runMin = ((Date.now() - bootMs) / 60000).toFixed(1);\n    stateEl.innerHTML = "<b>正在关闭服务并销毁运行时（本次就绪后已运行 " + runMin + " 分钟，停止计费）…</b>";\n    google.colab.kernel.invokeFunction("qwen.stop", [], {}).catch(() => {});\n  });\n})();\n</script>\n""".replace(\'__URL__\', \'http://%s:443/\' % SERVICE_FQDN)\ndisplay(HTML(_QWEN_CONSOLE_HTML))\ntry:\n    colab_output.eval_js(\'window.qwenOpenWebui()\', ignore_result=True)\nexcept Exception:\n    pass\n'}, 'qwen38-27b-rvn': {'alias': 'Qwen3.8-27B-RVN-Heretic-BF16', 'root_dir': '/content/qwen38-27b-rvn', 'min_vram_mib': 60000, 'code': '# 一键启动（由 colab_kernel.py 统一内核生成）：tmpfs 内存盘 + 32路下载 + 字节大小校验 + WebUI 本机文件桥 + Tailscale\n# 调试新模型/改配置时设 True\nDEBUG_VERIFY = False\nimport concurrent.futures, hashlib, json, os, shutil, socket, struct, subprocess, sys, threading, time\nfrom pathlib import Path\nimport requests\nfrom requests.adapters import HTTPAdapter\n\nt_boot = time.monotonic()\nSPEC = json.loads("{\\n  \\"slug\\": \\"qwen38-27b-rvn\\",\\n  \\"alias\\": \\"Qwen3.8-27B-RVN-Heretic-BF16\\",\\n  \\"root_dir\\": \\"/content/qwen38-27b-rvn\\",\\n  \\"ts_hostname\\": \\"qwen-rvn27b-colab\\",\\n  \\"min_vram_mib\\": 60000,\\n  \\"ram_reserve_gib\\": 45,\\n  \\"ple_shard_name\\": null,\\n  \\"template_family\\": \\"qwen38_fn\\",\\n  \\"temp\\": \\"1.0\\",\\n  \\"top_p\\": \\"0.95\\",\\n  \\"top_k\\": \\"20\\",\\n  \\"min_p\\": \\"0.0\\",\\n  \\"reasoning_effort_default\\": \\"high\\",\\n  \\"effort_prompts\\": {\\n    \\"xhigh\\": \\"\\u7528\\u4e2d\\u6587\\u601d\\u8003\\uff0c\\u5c3d\\u91cf\\u6df1\\u5165\\u3002\\",\\n    \\"high\\": \\"\\u7528\\u4e2d\\u6587\\u601d\\u8003\\u3002\\",\\n    \\"medium\\": \\"\\u7528\\u4e2d\\u6587\\u601d\\u8003\\uff0c\\u7b80\\u77ed\\u4e9b\\u3002\\",\\n    \\"low\\": \\"\\u7528\\u4e2d\\u6587\\u601d\\u8003\\uff0c\\u8d8a\\u77ed\\u8d8a\\u597d\\u3002\\"\\n  },\\n  \\"mtp_n_max\\": 3,\\n  \\"mtp_p_min\\": \\"0.45\\",\\n  \\"files\\": [\\n    {\\n      \\"role\\": \\"model\\",\\n      \\"repo\\": \\"0bserverx/Qwen3.8-27B-Heretic-Abliterated-Uncensored-GGUF\\",\\n      \\"rev\\": \\"ab8d7bd81d03117501e4da60cccdff4c80f0eea0\\",\\n      \\"path\\": \\"RVN-BF16.gguf\\",\\n      \\"size\\": 53808281888,\\n      \\"priority\\": 1\\n    },\\n    {\\n      \\"role\\": \\"mmproj\\",\\n      \\"repo\\": \\"unsloth/Qwen3.8-27B-GGUF\\",\\n      \\"rev\\": \\"4ca720788d1e01f1bff70c033e0d0028fd02e502\\",\\n      \\"path\\": \\"mmproj-BF16.gguf\\",\\n      \\"size\\": 931146432,\\n      \\"priority\\": 1\\n    },\\n    {\\n      \\"role\\": \\"draft\\",\\n      \\"repo\\": \\"unsloth/Qwen3.8-27B-GGUF\\",\\n      \\"rev\\": \\"4ca720788d1e01f1bff70c033e0d0028fd02e502\\",\\n      \\"path\\": \\"MTP/mtp-Qwen3.8-27B-Q4_0.gguf\\",\\n      \\"size\\": 1369590656,\\n      \\"priority\\": 1\\n    }\\n  ]\\n}")\nUI_CFG_DATA = json.loads("{\\n  \\"systemMessage\\": \\"\\u4f60\\u662f\\u4e2d\\u6587\\u957f\\u7bc7\\u5c0f\\u8bf4\\u5171\\u521b\\u642d\\u6863\\u3002\\\\n1. \\u5f00\\u4e86 local_novel_fs \\u65f6\\uff0c\\u52a8\\u7b14\\u524d\\u5148\\u8bfb \\u5199\\u4f5c\\u8981\\u6c42.md \\u548c\\u6b63\\u6587\\u672b\\u5c3e\\uff08tail_lines=150\\uff09\\u3002\\u65b0\\u7ae0\\u6216\\u4e0b\\u534a\\u7ae0\\u7528 write_novel_file \\u8ffd\\u52a0\\uff1b\\u91cd\\u5199\\u4e0a\\u4e00\\u8f6e\\u7528 replace_last_write\\uff0c\\u91cd\\u5199\\u6574\\u7ae0\\u7528 replace_last_chapter\\uff1b\\u5c0f\\u6539\\u7528 edit_novel_file\\uff1b\\u64a4\\u56de\\u7528 undo_novel_write\\u3002\\u6ca1\\u5f00\\u5de5\\u5177\\u5c31\\u76f4\\u63a5\\u5728\\u5bf9\\u8bdd\\u91cc\\u5199\\u3002\\\\n2. \\u67e5\\u539f\\u4f5c\\u7528 grep_reference\\uff1b\\u8bbe\\u5b9a\\u51b2\\u7a81\\u4ee5\\u7528\\u6237\\u5f53\\u524d\\u6307\\u4ee4\\u4e3a\\u51c6\\u3002\\\\n3. \\u7528\\u6237\\u8bf4\\u300c\\u6536\\u5de5\\u300d\\u65f6\\uff0c\\u7528 edit_novel_file \\u628a\\u65b0\\u5b9a\\u7684\\u8bbe\\u5b9a\\u3001\\u8fdb\\u5ea6\\u548c\\u4f0f\\u7b14\\u66f4\\u65b0\\u8fdb \\u5199\\u4f5c\\u8981\\u6c42.md\\uff0c\\u53ea\\u6539\\u53d8\\u5316\\u9879\\u3002\\\\n4. \\u7528\\u6237\\u70b9\\u540d\\u65f6\\u624d\\u8bfb\\u6280\\u80fd\\uff1agrill/\\u62f7\\u95ee \\u8bfb skill:grilling\\uff0c\\u4f18\\u5316\\u63d0\\u793a\\u8bcd \\u8bfb skill:prompting\\uff0cretro/\\u590d\\u76d8 \\u8bfb skill:retro\\u3002\\",\\n  \\"showSystemMessage\\": true,\\n  \\"showMessageStats\\": true,\\n  \\"enableContinueGeneration\\": true,\\n  \\"pasteLongTextToFileLen\\": 50000,\\n  \\"excludeReasoningFromContext\\": true,\\n  \\"renderThinkingAsMarkdown\\": true,\\n  \\"renderUserContentAsMarkdown\\": true,\\n  \\"titleGenerationUseLLM\\": false,\\n  \\"titleGenerationUseFirstLine\\": true,\\n  \\"mcpServers\\": \\"[{\\\\\\"id\\\\\\": \\\\\\"local-novel-fs\\\\\\", \\\\\\"name\\\\\\": \\\\\\"local_novel_fs\\\\\\", \\\\\\"displayName\\\\\\": \\\\\\"\\u672c\\u673a\\u5c0f\\u8bf4\\u76ee\\u5f55 (Colab\\u521b\\u4f5c)\\\\\\", \\\\\\"url\\\\\\": \\\\\\"http://127.0.0.1:3426/mcp\\\\\\", \\\\\\"enabled\\\\\\": true, \\\\\\"useProxy\\\\\\": true}]\\"\\n}")\n\nROOT = Path(SPEC[\'root_dir\'])\nDIST = ROOT / \'llama-dist\'\nMODEL_ALIAS = SPEC[\'alias\']\nKEY_FILE = ROOT / \'.llama-api-key\'\nPORT = 8081\nKEY = "<YOUR_LLAMA_API_KEY>"\nFILES = SPEC[\'files\']\nCPP_TAG = \'b12032\'\nCPP_URL = \'https://github.com/ggml-org/llama.cpp/releases/download/\' + CPP_TAG\n\ndef _local_path(f):\n    sub = f.get(\'local_subdir\')\n    return (ROOT / sub / Path(f[\'path\']).name) if sub else (ROOT / Path(f[\'path\']).name)\n\nMODEL = _local_path(FILES[0])\nMMPROJ = next((_local_path(f) for f in FILES if f[\'role\'] == \'mmproj\'), None)\nDRAFT = next((_local_path(f) for f in FILES if f[\'role\'] == \'draft\'), None)\nLORA = next((_local_path(f) for f in FILES if f[\'role\'] == \'lora\'), None)\n# PLE 表所在目录跟着各模型的 local_subdir 走（IQ4_XS / IQ4_NL …），不能写死。\nSHARD_DIR = next((_local_path(f).parent for f in FILES if Path(f[\'path\']).name == SPEC.get(\'ple_shard_name\')), ROOT)\n\n# 服务把权重读进显存后，/dev/shm 里只供显存用的副本就是冗余内存；删掉后靠这份记录（绑定服务 PID）在重跑本格时跳过重新下载。\nRELEASE_STATE = ROOT / \'released-copies.json\'\nHEAD_CACHE = ROOT / \'model-head.bin\'\nTMPFS_PREFIXES = (\'/dev/shm/\', \'/var/colab/\')\nFILE_NAMES = {_local_path(f).name for f in FILES}\n\ndef _server_proc():\n    # 本模型的 llama-server：(pid, argv)；没有则 (None, [])\n    for procdir in Path(\'/proc\').iterdir():\n        if not procdir.name.isdigit():\n            continue\n        try:\n            args = (procdir / \'cmdline\').read_bytes().decode(errors=\'replace\').split(\'\\0\')\n        except (FileNotFoundError, PermissionError, ProcessLookupError):\n            continue\n        if args and Path(args[0]).name == \'llama-server\' and str(MODEL) in args:\n            return int(procdir.name), args\n    return None, []\n\ndef _server_pid():\n    return _server_proc()[0]\n\ndef _proc_identity(pid):\n    # boot_id + 进程启动时刻，防止重启后的同模型服务恰好复用旧 PID 时误用释放记录\n    try:\n        boot = Path(\'/proc/sys/kernel/random/boot_id\').read_text().strip()\n        stat = Path(\'/proc/%d/stat\' % pid).read_text()\n        return boot + \':\' + stat.rsplit(\')\', 1)[1].split()[19]\n    except (OSError, IndexError, TypeError):\n        return None\n\ndef _cache_ram_of(args):\n    try:\n        return int(args[args.index(\'--cache-ram\') + 1])\n    except (ValueError, IndexError):\n        return None\n\ndef _stop_llama_servers():\n    for procdir in Path(\'/proc\').iterdir():\n        if not procdir.name.isdigit():\n            continue\n        try:\n            args = (procdir / \'cmdline\').read_bytes().decode(errors=\'replace\').split(\'\\0\')\n            if args and Path(args[0]).name == \'llama-server\':\n                os.kill(int(procdir.name), 15)\n                for _ in range(60):\n                    stat = (procdir / \'stat\').read_text() if procdir.exists() else \'\'\n                    if not stat or stat.split(\') \', 1)[1].startswith(\'Z\'):\n                        break\n                    time.sleep(0.5)\n                else:\n                    os.kill(int(procdir.name), 9)\n                    time.sleep(2)\n        except (FileNotFoundError, ProcessLookupError, PermissionError):\n            pass\n    old_proc = globals().get(\'qwen_server\')\n    if old_proc is not None:\n        try:\n            old_proc.wait(timeout=5)\n        except Exception:\n            pass\n    deadline = time.monotonic() + 20\n    while time.monotonic() < deadline:\n        alive = []\n        for procdir in Path(\'/proc\').iterdir():\n            try:\n                if procdir.name.isdigit() and Path((procdir / \'cmdline\').read_bytes().decode(errors=\'replace\').split(\'\\0\')[0]).name == \'llama-server\':\n                    alive.append(procdir.name)\n            except (FileNotFoundError, ProcessLookupError, PermissionError):\n                pass\n        if not alive:\n            return\n        time.sleep(1)\n    raise RuntimeError(\'旧 llama-server 停不掉（PID \' + \', \'.join(alive) + \'），已中止；请用停机按钮释放运行时后重来。\')\n\ndef _mem_available():\n    for l in Path(\'/proc/meminfo\').read_text().splitlines():\n        if l.startswith(\'MemAvailable:\'):\n            return int(l.split()[1]) * 1024\n    return 0\n\nSERVICE = \'svc:qwen\'\nSERVICE_FQDN = "qwen.<YOUR_TAILNET>.ts.net"\nSERVICE_TAG = \'tag:colab\'\nTS_HOSTNAME = SPEC[\'ts_hostname\']\nTS_CLIENT_ID = "<YOUR_TS_OAUTH_CLIENT_ID>"\nTS_CLIENT_SECRET = "<YOUR_TS_OAUTH_CLIENT_SECRET>"\nTS = \'/content/tsbin/tailscale\'\nTS_SOCK = \'/var/run/tailscale/tailscaled.sock\'\nTS_STATE = Path(\'/content/tailscaled.state\')\nTS_DIR = Path(\'/content/tsstatedir\')\n\ndef sh(c):\n    subprocess.run(c, shell=True, check=True)\n\ndef health():\n    try:\n        return requests.get(\'http://127.0.0.1:%d/health\' % PORT, timeout=2).status_code == 200\n    except Exception:\n        return False\n\ngpu_info = subprocess.getoutput(\'nvidia-smi --query-gpu=name,memory.total --format=csv,noheader,nounits\').strip()\nif not gpu_info or \',\' not in gpu_info:\n    raise RuntimeError(\'未检测到 NVIDIA GPU，请在右上角切换到 G4 GPU 运行时。输出：\' + gpu_info)\ngpu_name, gpu_mib_str = [x.strip() for x in gpu_info.split(\',\', 1)]\ngpu_mib = int(gpu_mib_str)\nprint(f\'检测到 GPU: {gpu_name} ({gpu_mib / 1024:.1f} GiB VRAM)\', flush=True)\nif gpu_mib < SPEC[\'min_vram_mib\']:\n    raise RuntimeError(f\'当前 GPU 显存仅 {gpu_mib / 1024:.1f} GiB（{gpu_name}），低于要求的 {SPEC["min_vram_mib"] / 1024:.0f} GiB，请切换到 G4 高内存运行时。\')\n\nROOT.mkdir(parents=True, exist_ok=True)\nfor _d in sorted(set(_local_path(f).parent for f in FILES)):\n    _d.mkdir(parents=True, exist_ok=True)\nDIST.mkdir(parents=True, exist_ok=True)\nKEY_FILE.write_text(KEY + \'\\n\', encoding=\'utf-8\', newline=\'\')\nKEY_FILE.chmod(0o600)\nAUTH = {\'Authorization\': \'Bearer \' + KEY}\n\ndef prepare_llama_cpp():\n    if next(DIST.rglob(\'llama-server\'), None) is not None:\n        return\n    print(f\'[并行任务] 下载并解压 llama.cpp {CPP_TAG} (CUDA 12.8) ...\', flush=True)\n    sh(f\'cd {DIST} && \'\n       f\'curl -fsSL -O {CPP_URL}/llama-{CPP_TAG}-bin-ubuntu-cuda-12.8-x64.tar.gz && \'\n       f\'curl -fsSL -O {CPP_URL}/cudart-llama-{CPP_TAG}-bin-ubuntu-cuda-12.8-x64.tar.gz && \'\n       f\'tar -xzf llama-{CPP_TAG}-bin-ubuntu-cuda-12.8-x64.tar.gz && \'\n       f\'tar -xzf cudart-llama-{CPP_TAG}-bin-ubuntu-cuda-12.8-x64.tar.gz && \'\n       f\'rm -f *.tar.gz\')\n    print(f\'[并行任务] llama.cpp {CPP_TAG} 就绪\', flush=True)\n\ndef ts(*a, timeout=60):\n    return subprocess.run([TS, \'--socket=\' + TS_SOCK, *a],\n                          stdin=subprocess.DEVNULL, capture_output=True, text=True, timeout=timeout)\n\ndef ts_state():\n    try:\n        return json.loads(ts(\'status\', \'--json\').stdout).get(\'BackendState\', \'\')\n    except Exception:\n        return \'\'\n\ndef _start_mcp_bridge():\n    srv = socket.socket(socket.AF_INET, socket.SOCK_STREAM)\n    srv.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)\n    try:\n        srv.bind((\'127.0.0.1\', 3426))\n    except OSError:\n        return\n    srv.listen(32)\n    def _pipe(a, b):\n        try:\n            while (d := a.recv(65536)):\n                b.sendall(d)\n        except Exception:\n            pass\n        finally:\n            try: a.close()\n            except Exception: pass\n            try: b.close()\n            except Exception: pass\n    while True:\n        try:\n            cli, _ = srv.accept()\n            up = socket.create_connection((\'127.0.0.1\', 1055), timeout=10)\n            up.sendall(bytes([5, 1, 0]))\n            assert up.recv(2) == bytes([5, 0])\n            up.sendall(bytes([5, 1, 0, 1]) + socket.inet_aton("<YOUR_LOCAL_TAILSCALE_IP>") + struct.pack(\'>H\', 3426))\n            assert up.recv(10)[1] == 0\n            threading.Thread(target=_pipe, args=(cli, up), daemon=True).start()\n            threading.Thread(target=_pipe, args=(up, cli), daemon=True).start()\n        except Exception:\n            pass\n\ndef prepare_tailscale():\n    if not Path(TS).exists():\n        sh(\'curl -fsSL https://pkgs.tailscale.com/stable/tailscale_1.104.1_amd64.tgz | tar -xz -C /content && \'\n           \'mv /content/tailscale_1.104.1_amd64 /content/tsbin\')\n    TS_DIR.mkdir(parents=True, exist_ok=True)\n    Path(\'/var/run/tailscale\').mkdir(parents=True, exist_ok=True)\n    if ts_state() in (\'\', \'NoState\', \'Stopped\'):\n        subprocess.Popen(\n            [\'/content/tsbin/tailscaled\', \'--tun=userspace-networking\',\n             \'--socks5-server=localhost:1055\', \'--state=\' + str(TS_STATE),\n             \'--statedir=\' + str(TS_DIR), \'--socket=\' + TS_SOCK],\n            stdout=open(\'/content/tailscaled.log\', \'a\', encoding=\'utf-8\', newline=\'\'),\n            stderr=subprocess.STDOUT, stdin=subprocess.DEVNULL, start_new_session=True,\n        )\n        for _ in range(40):\n            if Path(TS_SOCK).exists():\n                break\n            time.sleep(0.5)\n        for _ in range(40):\n            if ts_state() in (\'Running\', \'NeedsLogin\'):\n                break\n            time.sleep(0.5)\n    if ts_state() != \'Running\':\n        token = requests.post(\n            \'https://api.tailscale.com/api/v2/oauth/token\',\n            data={\'client_id\': TS_CLIENT_ID, \'client_secret\': TS_CLIENT_SECRET}, timeout=30\n        ).json()[\'access_token\']\n        key = requests.post(\n            \'https://api.tailscale.com/api/v2/tailnet/-/keys\',\n            headers={\'Authorization\': \'Bearer \' + token},\n            json={\'capabilities\': {\'devices\': {\'create\': {\n                \'reusable\': False, \'ephemeral\': True, \'preauthorized\': True, \'tags\': [SERVICE_TAG]\n            }}}}, timeout=30\n        ).json()[\'key\']\n        ts(\'up\', \'--auth-key=\' + key, \'--hostname=\' + TS_HOSTNAME,\n           \'--advertise-tags=\' + SERVICE_TAG, \'--accept-dns=false\', \'--timeout=60s\', timeout=120)\n        for _ in range(120):\n            if ts_state() == \'Running\':\n                break\n            time.sleep(0.5)\n    ts(\'serve\', \'reset\')\n    ts(\'serve\', \'--bg\', \'--service=\' + SERVICE, \'--tcp=443\', \'tcp://127.0.0.1:%d\' % PORT, timeout=90)\n    ts(\'serve\', \'--bg\', \'--https=443\', \'http://127.0.0.1:%d\' % PORT, timeout=90)\n    threading.Thread(target=_start_mcp_bridge, daemon=True).start()\n    print(f\'[并行任务] Tailscale Service ({SERVICE_FQDN}) 与本机小说文件 MCP 隧道已就绪\', flush=True)\n\nbg_executor = concurrent.futures.ThreadPoolExecutor(max_workers=2)\nfut_cpp = bg_executor.submit(prepare_llama_cpp)\nfut_ts = bg_executor.submit(prepare_tailscale)\n\nsubprocess.run([\'mount\', \'-o\', \'remount,size=115G\', \'/dev/shm\'], capture_output=True)\ntmpfs_dirs = []\nfor mline in Path(\'/proc/mounts\').read_text().splitlines():\n    mparts = mline.split()\n    if len(mparts) >= 3 and mparts[2] == \'tmpfs\' and mparts[1] in (\'/dev/shm\', \'/var/colab\'):\n        tmpfs_dirs.append(Path(mparts[1]))\nmp_free = {mp: shutil.disk_usage(mp).free for mp in tmpfs_dirs}\nmeminfo_txt = Path(\'/proc/meminfo\').read_text()\navail_kb = next((int(l.split()[1]) for l in meminfo_txt.splitlines() if l.startswith(\'MemAvailable:\')), 0)\nram_budget = max(0, avail_kb * 1024 - SPEC[\'ram_reserve_gib\'] * (1024**3))\n\nreleased_reuse = set()\nif RELEASE_STATE.exists():\n    try:\n        rel_state = json.loads(RELEASE_STATE.read_text(encoding=\'utf-8\'))\n        rel_ok = (isinstance(rel_state, dict) and isinstance(rel_state.get(\'pid\'), int)\n                  and isinstance(rel_state.get(\'files\'), list)\n                  and all(isinstance(n, str) and n in FILE_NAMES for n in rel_state[\'files\']))\n    except Exception:\n        rel_state, rel_ok = {}, False\n    pid_now = _server_pid()\n    if rel_ok and pid_now == rel_state[\'pid\'] and rel_state.get(\'proc_id\') and _proc_identity(pid_now) == rel_state[\'proc_id\']:\n        released_reuse = set(rel_state[\'files\'])\n        print(f\'服务仍在运行：{len(released_reuse)} 个已释放内存副本的分片无需重新下载\', flush=True)\n    else:\n        if not rel_ok:\n            print(\'⚠️ released-copies.json 无法识别，缺失的分片会重新下载\', flush=True)\n        RELEASE_STATE.unlink(missing_ok=True)\nif not released_reuse and _server_pid() and any(not _local_path(f).exists() for f in FILES):\n    # 记录不可用、服务还在且文件已删：先停服务再补下载，避免服务占着内存时又往内存盘塞几十 GiB。\n    print(\'⚠️ 服务在运行，但分片副本已删且释放记录不可用：先停掉旧服务，再重新下载。\', flush=True)\n    _stop_llama_servers()\n\nsorted_files = sorted(FILES, key=lambda x: x.get(\'priority\', 2))\nall_targets = [\n    (\'https://huggingface.co/\' + f[\'repo\'] + \'/resolve/\' + f[\'rev\'] + \'/\' + f[\'path\'],\n     _local_path(f), f[\'size\'])\n    for f in sorted_files\n]\ndownloads = []\ntmpfs_bytes = 0\ndisk_bytes = 0\nfor orig_url, logical_path, size in all_targets:\n    if logical_path.name in released_reuse:\n        continue\n    if logical_path.exists() and logical_path.stat().st_size == size:\n        continue\n    if logical_path.is_symlink() or logical_path.exists():\n        logical_path.unlink(missing_ok=True)\n    real_path = logical_path\n    if ram_budget >= size:\n        for mp in tmpfs_dirs:\n            if mp_free.get(mp, 0) >= size + 4 * (1024**3):\n                cand = mp / SPEC[\'slug\'] / logical_path.name\n                cand.parent.mkdir(parents=True, exist_ok=True)\n                # 只看大小会把 truncate 出来的稀疏占位文件（下载中断、上次报错）当成已下载；tmpfs 的 st_blocks 是实际占用。\n                if cand.exists() and cand.stat().st_size == size and cand.stat().st_blocks * 512 >= size:\n                    logical_path.symlink_to(cand)\n                    mp_free[mp] -= size\n                    ram_budget -= size\n                    real_path = None\n                    break\n                cand.unlink(missing_ok=True)\n                with cand.open(\'wb\') as f:\n                    f.truncate(size)\n                logical_path.symlink_to(cand)\n                mp_free[mp] -= size\n                ram_budget -= size\n                tmpfs_bytes += size\n                real_path = cand\n                break\n    if real_path is None:\n        continue\n    if real_path == logical_path:\n        free_disk = shutil.disk_usage(\'/content\').free\n        if free_disk < disk_bytes + size + 6 * (1024**3):\n            raise RuntimeError(\'本地 /content 与 tmpfs 空间不足，请重建运行时。\')\n        with real_path.open(\'wb\') as f:\n            f.truncate(size)\n        disk_bytes += size\n    downloads.append((orig_url, str(real_path), size))\n\nfreshly_downloaded = bool(downloads)\nif downloads:\n    need_bytes = sum(size for _, _, size in downloads)\n    print(f\'待并发下载: {need_bytes / (1024**3):.2f} GiB ({len(downloads)} 个文件) → DDR5 内存盘 tmpfs: {tmpfs_bytes / (1024**3):.1f} GiB | 本地磁盘: {disk_bytes / (1024**3):.1f} GiB\', flush=True)\n    CHUNK = 64 * 1024 * 1024\n    hf_tok = os.environ.get(\'HF_TOKEN\', \'\').strip()\n    if not hf_tok:\n        try:\n            from google.colab import userdata\n            hf_tok = (userdata.get(\'HF_TOKEN\') or \'\').strip()\n        except Exception:\n            hf_tok = \'\'\n\n    def _resolve_cdn(item):\n        from email.utils import parsedate_to_datetime\n        from datetime import datetime, timezone\n        orig_url, real_path, size = item\n        req_hdr = {\'Range\': \'bytes=0-0\'}\n        if hf_tok:\n            req_hdr[\'Authorization\'] = \'Bearer \' + hf_tok\n        for attempt in range(5):\n            try:\n                with requests.get(orig_url, headers=req_hdr, stream=True, timeout=30) as r:\n                    if r.status_code == 429:\n                        ra = r.headers.get(\'Retry-After\', \'\')\n                        try:\n                            wait_s = float(ra)\n                        except ValueError:\n                            try:\n                                wait_s = max(1.0, (parsedate_to_datetime(ra) - datetime.now(timezone.utc)).total_seconds())\n                            except Exception:\n                                wait_s = 2 ** (attempt + 1)\n                        time.sleep(min(wait_s, 60.0))\n                        continue\n                    r.raise_for_status()\n                    if r.status_code != 206:\n                        raise RuntimeError(f\'Range probe expected 206, got {r.status_code}\')\n                    cdn_url = r.url\n                    return [(cdn_url, orig_url, real_path, start, min(start + CHUNK, size) - 1)\n                            for start in range(0, size, CHUNK)]\n            except Exception:\n                if attempt == 4:\n                    raise\n                time.sleep(2 ** (attempt + 1))\n\n    range_tasks = []\n    with concurrent.futures.ThreadPoolExecutor(max_workers=len(downloads)) as rpool:\n        for chunk_list in rpool.map(_resolve_cdn, downloads):\n            range_tasks.extend(chunk_list)\n\n    tasks_file = Path(\'/tmp/colab_kernel_dl_tasks.json\')\n    tasks_file.write_text(json.dumps({\'need_bytes\': need_bytes, \'tasks\': range_tasks}), encoding=\'utf-8\', newline=\'\')\n    dl_script = Path(\'/tmp/colab_kernel_fast_dl.py\')\n    dl_script.write_text(\'\'\'import concurrent.futures, json, multiprocessing as mp, os, sys, time\nimport requests\nfrom requests.adapters import HTTPAdapter\n\ndata = json.loads(open(sys.argv[1], encoding="utf-8").read())\nneed_bytes = data["need_bytes"]\ntasks = data["tasks"]\nN_PROC = 8\nN_THREADS = 4\n\ndone_bytes = mp.Value("q", 0)\nt_start = time.monotonic()\nlast_print = mp.Value("d", t_start)\n\ndef worker_batch(batch):\n    sess = requests.Session()\n    adapter = HTTPAdapter(pool_connections=N_THREADS + 4, pool_maxsize=N_THREADS + 4)\n    sess.mount("https://", adapter)\n    sess.mount("http://", adapter)\n\n    def fetch_one(task):\n        cdn_url, orig_url, target, start, end = task\n        expected = end - start + 1\n        for attempt in range(5):\n            url = cdn_url if attempt < 2 else orig_url\n            try:\n                with sess.get(url, headers={"Range": f"bytes={start}-{end}"}, stream=True, timeout=(15, 120)) as resp:\n                    if resp.status_code != 206:\n                        raise RuntimeError(f"HTTP {resp.status_code}")\n                    fd = os.open(target, os.O_RDWR)\n                    try:\n                        pos = start\n                        for blk in resp.iter_content(8 * 1024 * 1024):\n                            if blk:\n                                os.pwrite(fd, blk, pos)\n                                pos += len(blk)\n                    finally:\n                        os.close(fd)\n                    if pos - start != expected:\n                        raise RuntimeError(f"short read {pos - start} != {expected}")\n                with done_bytes.get_lock():\n                    done_bytes.value += expected\n                    now = time.monotonic()\n                    if now - last_print.value >= 8:\n                        rate = done_bytes.value / max(now - t_start, 0.1) / (1024**2)\n                        print(f"  并发下载进度: {done_bytes.value / need_bytes:.0%} ({done_bytes.value / (1024**3):.1f} / {need_bytes / (1024**3):.1f} GiB)，均速 {rate:.0f} MB/s", flush=True)\n                        last_print.value = now\n                return\n            except Exception as err:\n                if attempt == 4:\n                    raise RuntimeError(f"{os.path.basename(target)} [{start}-{end}] 失败: {err}") from err\n                time.sleep(1.5 * (attempt + 1))\n\n    with concurrent.futures.ThreadPoolExecutor(max_workers=N_THREADS) as tpool:\n        futs = [tpool.submit(fetch_one, t) for t in batch]\n        for f in concurrent.futures.as_completed(futs):\n            f.result()\n\nif __name__ == "__main__":\n    batches = [tasks[i::N_PROC] for i in range(N_PROC) if tasks[i::N_PROC]]\n    ctx = mp.get_context("fork")\n    with ctx.Pool(processes=len(batches)) as pool:\n        pool.map(worker_batch, batches)\n\'\'\', encoding=\'utf-8\', newline=\'\')\n    t_dl_start = time.monotonic()\n    subprocess.run([sys.executable, str(dl_script), str(tasks_file)], check=True)\n    dl_sec = max(time.monotonic() - t_dl_start, 0.1)\n    print(f\'全部权重下载完成：用时 {dl_sec:.1f}s，平均 {need_bytes / (1024**2) / dl_sec:.0f} MB/s\', flush=True)\nelse:\n    print(f\'全部 {len(FILES)} 个权重与附件已存在于内存盘/本地，跳过下载。\', flush=True)\n\ndef verify_hashes():\n    # 仅调试模式调用。下载决策时已经对过一次字节大小；这里再 stat 一遍。\n    try:\n        for item in FILES:\n            path = _local_path(item)\n            if path.name in released_reuse:\n                continue\n            actual = path.stat().st_size if path.exists() else -1\n            if actual != item[\'size\']:\n                if path.exists():\n                    path.resolve().unlink(missing_ok=True)\n                    path.unlink(missing_ok=True)\n                raise RuntimeError(\'权重存在性或字节大小校验失败，请重新运行首格：\' + path.name)\n            print(\'SIZE OK:\', path.name, item[\'size\'], flush=True)\n    except BaseException:\n        if globals().get(\'qwen_server\') is not None:\n            qwen_server.terminate()\n        raise\n\nPARALLEL_SLOTS = 1\nSLOT_CONTEXT_TOKENS = 262144\nTOTAL_CONTEXT_TOKENS = SLOT_CONTEXT_TOKENS * PARALLEL_SLOTS\nUBATCH_SIZE = 2048\nMAX_OUTPUT_TOKENS = 65536\nSLOT_SAVE_DIR = ROOT / \'slot-saves\'\nSLOT_SAVE_DIR.mkdir(parents=True, exist_ok=True)\nCHAT_TMPL = ROOT / \'chat-template-clean.jinja\'\nUI_CFG = ROOT / \'webui-config.json\'\nnew_ui_text = json.dumps(UI_CFG_DATA, ensure_ascii=False, indent=2) + \'\\n\'\nold_ui_on_disk = UI_CFG.read_text(encoding=\'utf-8\') if UI_CFG.exists() else \'\'\n\nple_name = SPEC.get(\'ple_shard_name\')\nif ple_name:\n    ple_real = str((SHARD_DIR / ple_name).resolve())\n    ple_in_tmpfs = ple_real.startswith(TMPFS_PREFIXES)\nelse:\n    ple_in_tmpfs = False\n\n# PLE 表分片在运行期由 CPU 直接 mmap 读取，必须常驻；其余落在内存盘的分片加载后可删。\nrelease_keep = {ple_name} if ple_in_tmpfs else set()\nrelease_candidates = [f for f in FILES\n                      if _local_path(f).name not in release_keep\n                      and _local_path(f).exists()\n                      and str(_local_path(f).resolve()).startswith(TMPFS_PREFIXES)]\nreleasable_bytes = sum(f[\'size\'] for f in release_candidates)\n# 只用于新启动的服务；已在运行的服务保留自己的 --cache-ram（configured_server 不比较这一项，免得为改缓存上限重启）。\n# 只拿回收内存的一半给提示缓存，另一半留作余量（mmap 模式下 CPU 张量所在分片删文件后仍占内存）。\nPROMPT_CACHE_MIB = min(65536, 16384 + releasable_bytes // (2 * 1024**2))\n\ndef release_gpu_only_copies():\n    pid, args = _server_proc()\n    todo = [f for f in release_candidates if _local_path(f).exists()]\n    if not pid or not todo or not health():\n        return\n    names = sorted({_local_path(f).name for f in todo} | released_reuse)\n    tmp_state = RELEASE_STATE.with_suffix(\'.tmp\')\n    tmp_state.write_text(json.dumps({\'pid\': pid, \'proc_id\': _proc_identity(pid), \'files\': names}), encoding=\'utf-8\', newline=\'\')\n    os.replace(tmp_state, RELEASE_STATE)\n    if DEBUG_VERIFY:\n        before = _mem_available()\n    for f in todo:\n        lp = _local_path(f)\n        lp.resolve().unlink(missing_ok=True)\n        lp.unlink(missing_ok=True)\n    expect = sum(f[\'size\'] for f in todo)\n    if DEBUG_VERIFY:\n        time.sleep(1)\n        gained = _mem_available() - before\n        cap = _cache_ram_of(args) or 0\n        print(f\'已删除 {len(todo)} 个显存专用分片的内存盘副本：可用内存 +{gained / 1024**3:.1f} GiB（预计 {expect / 1024**3:.1f} GiB）| 当前服务提示缓存上限 {cap / 1024:.0f} GiB\', flush=True)\n        if gained < expect * 0.5:\n            print(\'⚠️ 实际回收不到预计的一半：部分分片仍被服务映射，提示缓存别塞满；服务重启后才会真正归还。\', flush=True)\n    else:\n        print(f\'已删除 {len(todo)} 个显存专用分片的内存盘副本（预计 {expect / 1024**3:.1f} GiB）\', flush=True)\n\nif MODEL.exists():\n    with MODEL.open(\'rb\') as f:\n        raw_head = f.read(16 * 1024 * 1024)\n    HEAD_CACHE.write_bytes(raw_head)\nelse:\n    raw_head = HEAD_CACHE.read_bytes()\ntmpl_key = b\'tokenizer.chat_template\'\nkey_pos = raw_head.index(struct.pack(\'<Q\', len(tmpl_key)) + tmpl_key)\nval_pos = key_pos + 8 + len(tmpl_key)\nassert struct.unpack_from(\'<I\', raw_head, val_pos)[0] == 8, \'tokenizer.chat_template 不是 GGUF 字符串类型\'\ntmpl_len = struct.unpack_from(\'<Q\', raw_head, val_pos + 4)[0]\nassert val_pos + 12 + tmpl_len <= len(raw_head), \'模板超出前 16 MiB\'\ntmpl_str = raw_head[val_pos + 12:val_pos + 12 + tmpl_len].decode(\'utf-8\')\n\n# 官方模板只有 xhigh/medium/low 三档；换成四档，并按 effort_prompts 在 system 顶部注入一句中文引导。\nuniversal_effort_block = \'\'\'{%- set reasoning_instructions = \'\' %}\n{%- if enable_thinking is undefined or enable_thinking is true %}\n    {%- set resolved_reasoning_effort = reasoning_effort|default(\'@default@\') %}\n    {%- if resolved_reasoning_effort not in (\'xhigh\', \'high\', \'medium\', \'low\') %}\n        {{- raise_exception(\'Unexpected reasoning effort \' ~ reasoning_effort ~ \'. Supported types are xhigh, high, medium, and low (default @default@).\') }}\n    {%- endif %}\n    {%- if resolved_reasoning_effort == \'xhigh\' %}\n        {%- set reasoning_instructions = \'@xhigh@\' %}\n    {%- elif resolved_reasoning_effort == \'high\' %}\n        {%- set reasoning_instructions = \'@high@\' %}\n    {%- elif resolved_reasoning_effort == \'medium\' %}\n        {%- set reasoning_instructions = \'@medium@\' %}\n    {%- elif resolved_reasoning_effort == \'low\' %}\n        {%- set reasoning_instructions = \'@low@\' %}\n    {%- endif %}\n{%- endif %}\'\'\'\nfor _lvl, _txt in SPEC[\'effort_prompts\'].items():\n    universal_effort_block = universal_effort_block.replace(\'@%s@\' % _lvl, _txt)\nuniversal_effort_block = universal_effort_block.replace(\'@default@\', SPEC.get(\'reasoning_effort_default\') or \'high\')\n\nfam = SPEC.get(\'template_family\', \'qwen38_fn\')\nif fam == \'qwen38_fn\':\n    old_effort_block = \'\'\'{%- if enable_thinking is undefined or enable_thinking is true %}\n    {%- set resolved_reasoning_effort = reasoning_effort|default(\'xhigh\') %}\n    {%- if resolved_reasoning_effort not in (\'xhigh\', \'medium\', \'low\') %}\n        {{- raise_exception(\'Unexpected reasoning effort \' ~ reasoning_effort ~ \'. Supported types are xhigh (default), medium, and low.\') }}\n    {%- endif %}\n    {%- if resolved_reasoning_effort == \'xhigh\' %}\n        {%- set reasoning_instructions = \'Reasoning effort is set to xhigh. Please think carefully through the task, validate key assumptions, consider plausible alternatives, and prioritize correctness, consistency, and clarity in the final answer.\' %}\n    {%- elif resolved_reasoning_effort == \'low\' %}\n        {%- set reasoning_instructions = \'Reasoning effort is set to low. Keep your thinking brief and focused, moving directly to the conclusion without unnecessary elaboration.\' %}\n    {%- endif %}\n{%- endif %}\'\'\'\n    # Unsloth 导出的原版 Qwen3.8 模板在 resolved_reasoning_effort 下多塞了 3 行 high->xhigh 折合；先剥掉再统一替换成四档中文引导。\n    unsloth_high_shim = \'\'\'    {%- if resolved_reasoning_effort == \'high\' %}\n        {%- set resolved_reasoning_effort = \'xhigh\' %}\n    {%- endif %}\n\'\'\'\n    tmpl_str = tmpl_str.replace(unsloth_high_shim, \'\', 1)\n    assert old_effort_block in tmpl_str, \'未找到预期的 reasoning_effort 模板块\'\n    tmpl_str = tmpl_str.replace(old_effort_block, universal_effort_block, 1)\n    old_preserve_line = \'{%- if preserve_thinking is undefined or preserve_thinking is true or loop.index0 > ns.last_query_index %}\'\n    assert old_preserve_line in tmpl_str, \'未找到预期的 preserve_thinking 模板行\'\n    # 官方默认 preserve_thinking 未定义即为真，历史轮次会被套上空 <think></think>。\n    # 只保留当前工具链里 reasoning_content 非空的轮次，避免 few-shot 诱导模型秒关思考。\n    # 生成起手块不改：官方只在 enable_thinking=false 时关思考，low 照常思考。\n    tmpl_str = tmpl_str.replace(\n        old_preserve_line,\n        \'{%- if loop.index0 > ns.last_query_index and reasoning_content %}\',\n        1,\n    )\nelif fam == \'gemma4\':\n    old_think_body = \'\'\'        {{- \'<|think|>\\\\n\' -}}\n        {%- set ns.prev_message_type = \'think\' -%}\'\'\'\n    new_think_body = \'\'\'        {{- \'<|think|>\\\\n\' + (reasoning_instructions + \'\\\\n\' if reasoning_instructions else \'\') -}}\n        {%- set ns.prev_message_type = \'think\' -%}\'\'\'\n    assert old_think_body in tmpl_str, \'未找到预期的 gemma4 <|think|> 注入块\'\n    tmpl_str = tmpl_str.replace(\n        \'{%- set enable_thinking = enable_thinking | default(false) -%}\',\n        \'{%- set enable_thinking = enable_thinking | default(true) -%}\',\n        1,\n    )\n    tmpl_str = (\n        "{%- if enable_thinking is not defined %}{%- set enable_thinking = true %}{%- endif %}\\n"\n        + universal_effort_block\n        + "\\n"\n        + tmpl_str.replace(old_think_body, new_think_body, 1)\n    )\nelif fam == \'glm5_next\':\n    old_glm_head = \'\'\'{%- set effective_reasoning_effort = reasoning_effort if reasoning_effort is defined and reasoning_effort in [\'low\', \'high\'] else \'max\' -%}\n{%- if effective_reasoning_effort is not none -%}<|system|>Reasoning Effort: {{ effective_reasoning_effort | capitalize }}{%- endif -%}\n{%- set clear_thinking = clear_thinking if clear_thinking is defined else false -%}\'\'\'\n    new_glm_head = (\n        universal_effort_block\n        + "\\n{%- set effective_reasoning_effort = \'max\' if resolved_reasoning_effort == \'xhigh\' else (\'low\' if resolved_reasoning_effort == \'low\' else \'high\') -%}"\n        + "\\n{%- if effective_reasoning_effort is not none -%}<|system|>Reasoning Effort: {{ effective_reasoning_effort | capitalize }}{{ (\'\\\\n\' + reasoning_instructions) if reasoning_instructions else \'\' }}{%- endif -%}"\n        + "\\n{%- set clear_thinking = clear_thinking if clear_thinking is defined else true -%}"\n    )\n    assert old_glm_head in tmpl_str, \'未找到预期的 glm5_next reasoning_effort 模板块\'\n    tmpl_str = tmpl_str.replace(old_glm_head, new_glm_head, 1)\nelif fam == \'qwen35\':\n    old_tools_head = "{%- if tools and tools is iterable and tools is not mapping %}\\n    {{- \'<|im_start|>system\\\\n\' }}"\n    new_tools_head = (\n        universal_effort_block\n        + "\\n{%- if tools and tools is iterable and tools is not mapping %}\\n"\n        + "    {{- \'<|im_start|>system\\\\n\' }}\\n"\n        + "    {%- if reasoning_instructions %}\\n"\n        + "        {{- reasoning_instructions + \'\\\\n\\\\n\' }}\\n"\n        + "    {%- endif %}"\n    )\n    assert old_tools_head in tmpl_str, \'未找到预期的 qwen35 tools system 头部\'\n    tmpl_str = tmpl_str.replace(old_tools_head, new_tools_head, 1)\n    old_tools_sys = \'\'\'    {%- if messages[0].role == \'system\' %}\n        {%- set content = render_content(messages[0].content, false, true)|trim %}\n        {%- if content %}\n            {{- \'\\\\n\\\\n\' + content }}\n        {%- endif %}\n    {%- endif %}\n    {{- \'<|im_end|>\\\\n\' }}\n{%- else %}\n    {%- if messages[0].role == \'system\' %}\n        {%- set content = render_content(messages[0].content, false, true)|trim %}\n        {{- \'<|im_start|>system\\\\n\' + content + \'<|im_end|>\\\\n\' }}\n    {%- endif %}\n{%- endif %}\'\'\'\n    new_tools_sys = \'\'\'    {%- if messages[0].role == \'system\' %}\n        {%- set content = render_content(messages[0].content, false, true)|trim %}\n        {%- if content %}\n            {{- \'\\\\n\\\\n\' + content }}\n        {%- endif %}\n    {%- endif %}\n    {{- \'<|im_end|>\\\\n\' }}\n{%- else %}\n    {%- if messages[0].role == \'system\' %}\n        {%- set content = render_content(messages[0].content, false, true)|trim %}\n        {{- \'<|im_start|>system\\\\n\' + (reasoning_instructions + \'\\\\n\\\\n\' if reasoning_instructions else \'\') + content + \'<|im_end|>\\\\n\' }}\n    {%- elif reasoning_instructions %}\n        {{- \'<|im_start|>system\\\\n\' + reasoning_instructions + \'<|im_end|>\\\\n\' }}\n    {%- endif %}\n{%- endif %}\'\'\'\n    assert old_tools_sys in tmpl_str, \'未找到预期的 qwen35 system 分支块\'\n    tmpl_str = tmpl_str.replace(old_tools_sys, new_tools_sys, 1)\n    old_hist_think = \'{%- if loop.index0 > ns.last_query_index %}\'\n    assert old_hist_think in tmpl_str, \'未找到预期的历史思考保留条件\'\n    tmpl_str = tmpl_str.replace(old_hist_think, \'{%- if loop.index0 > ns.last_query_index and reasoning_content %}\', 1)\n    qwen_gen_think_old = \'\'\'    {%- if enable_thinking is defined and enable_thinking is false %}\n        {{- \'<think>\\\\n\\\\n</think>\\\\n\\\\n\' }}\n    {%- else %}\n        {{- \'<think>\\\\n\' }}\n    {%- endif %}\'\'\'\n    qwen_gen_think_new = \'\'\'    {%- if (enable_thinking is defined and enable_thinking is false) or (reasoning_effort is defined and reasoning_effort == \'low\') %}\n        {{- \'<think>\\\\n\\\\n</think>\\\\n\\\\n\' }}\n    {%- else %}\n        {{- \'<think>\\\\n\' }}\n    {%- endif %}\'\'\'\n    assert qwen_gen_think_old in tmpl_str, \'未找到预期的 add_generation_prompt 思考起手块\'\n    tmpl_str = tmpl_str.replace(qwen_gen_think_old, qwen_gen_think_new, 1)\nelif fam == \'nemotron\':\n    old_trunc = \'{%- set truncate_history_thinking = truncate_history_thinking if truncate_history_thinking is defined else True %}\'\n    assert old_trunc in tmpl_str, \'未找到预期的 truncate_history_thinking 行\'\n    tmpl_str = tmpl_str.replace(old_trunc, \'{%- set truncate_history_thinking = false %}\', 1)\n    old_sys_block = \'\'\'{%- if messages[0]["role"] == "system" %}\n    {%- set system_message = messages[0]["content"] %}\n    {%- set loop_messages = messages[1:] %}\n{%- else %}\n    {%- set system_message = "" %}\n    {%- set loop_messages = messages %}\n{%- endif %}\'\'\'\n    new_sys_block = (\n        universal_effort_block\n        + \'\'\'\n{%- if messages[0]["role"] == "system" %}\n    {%- set system_message = (reasoning_instructions ~ "\\\\n\\\\n" if reasoning_instructions else "") ~ render_content(messages[0]["content"]) %}\n    {%- set loop_messages = messages[1:] %}\n{%- else %}\n    {%- set system_message = reasoning_instructions %}\n    {%- set loop_messages = messages %}\n{%- endif %}\'\'\'\n    )\n    assert old_sys_block in tmpl_str, \'未找到预期的 nemotron system_message 块\'\n    tmpl_str = tmpl_str.replace(old_sys_block, new_sys_block, 1)\n    nemo_gen_think_old = \'\'\'{%- if add_generation_prompt %}\n    {%- if enable_thinking %}\n        {{- \'<|im_start|>assistant\\\\n<think>\\\\n\' }}\n    {%- else %}\n        {{- \'<|im_start|>assistant\\\\n<think></think>\' }}\n    {%- endif %}\n{%- endif %}\'\'\'\n    nemo_gen_think_new = \'\'\'{%- if add_generation_prompt %}\n    {{- \'<|im_start|>assistant\\\\n\' }}\n    {%- if (enable_thinking is defined and enable_thinking is false) or (reasoning_effort is defined and reasoning_effort == \'low\') %}\n        {{- \'<think></think>\' }}\n    {%- else %}\n        {{- \'<think>\\\\n\' }}\n    {%- endif %}\n{%- endif -%}\'\'\'\n    assert nemo_gen_think_old in tmpl_str, \'未找到预期的 nemotron add_generation_prompt 思考起手块\'\n    tmpl_str = tmpl_str.replace(nemo_gen_think_old, nemo_gen_think_new, 1)\nelse:\n    raise RuntimeError(f\'未知 template_family={fam}：必须在 colab_kernel.py 中接入 universal_effort_block 四档思考支持\')\n\nif LORA and LORA.exists():\n    # GCSA-AiLab/GLM-5.3-Flash-Abliterated-LoRA-GGUF 头部写成了 glm5next（缺连字符），原地修补为 glm5-next 并消掉 1 字节对齐填充，保持张量偏移与文件大小不变。\n    with LORA.open(\'r+b\') as _lf:\n        _lbuf = _lf.read(32768)\n        _bad_arch = b\'\\x08\\x00\\x00\\x00\\x00\\x00\\x00\\x00glm5next\'\n        _good_arch = b\'\\x09\\x00\\x00\\x00\\x00\\x00\\x00\\x00glm5-next\'\n        if _bad_arch in _lbuf:\n            _pad_idx = _lbuf.find(b\'\\x00\' * 8, 16384)\n            assert _pad_idx > 0, \'LoRA GGUF 未找到对齐填充区\'\n            _lbuf = _lbuf.replace(_bad_arch, _good_arch, 1)\n            _lbuf = _lbuf[:_pad_idx] + _lbuf[_pad_idx + 1:]\n            _lf.seek(0)\n            _lf.write(_lbuf)\nold_tmpl_on_disk = CHAT_TMPL.read_text(encoding=\'utf-8\') if CHAT_TMPL.exists() else \'\'\nCHAT_TMPL.write_text(tmpl_str, encoding=\'utf-8\', newline=\'\')\nUI_CFG.write_text(new_ui_text, encoding=\'utf-8\', newline=\'\')\n\ndef _fp_args(args):\n    # 指纹跳过二进制路径和 --cache-ram（只改缓存上限不重启）。\n    out = []\n    i = 1 if args and Path(args[0]).name == \'llama-server\' else 0\n    while i < len(args):\n        a = args[i]\n        if not a:\n            i += 1\n            continue\n        if a == \'--cache-ram\':\n            i += 2\n            continue\n        out.append(a)\n        i += 1\n    return out\n\ndef _boot_fp(args, tmpl, ui):\n    blob = chr(0).join(_fp_args(args)) + chr(0) + tmpl + chr(0) + ui\n    return hashlib.sha1(blob.encode()).hexdigest()\n\nload_flags = [\'--lazy-mode\', \'off\'] if ple_in_tmpfs else [\'-lm\', \'none\', \'--lazy-mode\', \'off\']\nserver_args = [\'-m\', str(MODEL)]\nif MMPROJ:\n    server_args += [\'--mmproj\', str(MMPROJ)]\nif DRAFT and SPEC.get(\'mtp_n_max\'):\n    server_args += [\'--model-draft\', str(DRAFT), \'--spec-type\', \'draft-mtp\',\n                    \'--spec-draft-n-max\', str(SPEC[\'mtp_n_max\']),\n                    \'--spec-draft-p-min\', str(SPEC[\'mtp_p_min\'])]\nif LORA:\n    server_args += [\'--lora\', str(LORA)]\nif SPEC.get(\'extra_server_flags\'):\n    server_args += list(SPEC[\'extra_server_flags\'])\nserver_args += [\n    \'--slot-save-path\', str(SLOT_SAVE_DIR),\n    \'--device\', \'CUDA0\',\n    \'-ngl\', \'99\',\n    *load_flags,\n    \'--cache-ram\', str(PROMPT_CACHE_MIB),\n    \'-c\', str(TOTAL_CONTEXT_TOKENS),\n    \'--parallel\', str(PARALLEL_SLOTS),\n    \'--host\', \'127.0.0.1\',\n    \'--port\', str(PORT),\n    \'--alias\', MODEL_ALIAS,\n    \'--jinja\',\n    \'--chat-template-file\', str(CHAT_TMPL),\n    \'--ui-config-file\', str(UI_CFG),\n    \'--ui-mcp-proxy\',\n    \'--reasoning\', \'on\',\n]\nif SPEC.get(\'reasoning_effort_default\'):\n    server_args += [\'--reasoning-effort\', SPEC[\'reasoning_effort_default\']]\nserver_args += [\n    \'--temp\', SPEC[\'temp\'],\n    \'--top-p\', SPEC[\'top_p\'],\n    \'--top-k\', SPEC[\'top_k\'],\n    \'--min-p\', SPEC[\'min_p\'],\n    *SPEC.get(\'penalty_args\', []),\n    \'--flash-attn\', \'on\',\n    \'--cache-type-k\', \'q8_0\',\n    \'--cache-type-v\', \'q8_0\',\n    \'--batch-size\', \'4096\',\n    \'--ubatch-size\', str(UBATCH_SIZE),\n    \'--predict\', str(MAX_OUTPUT_TOKENS),\n]\nexpected_fp = _boot_fp(server_args, tmpl_str, new_ui_text)\n\ndef configured_server():\n    # 已在跑且启动指纹一致就跳过重启。指纹 = sha1(argv 去掉二进制和 --cache-ram + 模板 + webui-config)。\n    if not health():\n        return False\n    _pid, args = _server_proc()\n    if not args or _boot_fp(args, old_tmpl_on_disk, old_ui_on_disk) != expected_fp:\n        return False\n    if DEBUG_VERIFY:\n        props = requests.get(\'http://127.0.0.1:%d/props\' % PORT, headers=AUTH, timeout=10).json()\n        if props.get(\'default_generation_settings\', {}).get(\'n_ctx\') != SLOT_CONTEXT_TOKENS:\n            return False\n        if props.get(\'total_slots\') != PARALLEL_SLOTS:\n            return False\n        if props.get(\'chat_template_caps\', {}).get(\'reasoning_effort_levels\') != [\'low\', \'medium\', \'high\', \'xhigh\']:\n            return False\n    return True\n\nserver_just_started = False\nif health() and not configured_server():\n    print(\'检测到旧配置或不同模型服务：重启 llama-server ...\', flush=True)\n    if not released_reuse:\n        _stop_llama_servers()\n\nif configured_server():\n    print(f\'模型已在运行：{MODEL_ALIAS} / 单路 × {SLOT_CONTEXT_TOKENS} 上下文 / {(_cache_ram_of(_server_proc()[1]) or 0) / 1024:.0f} GiB Prompt Cache\', flush=True)\nelse:\n    missing_now = [_local_path(f).name for f in FILES if not _local_path(f).exists()]\n    if released_reuse or missing_now:\n        _stop_llama_servers()\n        RELEASE_STATE.unlink(missing_ok=True)\n        raise RuntimeError(\'需要重新启动服务，但缺少上次为扩大提示缓存而删掉的分片副本：\'\n                           + \', \'.join(sorted(set(missing_now) | released_reuse))\n                           + \'。旧服务已停，请再运行一次本格，会重新下载这些分片。\')\n    _stop_llama_servers()\n    if ple_name and not freshly_downloaded and not ple_in_tmpfs:\n        meminfo = Path(\'/proc/meminfo\').read_text()\n        cached_kb = next((int(l.split()[1]) for l in meminfo.splitlines() if l.startswith(\'Cached:\')), 0)\n        if cached_kb < 40 * 1024 * 1024:\n            print(\'预热 PLE 表所在分片到主机内存 Page Cache ...\', flush=True)\n            with (SHARD_DIR / ple_name).open(\'rb\') as f:\n                while f.read(32 * 1024 * 1024):\n                    pass\n\n    fut_cpp.result()\n    SERVER = next(DIST.rglob(\'llama-server\'))\n    env = os.environ.copy()\n    libdirs = {str(SERVER.parent)}\n    for pat in (\'libcudart.so.12\', \'libcublas.so.12\', \'libcublasLt.so.12\'):\n        libdirs.update(str(p.parent) for p in DIST.rglob(pat))\n    env[\'LD_LIBRARY_PATH\'] = \':\'.join(sorted(libdirs)) + \':\' + env.get(\'LD_LIBRARY_PATH\', \'\')\n    cmd = [str(SERVER), *server_args]\n    log = (ROOT / \'server.log\').open(\'a\', encoding=\'utf-8\', newline=\'\')\n    log.write(chr(10) + \'=== START %s ===\' % time.strftime(\'%Y-%m-%d %H:%M:%S\') + chr(10))\n    log.flush()\n    proc = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT, env=env, stdin=subprocess.DEVNULL, start_new_session=True)\n    qwen_server = proc\n    t0 = time.monotonic()\n    # LAUNCH_DETACHED=1（colab_cli launch）：进程已 start_new_session，不等本机 /health，交给后面的后台线程。\n    if os.environ.get(\'LAUNCH_DETACHED\') == \'1\':\n        print(\'LAUNCH_DETACHED: llama-server pid %s\' % proc.pid, flush=True)\n        server_just_started = True\n    else:\n        print(f\'加载 {MODEL_ALIAS} 到 GPU ...\', flush=True)\n        while time.monotonic() - t0 < 600:\n            if proc.poll() is not None:\n                raise RuntimeError((ROOT / \'server.log\').read_text(errors=\'replace\')[-3500:])\n            if health():\n                break\n            time.sleep(2 if DEBUG_VERIFY else 0.5)\n        else:\n            proc.terminate()\n            raise RuntimeError(f\'模型加载超时，请查看 {ROOT}/server.log\')\n        server_just_started = True\n\ndef _warmup_cuda():\n    # 第一次推理会捕获 CUDA graph，短提示通常要数秒到十几秒，不是零点几秒；放后台不挡就绪横幅。只发这一次 max_tokens=4。\n    requests.post(\'http://127.0.0.1:%d/v1/chat/completions\' % PORT, headers=AUTH, timeout=600,\n                  json={\'model\': MODEL_ALIAS,\n                        \'messages\': [{\'role\': \'user\', \'content\': \'请只回复“就绪”两个字。\'}],\n                        \'max_tokens\': 4, \'temperature\': 0, \'reasoning_effort\': \'low\'})\n\nif DEBUG_VERIFY:\n    if server_just_started:\n        print(\'服务就绪 %.0fs，核验四档思考能力并发送预热请求初始化 CUDA 图 ...\' % (time.monotonic() - t0), flush=True)\n        props_now = requests.get(\'http://127.0.0.1:%d/props\' % PORT, headers=AUTH, timeout=10).json()\n        levels_now = props_now.get(\'chat_template_caps\', {}).get(\'reasoning_effort_levels\', [])\n        if levels_now != [\'low\', \'medium\', \'high\', \'xhigh\']:\n            proc.terminate()\n            raise RuntimeError(f\'思考档位探测未达四档标准 (期望 low/medium/high/xhigh): {levels_now}\')\n        print(\'思考档位门禁通过：\', levels_now, \'| 默认档位：\', SPEC.get(\'reasoning_effort_default\', \'high\'), flush=True)\n        _warmup_cuda()\n    verify_hashes()\n    release_gpu_only_copies()\n    fut_ts.result()\n    status = ts(\'serve\', \'status\').stdout\n    if SERVICE_FQDN not in status:\n        raise RuntimeError(\'Service 未生效，serve status:\' + chr(10) + status)\nelse:\n    # 就绪横幅不等这些：预热、删内存盘副本、等 Tailscale。\n    def _bg_finish():\n        try:\n            if server_just_started:\n                if os.environ.get(\'LAUNCH_DETACHED\') == \'1\':\n                    # 主线程已返回。预热必须先等本机 /health，否则连接失败会跳过后面的内存盘释放和 Tailscale 收尾。\n                    deadline = time.monotonic() + 600\n                    while time.monotonic() < deadline:\n                        if proc.poll() is not None:\n                            raise RuntimeError((ROOT / \'server.log\').read_text(errors=\'replace\')[-3500:])\n                        if health():\n                            break\n                        time.sleep(0.5)\n                    else:\n                        raise RuntimeError(\'模型加载超时，请查看 %s/server.log\' % ROOT)\n                _warmup_cuda()\n            release_gpu_only_copies()\n            fut_cpp.result()\n            fut_ts.result()\n        except Exception as exc:\n            print(\'后台收尾失败：\', exc, flush=True)\n    threading.Thread(target=_bg_finish, daemon=True).start()\n    print(\'生产模式：预热、释放内存盘副本、Tailscale 在后台收尾，不挡住下面的就绪横幅。\', flush=True)\n\nprint(\'=\' * 64)\nprint(\'总启动耗时： %.1fs\' % (time.monotonic() - t_boot), flush=True)\nprint(\'网页版：     http://%s:443/\' % SERVICE_FQDN, flush=True)\nprint(\'接口：       http://%s:443/v1\' % SERVICE_FQDN, flush=True)\nprint(\'模型名：     %s\' % MODEL_ALIAS, flush=True)\nprint(\'=\' * 64)\n\n# 6. 就绪横幅 + WebUI 标签页 + 停机按钮\n# 浏览器只允许 window.open 的发起帧关闭该标签页，所以 WebUI 由本格输出帧打开，本格停机按钮先关它再释放运行时。\nfrom google.colab import output as colab_output, runtime\nfrom IPython.display import HTML, display\n\ndef _qwen_stop():\n    if globals().get(\'qwen_server\') and qwen_server.poll() is None:\n        qwen_server.terminate()\n    if Path(TS).exists():\n        try:\n            subprocess.run([TS, \'--socket=\' + TS_SOCK, \'down\'], timeout=5, capture_output=True)\n        except Exception:\n            pass\n    runtime.unassign()\n\ncolab_output.register_callback(\'qwen.stop\', _qwen_stop)\n\n_QWEN_CONSOLE_HTML = """\n<div style="margin:10px 0;padding:12px;border-radius:8px;background:#1e293b;color:#f8fafc;font-size:15px;line-height:1.6;">\n  🚀 <b>服务已就绪！</b>\n  <a id="qwen-open" href="__URL__" style="color:#38bdf8;font-weight:bold;text-decoration:underline;margin-left:8px;">👉 点击打开 WebUI 交互界面 (免密直连)</a>\n  <div style="margin-top:10px;">\n    <button id="qwen-stop" style="background:#f0ad4e;color:#111;border:0;border-radius:4px;padding:6px 14px;font-size:14px;cursor:pointer;">🛑 结束 G4 运行时 (省 CU)</button>\n  </div>\n  <div id="qwen-state" style="margin-top:8px;font-size:14px;"><b>当前状态：</b><span style="color:#5cb85c">🔓 Qwen3.8 27B RVN Heretic BF16 原精度满血，满速运行</span> — 用完点黄色按钮立即释放 GPU，WebUI 标签页会随之自动关闭</div>\n</div>\n<script>\n(() => {\n  const URL = "__URL__";\n  const stateEl = document.getElementById("qwen-state");\n  const bootMs = Date.now();\n  let win = null;\n  // 只由本次运行的 eval_js 调用；重开笔记本时旧输出里的脚本会重跑，但不会自动弹窗\n  window.qwenOpenWebui = () => {\n    win = window.open(URL, "qwen-webui");\n    if (!win) stateEl.textContent = "浏览器拦截了自动弹窗，请点上面的链接打开";\n  };\n  document.getElementById("qwen-open").addEventListener("click", (e) => {\n    e.preventDefault();\n    window.qwenOpenWebui();\n  });\n  document.getElementById("qwen-stop").addEventListener("click", (e) => {\n    e.currentTarget.disabled = true;\n    e.currentTarget.textContent = "正在释放…";\n    if (win && !win.closed) win.close();\n    const runMin = ((Date.now() - bootMs) / 60000).toFixed(1);\n    stateEl.innerHTML = "<b>正在关闭服务并销毁运行时（本次就绪后已运行 " + runMin + " 分钟，停止计费）…</b>";\n    google.colab.kernel.invokeFunction("qwen.stop", [], {}).catch(() => {});\n  });\n})();\n</script>\n""".replace(\'__URL__\', \'http://%s:443/\' % SERVICE_FQDN)\ndisplay(HTML(_QWEN_CONSOLE_HTML))\ntry:\n    colab_output.eval_js(\'window.qwenOpenWebui()\', ignore_result=True)\nexcept Exception:\n    pass\n'}, 'glm53-flash': {'alias': 'GLM-5.3-Flash-Uncensored-GSQ-RCO-Q4', 'root_dir': '/content/glm53-flash', 'min_vram_mib': 85000, 'code': '# 一键启动（由 colab_kernel.py 统一内核生成）：tmpfs 内存盘 + 32路下载 + 字节大小校验 + WebUI 本机文件桥 + Tailscale\n# 调试新模型/改配置时设 True\nDEBUG_VERIFY = False\nimport concurrent.futures, hashlib, json, os, shutil, socket, struct, subprocess, sys, threading, time\nfrom pathlib import Path\nimport requests\nfrom requests.adapters import HTTPAdapter\n\nt_boot = time.monotonic()\nSPEC = json.loads("{\\n  \\"slug\\": \\"glm53-flash\\",\\n  \\"alias\\": \\"GLM-5.3-Flash-Uncensored-GSQ-RCO-Q4\\",\\n  \\"root_dir\\": \\"/content/glm53-flash\\",\\n  \\"ts_hostname\\": \\"glm53-flash-colab\\",\\n  \\"min_vram_mib\\": 85000,\\n  \\"ram_reserve_gib\\": 65,\\n  \\"ple_shard_name\\": null,\\n  \\"template_family\\": \\"glm5_next\\",\\n  \\"temp\\": \\"1.0\\",\\n  \\"top_p\\": \\"0.95\\",\\n  \\"top_k\\": \\"0\\",\\n  \\"min_p\\": \\"0.0\\",\\n  \\"reasoning_effort_default\\": \\"high\\",\\n  \\"effort_prompts\\": {\\n    \\"xhigh\\": \\"\\u7528\\u4e2d\\u6587\\u601d\\u8003\\uff0c\\u5c3d\\u91cf\\u6df1\\u5165\\u3002\\",\\n    \\"high\\": \\"\\u7528\\u4e2d\\u6587\\u601d\\u8003\\u3002\\",\\n    \\"medium\\": \\"\\u7528\\u4e2d\\u6587\\u601d\\u8003\\uff0c\\u7b80\\u77ed\\u4e9b\\u3002\\",\\n    \\"low\\": \\"\\u7528\\u4e2d\\u6587\\u601d\\u8003\\uff0c\\u8d8a\\u77ed\\u8d8a\\u597d\\u3002\\"\\n  },\\n  \\"mtp_n_max\\": null,\\n  \\"mtp_p_min\\": null,\\n  \\"extra_server_flags\\": [\\n    \\"--n-cpu-moe\\",\\n    \\"19\\"\\n  ],\\n  \\"files\\": [\\n    {\\n      \\"role\\": \\"model\\",\\n      \\"repo\\": \\"GCSA-AiLab/GLM-5.3-Flash-Uncensored-RCO-GSQ-GGUF\\",\\n      \\"rev\\": \\"1827843362350fbc87ee6b478d9dd09e33d9fb50\\",\\n      \\"path\\": \\"Q4/GLM-5.3-Flash-Uncensored-GSQ-RCO-Q4.gguf\\",\\n      \\"local_subdir\\": \\"Q4\\",\\n      \\"size\\": 137073273280,\\n      \\"priority\\": 2\\n    },\\n    {\\n      \\"role\\": \\"mmproj\\",\\n      \\"repo\\": \\"unsloth/GLM-5.3-Flash-GGUF\\",\\n      \\"rev\\": \\"a38483c8cd5df544f53d70fb281afe97369d5ab6\\",\\n      \\"path\\": \\"mmproj-BF16.gguf\\",\\n      \\"size\\": 1164010176,\\n      \\"priority\\": 1\\n    }\\n  ]\\n}")\nUI_CFG_DATA = json.loads("{\\n  \\"systemMessage\\": \\"\\u4f60\\u662f\\u4e2d\\u6587\\u957f\\u7bc7\\u5c0f\\u8bf4\\u5171\\u521b\\u642d\\u6863\\u3002\\\\n1. \\u5f00\\u4e86 local_novel_fs \\u65f6\\uff0c\\u52a8\\u7b14\\u524d\\u5148\\u8bfb \\u5199\\u4f5c\\u8981\\u6c42.md \\u548c\\u6b63\\u6587\\u672b\\u5c3e\\uff08tail_lines=150\\uff09\\u3002\\u65b0\\u7ae0\\u6216\\u4e0b\\u534a\\u7ae0\\u7528 write_novel_file \\u8ffd\\u52a0\\uff1b\\u91cd\\u5199\\u4e0a\\u4e00\\u8f6e\\u7528 replace_last_write\\uff0c\\u91cd\\u5199\\u6574\\u7ae0\\u7528 replace_last_chapter\\uff1b\\u5c0f\\u6539\\u7528 edit_novel_file\\uff1b\\u64a4\\u56de\\u7528 undo_novel_write\\u3002\\u6ca1\\u5f00\\u5de5\\u5177\\u5c31\\u76f4\\u63a5\\u5728\\u5bf9\\u8bdd\\u91cc\\u5199\\u3002\\\\n2. \\u67e5\\u539f\\u4f5c\\u7528 grep_reference\\uff1b\\u8bbe\\u5b9a\\u51b2\\u7a81\\u4ee5\\u7528\\u6237\\u5f53\\u524d\\u6307\\u4ee4\\u4e3a\\u51c6\\u3002\\\\n3. \\u7528\\u6237\\u8bf4\\u300c\\u6536\\u5de5\\u300d\\u65f6\\uff0c\\u7528 edit_novel_file \\u628a\\u65b0\\u5b9a\\u7684\\u8bbe\\u5b9a\\u3001\\u8fdb\\u5ea6\\u548c\\u4f0f\\u7b14\\u66f4\\u65b0\\u8fdb \\u5199\\u4f5c\\u8981\\u6c42.md\\uff0c\\u53ea\\u6539\\u53d8\\u5316\\u9879\\u3002\\\\n4. \\u7528\\u6237\\u70b9\\u540d\\u65f6\\u624d\\u8bfb\\u6280\\u80fd\\uff1agrill/\\u62f7\\u95ee \\u8bfb skill:grilling\\uff0c\\u4f18\\u5316\\u63d0\\u793a\\u8bcd \\u8bfb skill:prompting\\uff0cretro/\\u590d\\u76d8 \\u8bfb skill:retro\\u3002\\",\\n  \\"showSystemMessage\\": true,\\n  \\"showMessageStats\\": true,\\n  \\"enableContinueGeneration\\": true,\\n  \\"pasteLongTextToFileLen\\": 50000,\\n  \\"excludeReasoningFromContext\\": true,\\n  \\"renderThinkingAsMarkdown\\": true,\\n  \\"renderUserContentAsMarkdown\\": true,\\n  \\"titleGenerationUseLLM\\": false,\\n  \\"titleGenerationUseFirstLine\\": true,\\n  \\"mcpServers\\": \\"[{\\\\\\"id\\\\\\": \\\\\\"local-novel-fs\\\\\\", \\\\\\"name\\\\\\": \\\\\\"local_novel_fs\\\\\\", \\\\\\"displayName\\\\\\": \\\\\\"\\u672c\\u673a\\u5c0f\\u8bf4\\u76ee\\u5f55 (Colab\\u521b\\u4f5c)\\\\\\", \\\\\\"url\\\\\\": \\\\\\"http://127.0.0.1:3426/mcp\\\\\\", \\\\\\"enabled\\\\\\": true, \\\\\\"useProxy\\\\\\": true}]\\"\\n}")\n\nROOT = Path(SPEC[\'root_dir\'])\nDIST = ROOT / \'llama-dist\'\nMODEL_ALIAS = SPEC[\'alias\']\nKEY_FILE = ROOT / \'.llama-api-key\'\nPORT = 8081\nKEY = "<YOUR_LLAMA_API_KEY>"\nFILES = SPEC[\'files\']\nCPP_TAG = \'b12032\'\nCPP_URL = \'https://github.com/ggml-org/llama.cpp/releases/download/\' + CPP_TAG\n\ndef _local_path(f):\n    sub = f.get(\'local_subdir\')\n    return (ROOT / sub / Path(f[\'path\']).name) if sub else (ROOT / Path(f[\'path\']).name)\n\nMODEL = _local_path(FILES[0])\nMMPROJ = next((_local_path(f) for f in FILES if f[\'role\'] == \'mmproj\'), None)\nDRAFT = next((_local_path(f) for f in FILES if f[\'role\'] == \'draft\'), None)\nLORA = next((_local_path(f) for f in FILES if f[\'role\'] == \'lora\'), None)\n# PLE 表所在目录跟着各模型的 local_subdir 走（IQ4_XS / IQ4_NL …），不能写死。\nSHARD_DIR = next((_local_path(f).parent for f in FILES if Path(f[\'path\']).name == SPEC.get(\'ple_shard_name\')), ROOT)\n\n# 服务把权重读进显存后，/dev/shm 里只供显存用的副本就是冗余内存；删掉后靠这份记录（绑定服务 PID）在重跑本格时跳过重新下载。\nRELEASE_STATE = ROOT / \'released-copies.json\'\nHEAD_CACHE = ROOT / \'model-head.bin\'\nTMPFS_PREFIXES = (\'/dev/shm/\', \'/var/colab/\')\nFILE_NAMES = {_local_path(f).name for f in FILES}\n\ndef _server_proc():\n    # 本模型的 llama-server：(pid, argv)；没有则 (None, [])\n    for procdir in Path(\'/proc\').iterdir():\n        if not procdir.name.isdigit():\n            continue\n        try:\n            args = (procdir / \'cmdline\').read_bytes().decode(errors=\'replace\').split(\'\\0\')\n        except (FileNotFoundError, PermissionError, ProcessLookupError):\n            continue\n        if args and Path(args[0]).name == \'llama-server\' and str(MODEL) in args:\n            return int(procdir.name), args\n    return None, []\n\ndef _server_pid():\n    return _server_proc()[0]\n\ndef _proc_identity(pid):\n    # boot_id + 进程启动时刻，防止重启后的同模型服务恰好复用旧 PID 时误用释放记录\n    try:\n        boot = Path(\'/proc/sys/kernel/random/boot_id\').read_text().strip()\n        stat = Path(\'/proc/%d/stat\' % pid).read_text()\n        return boot + \':\' + stat.rsplit(\')\', 1)[1].split()[19]\n    except (OSError, IndexError, TypeError):\n        return None\n\ndef _cache_ram_of(args):\n    try:\n        return int(args[args.index(\'--cache-ram\') + 1])\n    except (ValueError, IndexError):\n        return None\n\ndef _stop_llama_servers():\n    for procdir in Path(\'/proc\').iterdir():\n        if not procdir.name.isdigit():\n            continue\n        try:\n            args = (procdir / \'cmdline\').read_bytes().decode(errors=\'replace\').split(\'\\0\')\n            if args and Path(args[0]).name == \'llama-server\':\n                os.kill(int(procdir.name), 15)\n                for _ in range(60):\n                    stat = (procdir / \'stat\').read_text() if procdir.exists() else \'\'\n                    if not stat or stat.split(\') \', 1)[1].startswith(\'Z\'):\n                        break\n                    time.sleep(0.5)\n                else:\n                    os.kill(int(procdir.name), 9)\n                    time.sleep(2)\n        except (FileNotFoundError, ProcessLookupError, PermissionError):\n            pass\n    old_proc = globals().get(\'qwen_server\')\n    if old_proc is not None:\n        try:\n            old_proc.wait(timeout=5)\n        except Exception:\n            pass\n    deadline = time.monotonic() + 20\n    while time.monotonic() < deadline:\n        alive = []\n        for procdir in Path(\'/proc\').iterdir():\n            try:\n                if procdir.name.isdigit() and Path((procdir / \'cmdline\').read_bytes().decode(errors=\'replace\').split(\'\\0\')[0]).name == \'llama-server\':\n                    alive.append(procdir.name)\n            except (FileNotFoundError, ProcessLookupError, PermissionError):\n                pass\n        if not alive:\n            return\n        time.sleep(1)\n    raise RuntimeError(\'旧 llama-server 停不掉（PID \' + \', \'.join(alive) + \'），已中止；请用停机按钮释放运行时后重来。\')\n\ndef _mem_available():\n    for l in Path(\'/proc/meminfo\').read_text().splitlines():\n        if l.startswith(\'MemAvailable:\'):\n            return int(l.split()[1]) * 1024\n    return 0\n\nSERVICE = \'svc:qwen\'\nSERVICE_FQDN = "qwen.<YOUR_TAILNET>.ts.net"\nSERVICE_TAG = \'tag:colab\'\nTS_HOSTNAME = SPEC[\'ts_hostname\']\nTS_CLIENT_ID = "<YOUR_TS_OAUTH_CLIENT_ID>"\nTS_CLIENT_SECRET = "<YOUR_TS_OAUTH_CLIENT_SECRET>"\nTS = \'/content/tsbin/tailscale\'\nTS_SOCK = \'/var/run/tailscale/tailscaled.sock\'\nTS_STATE = Path(\'/content/tailscaled.state\')\nTS_DIR = Path(\'/content/tsstatedir\')\n\ndef sh(c):\n    subprocess.run(c, shell=True, check=True)\n\ndef health():\n    try:\n        return requests.get(\'http://127.0.0.1:%d/health\' % PORT, timeout=2).status_code == 200\n    except Exception:\n        return False\n\ngpu_info = subprocess.getoutput(\'nvidia-smi --query-gpu=name,memory.total --format=csv,noheader,nounits\').strip()\nif not gpu_info or \',\' not in gpu_info:\n    raise RuntimeError(\'未检测到 NVIDIA GPU，请在右上角切换到 G4 GPU 运行时。输出：\' + gpu_info)\ngpu_name, gpu_mib_str = [x.strip() for x in gpu_info.split(\',\', 1)]\ngpu_mib = int(gpu_mib_str)\nprint(f\'检测到 GPU: {gpu_name} ({gpu_mib / 1024:.1f} GiB VRAM)\', flush=True)\nif gpu_mib < SPEC[\'min_vram_mib\']:\n    raise RuntimeError(f\'当前 GPU 显存仅 {gpu_mib / 1024:.1f} GiB（{gpu_name}），低于要求的 {SPEC["min_vram_mib"] / 1024:.0f} GiB，请切换到 G4 高内存运行时。\')\n\nROOT.mkdir(parents=True, exist_ok=True)\nfor _d in sorted(set(_local_path(f).parent for f in FILES)):\n    _d.mkdir(parents=True, exist_ok=True)\nDIST.mkdir(parents=True, exist_ok=True)\nKEY_FILE.write_text(KEY + \'\\n\', encoding=\'utf-8\', newline=\'\')\nKEY_FILE.chmod(0o600)\nAUTH = {\'Authorization\': \'Bearer \' + KEY}\n\ndef prepare_llama_cpp():\n    if next(DIST.rglob(\'llama-server\'), None) is not None:\n        return\n    print(f\'[并行任务] 下载并解压 llama.cpp {CPP_TAG} (CUDA 12.8) ...\', flush=True)\n    sh(f\'cd {DIST} && \'\n       f\'curl -fsSL -O {CPP_URL}/llama-{CPP_TAG}-bin-ubuntu-cuda-12.8-x64.tar.gz && \'\n       f\'curl -fsSL -O {CPP_URL}/cudart-llama-{CPP_TAG}-bin-ubuntu-cuda-12.8-x64.tar.gz && \'\n       f\'tar -xzf llama-{CPP_TAG}-bin-ubuntu-cuda-12.8-x64.tar.gz && \'\n       f\'tar -xzf cudart-llama-{CPP_TAG}-bin-ubuntu-cuda-12.8-x64.tar.gz && \'\n       f\'rm -f *.tar.gz\')\n    print(f\'[并行任务] llama.cpp {CPP_TAG} 就绪\', flush=True)\n\ndef ts(*a, timeout=60):\n    return subprocess.run([TS, \'--socket=\' + TS_SOCK, *a],\n                          stdin=subprocess.DEVNULL, capture_output=True, text=True, timeout=timeout)\n\ndef ts_state():\n    try:\n        return json.loads(ts(\'status\', \'--json\').stdout).get(\'BackendState\', \'\')\n    except Exception:\n        return \'\'\n\ndef _start_mcp_bridge():\n    srv = socket.socket(socket.AF_INET, socket.SOCK_STREAM)\n    srv.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)\n    try:\n        srv.bind((\'127.0.0.1\', 3426))\n    except OSError:\n        return\n    srv.listen(32)\n    def _pipe(a, b):\n        try:\n            while (d := a.recv(65536)):\n                b.sendall(d)\n        except Exception:\n            pass\n        finally:\n            try: a.close()\n            except Exception: pass\n            try: b.close()\n            except Exception: pass\n    while True:\n        try:\n            cli, _ = srv.accept()\n            up = socket.create_connection((\'127.0.0.1\', 1055), timeout=10)\n            up.sendall(bytes([5, 1, 0]))\n            assert up.recv(2) == bytes([5, 0])\n            up.sendall(bytes([5, 1, 0, 1]) + socket.inet_aton("<YOUR_LOCAL_TAILSCALE_IP>") + struct.pack(\'>H\', 3426))\n            assert up.recv(10)[1] == 0\n            threading.Thread(target=_pipe, args=(cli, up), daemon=True).start()\n            threading.Thread(target=_pipe, args=(up, cli), daemon=True).start()\n        except Exception:\n            pass\n\ndef prepare_tailscale():\n    if not Path(TS).exists():\n        sh(\'curl -fsSL https://pkgs.tailscale.com/stable/tailscale_1.104.1_amd64.tgz | tar -xz -C /content && \'\n           \'mv /content/tailscale_1.104.1_amd64 /content/tsbin\')\n    TS_DIR.mkdir(parents=True, exist_ok=True)\n    Path(\'/var/run/tailscale\').mkdir(parents=True, exist_ok=True)\n    if ts_state() in (\'\', \'NoState\', \'Stopped\'):\n        subprocess.Popen(\n            [\'/content/tsbin/tailscaled\', \'--tun=userspace-networking\',\n             \'--socks5-server=localhost:1055\', \'--state=\' + str(TS_STATE),\n             \'--statedir=\' + str(TS_DIR), \'--socket=\' + TS_SOCK],\n            stdout=open(\'/content/tailscaled.log\', \'a\', encoding=\'utf-8\', newline=\'\'),\n            stderr=subprocess.STDOUT, stdin=subprocess.DEVNULL, start_new_session=True,\n        )\n        for _ in range(40):\n            if Path(TS_SOCK).exists():\n                break\n            time.sleep(0.5)\n        for _ in range(40):\n            if ts_state() in (\'Running\', \'NeedsLogin\'):\n                break\n            time.sleep(0.5)\n    if ts_state() != \'Running\':\n        token = requests.post(\n            \'https://api.tailscale.com/api/v2/oauth/token\',\n            data={\'client_id\': TS_CLIENT_ID, \'client_secret\': TS_CLIENT_SECRET}, timeout=30\n        ).json()[\'access_token\']\n        key = requests.post(\n            \'https://api.tailscale.com/api/v2/tailnet/-/keys\',\n            headers={\'Authorization\': \'Bearer \' + token},\n            json={\'capabilities\': {\'devices\': {\'create\': {\n                \'reusable\': False, \'ephemeral\': True, \'preauthorized\': True, \'tags\': [SERVICE_TAG]\n            }}}}, timeout=30\n        ).json()[\'key\']\n        ts(\'up\', \'--auth-key=\' + key, \'--hostname=\' + TS_HOSTNAME,\n           \'--advertise-tags=\' + SERVICE_TAG, \'--accept-dns=false\', \'--timeout=60s\', timeout=120)\n        for _ in range(120):\n            if ts_state() == \'Running\':\n                break\n            time.sleep(0.5)\n    ts(\'serve\', \'reset\')\n    ts(\'serve\', \'--bg\', \'--service=\' + SERVICE, \'--tcp=443\', \'tcp://127.0.0.1:%d\' % PORT, timeout=90)\n    ts(\'serve\', \'--bg\', \'--https=443\', \'http://127.0.0.1:%d\' % PORT, timeout=90)\n    threading.Thread(target=_start_mcp_bridge, daemon=True).start()\n    print(f\'[并行任务] Tailscale Service ({SERVICE_FQDN}) 与本机小说文件 MCP 隧道已就绪\', flush=True)\n\nbg_executor = concurrent.futures.ThreadPoolExecutor(max_workers=2)\nfut_cpp = bg_executor.submit(prepare_llama_cpp)\nfut_ts = bg_executor.submit(prepare_tailscale)\n\nsubprocess.run([\'mount\', \'-o\', \'remount,size=115G\', \'/dev/shm\'], capture_output=True)\ntmpfs_dirs = []\nfor mline in Path(\'/proc/mounts\').read_text().splitlines():\n    mparts = mline.split()\n    if len(mparts) >= 3 and mparts[2] == \'tmpfs\' and mparts[1] in (\'/dev/shm\', \'/var/colab\'):\n        tmpfs_dirs.append(Path(mparts[1]))\nmp_free = {mp: shutil.disk_usage(mp).free for mp in tmpfs_dirs}\nmeminfo_txt = Path(\'/proc/meminfo\').read_text()\navail_kb = next((int(l.split()[1]) for l in meminfo_txt.splitlines() if l.startswith(\'MemAvailable:\')), 0)\nram_budget = max(0, avail_kb * 1024 - SPEC[\'ram_reserve_gib\'] * (1024**3))\n\nreleased_reuse = set()\nif RELEASE_STATE.exists():\n    try:\n        rel_state = json.loads(RELEASE_STATE.read_text(encoding=\'utf-8\'))\n        rel_ok = (isinstance(rel_state, dict) and isinstance(rel_state.get(\'pid\'), int)\n                  and isinstance(rel_state.get(\'files\'), list)\n                  and all(isinstance(n, str) and n in FILE_NAMES for n in rel_state[\'files\']))\n    except Exception:\n        rel_state, rel_ok = {}, False\n    pid_now = _server_pid()\n    if rel_ok and pid_now == rel_state[\'pid\'] and rel_state.get(\'proc_id\') and _proc_identity(pid_now) == rel_state[\'proc_id\']:\n        released_reuse = set(rel_state[\'files\'])\n        print(f\'服务仍在运行：{len(released_reuse)} 个已释放内存副本的分片无需重新下载\', flush=True)\n    else:\n        if not rel_ok:\n            print(\'⚠️ released-copies.json 无法识别，缺失的分片会重新下载\', flush=True)\n        RELEASE_STATE.unlink(missing_ok=True)\nif not released_reuse and _server_pid() and any(not _local_path(f).exists() for f in FILES):\n    # 记录不可用、服务还在且文件已删：先停服务再补下载，避免服务占着内存时又往内存盘塞几十 GiB。\n    print(\'⚠️ 服务在运行，但分片副本已删且释放记录不可用：先停掉旧服务，再重新下载。\', flush=True)\n    _stop_llama_servers()\n\nsorted_files = sorted(FILES, key=lambda x: x.get(\'priority\', 2))\nall_targets = [\n    (\'https://huggingface.co/\' + f[\'repo\'] + \'/resolve/\' + f[\'rev\'] + \'/\' + f[\'path\'],\n     _local_path(f), f[\'size\'])\n    for f in sorted_files\n]\ndownloads = []\ntmpfs_bytes = 0\ndisk_bytes = 0\nfor orig_url, logical_path, size in all_targets:\n    if logical_path.name in released_reuse:\n        continue\n    if logical_path.exists() and logical_path.stat().st_size == size:\n        continue\n    if logical_path.is_symlink() or logical_path.exists():\n        logical_path.unlink(missing_ok=True)\n    real_path = logical_path\n    if ram_budget >= size:\n        for mp in tmpfs_dirs:\n            if mp_free.get(mp, 0) >= size + 4 * (1024**3):\n                cand = mp / SPEC[\'slug\'] / logical_path.name\n                cand.parent.mkdir(parents=True, exist_ok=True)\n                # 只看大小会把 truncate 出来的稀疏占位文件（下载中断、上次报错）当成已下载；tmpfs 的 st_blocks 是实际占用。\n                if cand.exists() and cand.stat().st_size == size and cand.stat().st_blocks * 512 >= size:\n                    logical_path.symlink_to(cand)\n                    mp_free[mp] -= size\n                    ram_budget -= size\n                    real_path = None\n                    break\n                cand.unlink(missing_ok=True)\n                with cand.open(\'wb\') as f:\n                    f.truncate(size)\n                logical_path.symlink_to(cand)\n                mp_free[mp] -= size\n                ram_budget -= size\n                tmpfs_bytes += size\n                real_path = cand\n                break\n    if real_path is None:\n        continue\n    if real_path == logical_path:\n        free_disk = shutil.disk_usage(\'/content\').free\n        if free_disk < disk_bytes + size + 6 * (1024**3):\n            raise RuntimeError(\'本地 /content 与 tmpfs 空间不足，请重建运行时。\')\n        with real_path.open(\'wb\') as f:\n            f.truncate(size)\n        disk_bytes += size\n    downloads.append((orig_url, str(real_path), size))\n\nfreshly_downloaded = bool(downloads)\nif downloads:\n    need_bytes = sum(size for _, _, size in downloads)\n    print(f\'待并发下载: {need_bytes / (1024**3):.2f} GiB ({len(downloads)} 个文件) → DDR5 内存盘 tmpfs: {tmpfs_bytes / (1024**3):.1f} GiB | 本地磁盘: {disk_bytes / (1024**3):.1f} GiB\', flush=True)\n    CHUNK = 64 * 1024 * 1024\n    hf_tok = os.environ.get(\'HF_TOKEN\', \'\').strip()\n    if not hf_tok:\n        try:\n            from google.colab import userdata\n            hf_tok = (userdata.get(\'HF_TOKEN\') or \'\').strip()\n        except Exception:\n            hf_tok = \'\'\n\n    def _resolve_cdn(item):\n        from email.utils import parsedate_to_datetime\n        from datetime import datetime, timezone\n        orig_url, real_path, size = item\n        req_hdr = {\'Range\': \'bytes=0-0\'}\n        if hf_tok:\n            req_hdr[\'Authorization\'] = \'Bearer \' + hf_tok\n        for attempt in range(5):\n            try:\n                with requests.get(orig_url, headers=req_hdr, stream=True, timeout=30) as r:\n                    if r.status_code == 429:\n                        ra = r.headers.get(\'Retry-After\', \'\')\n                        try:\n                            wait_s = float(ra)\n                        except ValueError:\n                            try:\n                                wait_s = max(1.0, (parsedate_to_datetime(ra) - datetime.now(timezone.utc)).total_seconds())\n                            except Exception:\n                                wait_s = 2 ** (attempt + 1)\n                        time.sleep(min(wait_s, 60.0))\n                        continue\n                    r.raise_for_status()\n                    if r.status_code != 206:\n                        raise RuntimeError(f\'Range probe expected 206, got {r.status_code}\')\n                    cdn_url = r.url\n                    return [(cdn_url, orig_url, real_path, start, min(start + CHUNK, size) - 1)\n                            for start in range(0, size, CHUNK)]\n            except Exception:\n                if attempt == 4:\n                    raise\n                time.sleep(2 ** (attempt + 1))\n\n    range_tasks = []\n    with concurrent.futures.ThreadPoolExecutor(max_workers=len(downloads)) as rpool:\n        for chunk_list in rpool.map(_resolve_cdn, downloads):\n            range_tasks.extend(chunk_list)\n\n    tasks_file = Path(\'/tmp/colab_kernel_dl_tasks.json\')\n    tasks_file.write_text(json.dumps({\'need_bytes\': need_bytes, \'tasks\': range_tasks}), encoding=\'utf-8\', newline=\'\')\n    dl_script = Path(\'/tmp/colab_kernel_fast_dl.py\')\n    dl_script.write_text(\'\'\'import concurrent.futures, json, multiprocessing as mp, os, sys, time\nimport requests\nfrom requests.adapters import HTTPAdapter\n\ndata = json.loads(open(sys.argv[1], encoding="utf-8").read())\nneed_bytes = data["need_bytes"]\ntasks = data["tasks"]\nN_PROC = 8\nN_THREADS = 4\n\ndone_bytes = mp.Value("q", 0)\nt_start = time.monotonic()\nlast_print = mp.Value("d", t_start)\n\ndef worker_batch(batch):\n    sess = requests.Session()\n    adapter = HTTPAdapter(pool_connections=N_THREADS + 4, pool_maxsize=N_THREADS + 4)\n    sess.mount("https://", adapter)\n    sess.mount("http://", adapter)\n\n    def fetch_one(task):\n        cdn_url, orig_url, target, start, end = task\n        expected = end - start + 1\n        for attempt in range(5):\n            url = cdn_url if attempt < 2 else orig_url\n            try:\n                with sess.get(url, headers={"Range": f"bytes={start}-{end}"}, stream=True, timeout=(15, 120)) as resp:\n                    if resp.status_code != 206:\n                        raise RuntimeError(f"HTTP {resp.status_code}")\n                    fd = os.open(target, os.O_RDWR)\n                    try:\n                        pos = start\n                        for blk in resp.iter_content(8 * 1024 * 1024):\n                            if blk:\n                                os.pwrite(fd, blk, pos)\n                                pos += len(blk)\n                    finally:\n                        os.close(fd)\n                    if pos - start != expected:\n                        raise RuntimeError(f"short read {pos - start} != {expected}")\n                with done_bytes.get_lock():\n                    done_bytes.value += expected\n                    now = time.monotonic()\n                    if now - last_print.value >= 8:\n                        rate = done_bytes.value / max(now - t_start, 0.1) / (1024**2)\n                        print(f"  并发下载进度: {done_bytes.value / need_bytes:.0%} ({done_bytes.value / (1024**3):.1f} / {need_bytes / (1024**3):.1f} GiB)，均速 {rate:.0f} MB/s", flush=True)\n                        last_print.value = now\n                return\n            except Exception as err:\n                if attempt == 4:\n                    raise RuntimeError(f"{os.path.basename(target)} [{start}-{end}] 失败: {err}") from err\n                time.sleep(1.5 * (attempt + 1))\n\n    with concurrent.futures.ThreadPoolExecutor(max_workers=N_THREADS) as tpool:\n        futs = [tpool.submit(fetch_one, t) for t in batch]\n        for f in concurrent.futures.as_completed(futs):\n            f.result()\n\nif __name__ == "__main__":\n    batches = [tasks[i::N_PROC] for i in range(N_PROC) if tasks[i::N_PROC]]\n    ctx = mp.get_context("fork")\n    with ctx.Pool(processes=len(batches)) as pool:\n        pool.map(worker_batch, batches)\n\'\'\', encoding=\'utf-8\', newline=\'\')\n    t_dl_start = time.monotonic()\n    subprocess.run([sys.executable, str(dl_script), str(tasks_file)], check=True)\n    dl_sec = max(time.monotonic() - t_dl_start, 0.1)\n    print(f\'全部权重下载完成：用时 {dl_sec:.1f}s，平均 {need_bytes / (1024**2) / dl_sec:.0f} MB/s\', flush=True)\nelse:\n    print(f\'全部 {len(FILES)} 个权重与附件已存在于内存盘/本地，跳过下载。\', flush=True)\n\ndef verify_hashes():\n    # 仅调试模式调用。下载决策时已经对过一次字节大小；这里再 stat 一遍。\n    try:\n        for item in FILES:\n            path = _local_path(item)\n            if path.name in released_reuse:\n                continue\n            actual = path.stat().st_size if path.exists() else -1\n            if actual != item[\'size\']:\n                if path.exists():\n                    path.resolve().unlink(missing_ok=True)\n                    path.unlink(missing_ok=True)\n                raise RuntimeError(\'权重存在性或字节大小校验失败，请重新运行首格：\' + path.name)\n            print(\'SIZE OK:\', path.name, item[\'size\'], flush=True)\n    except BaseException:\n        if globals().get(\'qwen_server\') is not None:\n            qwen_server.terminate()\n        raise\n\nPARALLEL_SLOTS = 1\nSLOT_CONTEXT_TOKENS = 262144\nTOTAL_CONTEXT_TOKENS = SLOT_CONTEXT_TOKENS * PARALLEL_SLOTS\nUBATCH_SIZE = 2048\nMAX_OUTPUT_TOKENS = 65536\nSLOT_SAVE_DIR = ROOT / \'slot-saves\'\nSLOT_SAVE_DIR.mkdir(parents=True, exist_ok=True)\nCHAT_TMPL = ROOT / \'chat-template-clean.jinja\'\nUI_CFG = ROOT / \'webui-config.json\'\nnew_ui_text = json.dumps(UI_CFG_DATA, ensure_ascii=False, indent=2) + \'\\n\'\nold_ui_on_disk = UI_CFG.read_text(encoding=\'utf-8\') if UI_CFG.exists() else \'\'\n\nple_name = SPEC.get(\'ple_shard_name\')\nif ple_name:\n    ple_real = str((SHARD_DIR / ple_name).resolve())\n    ple_in_tmpfs = ple_real.startswith(TMPFS_PREFIXES)\nelse:\n    ple_in_tmpfs = False\n\n# PLE 表分片在运行期由 CPU 直接 mmap 读取，必须常驻；其余落在内存盘的分片加载后可删。\nrelease_keep = {ple_name} if ple_in_tmpfs else set()\nrelease_candidates = [f for f in FILES\n                      if _local_path(f).name not in release_keep\n                      and _local_path(f).exists()\n                      and str(_local_path(f).resolve()).startswith(TMPFS_PREFIXES)]\nreleasable_bytes = sum(f[\'size\'] for f in release_candidates)\n# 只用于新启动的服务；已在运行的服务保留自己的 --cache-ram（configured_server 不比较这一项，免得为改缓存上限重启）。\n# 只拿回收内存的一半给提示缓存，另一半留作余量（mmap 模式下 CPU 张量所在分片删文件后仍占内存）。\nPROMPT_CACHE_MIB = min(65536, 16384 + releasable_bytes // (2 * 1024**2))\n\ndef release_gpu_only_copies():\n    pid, args = _server_proc()\n    todo = [f for f in release_candidates if _local_path(f).exists()]\n    if not pid or not todo or not health():\n        return\n    names = sorted({_local_path(f).name for f in todo} | released_reuse)\n    tmp_state = RELEASE_STATE.with_suffix(\'.tmp\')\n    tmp_state.write_text(json.dumps({\'pid\': pid, \'proc_id\': _proc_identity(pid), \'files\': names}), encoding=\'utf-8\', newline=\'\')\n    os.replace(tmp_state, RELEASE_STATE)\n    if DEBUG_VERIFY:\n        before = _mem_available()\n    for f in todo:\n        lp = _local_path(f)\n        lp.resolve().unlink(missing_ok=True)\n        lp.unlink(missing_ok=True)\n    expect = sum(f[\'size\'] for f in todo)\n    if DEBUG_VERIFY:\n        time.sleep(1)\n        gained = _mem_available() - before\n        cap = _cache_ram_of(args) or 0\n        print(f\'已删除 {len(todo)} 个显存专用分片的内存盘副本：可用内存 +{gained / 1024**3:.1f} GiB（预计 {expect / 1024**3:.1f} GiB）| 当前服务提示缓存上限 {cap / 1024:.0f} GiB\', flush=True)\n        if gained < expect * 0.5:\n            print(\'⚠️ 实际回收不到预计的一半：部分分片仍被服务映射，提示缓存别塞满；服务重启后才会真正归还。\', flush=True)\n    else:\n        print(f\'已删除 {len(todo)} 个显存专用分片的内存盘副本（预计 {expect / 1024**3:.1f} GiB）\', flush=True)\n\nif MODEL.exists():\n    with MODEL.open(\'rb\') as f:\n        raw_head = f.read(16 * 1024 * 1024)\n    HEAD_CACHE.write_bytes(raw_head)\nelse:\n    raw_head = HEAD_CACHE.read_bytes()\ntmpl_key = b\'tokenizer.chat_template\'\nkey_pos = raw_head.index(struct.pack(\'<Q\', len(tmpl_key)) + tmpl_key)\nval_pos = key_pos + 8 + len(tmpl_key)\nassert struct.unpack_from(\'<I\', raw_head, val_pos)[0] == 8, \'tokenizer.chat_template 不是 GGUF 字符串类型\'\ntmpl_len = struct.unpack_from(\'<Q\', raw_head, val_pos + 4)[0]\nassert val_pos + 12 + tmpl_len <= len(raw_head), \'模板超出前 16 MiB\'\ntmpl_str = raw_head[val_pos + 12:val_pos + 12 + tmpl_len].decode(\'utf-8\')\n\n# 官方模板只有 xhigh/medium/low 三档；换成四档，并按 effort_prompts 在 system 顶部注入一句中文引导。\nuniversal_effort_block = \'\'\'{%- set reasoning_instructions = \'\' %}\n{%- if enable_thinking is undefined or enable_thinking is true %}\n    {%- set resolved_reasoning_effort = reasoning_effort|default(\'@default@\') %}\n    {%- if resolved_reasoning_effort not in (\'xhigh\', \'high\', \'medium\', \'low\') %}\n        {{- raise_exception(\'Unexpected reasoning effort \' ~ reasoning_effort ~ \'. Supported types are xhigh, high, medium, and low (default @default@).\') }}\n    {%- endif %}\n    {%- if resolved_reasoning_effort == \'xhigh\' %}\n        {%- set reasoning_instructions = \'@xhigh@\' %}\n    {%- elif resolved_reasoning_effort == \'high\' %}\n        {%- set reasoning_instructions = \'@high@\' %}\n    {%- elif resolved_reasoning_effort == \'medium\' %}\n        {%- set reasoning_instructions = \'@medium@\' %}\n    {%- elif resolved_reasoning_effort == \'low\' %}\n        {%- set reasoning_instructions = \'@low@\' %}\n    {%- endif %}\n{%- endif %}\'\'\'\nfor _lvl, _txt in SPEC[\'effort_prompts\'].items():\n    universal_effort_block = universal_effort_block.replace(\'@%s@\' % _lvl, _txt)\nuniversal_effort_block = universal_effort_block.replace(\'@default@\', SPEC.get(\'reasoning_effort_default\') or \'high\')\n\nfam = SPEC.get(\'template_family\', \'qwen38_fn\')\nif fam == \'qwen38_fn\':\n    old_effort_block = \'\'\'{%- if enable_thinking is undefined or enable_thinking is true %}\n    {%- set resolved_reasoning_effort = reasoning_effort|default(\'xhigh\') %}\n    {%- if resolved_reasoning_effort not in (\'xhigh\', \'medium\', \'low\') %}\n        {{- raise_exception(\'Unexpected reasoning effort \' ~ reasoning_effort ~ \'. Supported types are xhigh (default), medium, and low.\') }}\n    {%- endif %}\n    {%- if resolved_reasoning_effort == \'xhigh\' %}\n        {%- set reasoning_instructions = \'Reasoning effort is set to xhigh. Please think carefully through the task, validate key assumptions, consider plausible alternatives, and prioritize correctness, consistency, and clarity in the final answer.\' %}\n    {%- elif resolved_reasoning_effort == \'low\' %}\n        {%- set reasoning_instructions = \'Reasoning effort is set to low. Keep your thinking brief and focused, moving directly to the conclusion without unnecessary elaboration.\' %}\n    {%- endif %}\n{%- endif %}\'\'\'\n    # Unsloth 导出的原版 Qwen3.8 模板在 resolved_reasoning_effort 下多塞了 3 行 high->xhigh 折合；先剥掉再统一替换成四档中文引导。\n    unsloth_high_shim = \'\'\'    {%- if resolved_reasoning_effort == \'high\' %}\n        {%- set resolved_reasoning_effort = \'xhigh\' %}\n    {%- endif %}\n\'\'\'\n    tmpl_str = tmpl_str.replace(unsloth_high_shim, \'\', 1)\n    assert old_effort_block in tmpl_str, \'未找到预期的 reasoning_effort 模板块\'\n    tmpl_str = tmpl_str.replace(old_effort_block, universal_effort_block, 1)\n    old_preserve_line = \'{%- if preserve_thinking is undefined or preserve_thinking is true or loop.index0 > ns.last_query_index %}\'\n    assert old_preserve_line in tmpl_str, \'未找到预期的 preserve_thinking 模板行\'\n    # 官方默认 preserve_thinking 未定义即为真，历史轮次会被套上空 <think></think>。\n    # 只保留当前工具链里 reasoning_content 非空的轮次，避免 few-shot 诱导模型秒关思考。\n    # 生成起手块不改：官方只在 enable_thinking=false 时关思考，low 照常思考。\n    tmpl_str = tmpl_str.replace(\n        old_preserve_line,\n        \'{%- if loop.index0 > ns.last_query_index and reasoning_content %}\',\n        1,\n    )\nelif fam == \'gemma4\':\n    old_think_body = \'\'\'        {{- \'<|think|>\\\\n\' -}}\n        {%- set ns.prev_message_type = \'think\' -%}\'\'\'\n    new_think_body = \'\'\'        {{- \'<|think|>\\\\n\' + (reasoning_instructions + \'\\\\n\' if reasoning_instructions else \'\') -}}\n        {%- set ns.prev_message_type = \'think\' -%}\'\'\'\n    assert old_think_body in tmpl_str, \'未找到预期的 gemma4 <|think|> 注入块\'\n    tmpl_str = tmpl_str.replace(\n        \'{%- set enable_thinking = enable_thinking | default(false) -%}\',\n        \'{%- set enable_thinking = enable_thinking | default(true) -%}\',\n        1,\n    )\n    tmpl_str = (\n        "{%- if enable_thinking is not defined %}{%- set enable_thinking = true %}{%- endif %}\\n"\n        + universal_effort_block\n        + "\\n"\n        + tmpl_str.replace(old_think_body, new_think_body, 1)\n    )\nelif fam == \'glm5_next\':\n    old_glm_head = \'\'\'{%- set effective_reasoning_effort = reasoning_effort if reasoning_effort is defined and reasoning_effort in [\'low\', \'high\'] else \'max\' -%}\n{%- if effective_reasoning_effort is not none -%}<|system|>Reasoning Effort: {{ effective_reasoning_effort | capitalize }}{%- endif -%}\n{%- set clear_thinking = clear_thinking if clear_thinking is defined else false -%}\'\'\'\n    new_glm_head = (\n        universal_effort_block\n        + "\\n{%- set effective_reasoning_effort = \'max\' if resolved_reasoning_effort == \'xhigh\' else (\'low\' if resolved_reasoning_effort == \'low\' else \'high\') -%}"\n        + "\\n{%- if effective_reasoning_effort is not none -%}<|system|>Reasoning Effort: {{ effective_reasoning_effort | capitalize }}{{ (\'\\\\n\' + reasoning_instructions) if reasoning_instructions else \'\' }}{%- endif -%}"\n        + "\\n{%- set clear_thinking = clear_thinking if clear_thinking is defined else true -%}"\n    )\n    assert old_glm_head in tmpl_str, \'未找到预期的 glm5_next reasoning_effort 模板块\'\n    tmpl_str = tmpl_str.replace(old_glm_head, new_glm_head, 1)\nelif fam == \'qwen35\':\n    old_tools_head = "{%- if tools and tools is iterable and tools is not mapping %}\\n    {{- \'<|im_start|>system\\\\n\' }}"\n    new_tools_head = (\n        universal_effort_block\n        + "\\n{%- if tools and tools is iterable and tools is not mapping %}\\n"\n        + "    {{- \'<|im_start|>system\\\\n\' }}\\n"\n        + "    {%- if reasoning_instructions %}\\n"\n        + "        {{- reasoning_instructions + \'\\\\n\\\\n\' }}\\n"\n        + "    {%- endif %}"\n    )\n    assert old_tools_head in tmpl_str, \'未找到预期的 qwen35 tools system 头部\'\n    tmpl_str = tmpl_str.replace(old_tools_head, new_tools_head, 1)\n    old_tools_sys = \'\'\'    {%- if messages[0].role == \'system\' %}\n        {%- set content = render_content(messages[0].content, false, true)|trim %}\n        {%- if content %}\n            {{- \'\\\\n\\\\n\' + content }}\n        {%- endif %}\n    {%- endif %}\n    {{- \'<|im_end|>\\\\n\' }}\n{%- else %}\n    {%- if messages[0].role == \'system\' %}\n        {%- set content = render_content(messages[0].content, false, true)|trim %}\n        {{- \'<|im_start|>system\\\\n\' + content + \'<|im_end|>\\\\n\' }}\n    {%- endif %}\n{%- endif %}\'\'\'\n    new_tools_sys = \'\'\'    {%- if messages[0].role == \'system\' %}\n        {%- set content = render_content(messages[0].content, false, true)|trim %}\n        {%- if content %}\n            {{- \'\\\\n\\\\n\' + content }}\n        {%- endif %}\n    {%- endif %}\n    {{- \'<|im_end|>\\\\n\' }}\n{%- else %}\n    {%- if messages[0].role == \'system\' %}\n        {%- set content = render_content(messages[0].content, false, true)|trim %}\n        {{- \'<|im_start|>system\\\\n\' + (reasoning_instructions + \'\\\\n\\\\n\' if reasoning_instructions else \'\') + content + \'<|im_end|>\\\\n\' }}\n    {%- elif reasoning_instructions %}\n        {{- \'<|im_start|>system\\\\n\' + reasoning_instructions + \'<|im_end|>\\\\n\' }}\n    {%- endif %}\n{%- endif %}\'\'\'\n    assert old_tools_sys in tmpl_str, \'未找到预期的 qwen35 system 分支块\'\n    tmpl_str = tmpl_str.replace(old_tools_sys, new_tools_sys, 1)\n    old_hist_think = \'{%- if loop.index0 > ns.last_query_index %}\'\n    assert old_hist_think in tmpl_str, \'未找到预期的历史思考保留条件\'\n    tmpl_str = tmpl_str.replace(old_hist_think, \'{%- if loop.index0 > ns.last_query_index and reasoning_content %}\', 1)\n    qwen_gen_think_old = \'\'\'    {%- if enable_thinking is defined and enable_thinking is false %}\n        {{- \'<think>\\\\n\\\\n</think>\\\\n\\\\n\' }}\n    {%- else %}\n        {{- \'<think>\\\\n\' }}\n    {%- endif %}\'\'\'\n    qwen_gen_think_new = \'\'\'    {%- if (enable_thinking is defined and enable_thinking is false) or (reasoning_effort is defined and reasoning_effort == \'low\') %}\n        {{- \'<think>\\\\n\\\\n</think>\\\\n\\\\n\' }}\n    {%- else %}\n        {{- \'<think>\\\\n\' }}\n    {%- endif %}\'\'\'\n    assert qwen_gen_think_old in tmpl_str, \'未找到预期的 add_generation_prompt 思考起手块\'\n    tmpl_str = tmpl_str.replace(qwen_gen_think_old, qwen_gen_think_new, 1)\nelif fam == \'nemotron\':\n    old_trunc = \'{%- set truncate_history_thinking = truncate_history_thinking if truncate_history_thinking is defined else True %}\'\n    assert old_trunc in tmpl_str, \'未找到预期的 truncate_history_thinking 行\'\n    tmpl_str = tmpl_str.replace(old_trunc, \'{%- set truncate_history_thinking = false %}\', 1)\n    old_sys_block = \'\'\'{%- if messages[0]["role"] == "system" %}\n    {%- set system_message = messages[0]["content"] %}\n    {%- set loop_messages = messages[1:] %}\n{%- else %}\n    {%- set system_message = "" %}\n    {%- set loop_messages = messages %}\n{%- endif %}\'\'\'\n    new_sys_block = (\n        universal_effort_block\n        + \'\'\'\n{%- if messages[0]["role"] == "system" %}\n    {%- set system_message = (reasoning_instructions ~ "\\\\n\\\\n" if reasoning_instructions else "") ~ render_content(messages[0]["content"]) %}\n    {%- set loop_messages = messages[1:] %}\n{%- else %}\n    {%- set system_message = reasoning_instructions %}\n    {%- set loop_messages = messages %}\n{%- endif %}\'\'\'\n    )\n    assert old_sys_block in tmpl_str, \'未找到预期的 nemotron system_message 块\'\n    tmpl_str = tmpl_str.replace(old_sys_block, new_sys_block, 1)\n    nemo_gen_think_old = \'\'\'{%- if add_generation_prompt %}\n    {%- if enable_thinking %}\n        {{- \'<|im_start|>assistant\\\\n<think>\\\\n\' }}\n    {%- else %}\n        {{- \'<|im_start|>assistant\\\\n<think></think>\' }}\n    {%- endif %}\n{%- endif %}\'\'\'\n    nemo_gen_think_new = \'\'\'{%- if add_generation_prompt %}\n    {{- \'<|im_start|>assistant\\\\n\' }}\n    {%- if (enable_thinking is defined and enable_thinking is false) or (reasoning_effort is defined and reasoning_effort == \'low\') %}\n        {{- \'<think></think>\' }}\n    {%- else %}\n        {{- \'<think>\\\\n\' }}\n    {%- endif %}\n{%- endif -%}\'\'\'\n    assert nemo_gen_think_old in tmpl_str, \'未找到预期的 nemotron add_generation_prompt 思考起手块\'\n    tmpl_str = tmpl_str.replace(nemo_gen_think_old, nemo_gen_think_new, 1)\nelse:\n    raise RuntimeError(f\'未知 template_family={fam}：必须在 colab_kernel.py 中接入 universal_effort_block 四档思考支持\')\n\nif LORA and LORA.exists():\n    # GCSA-AiLab/GLM-5.3-Flash-Abliterated-LoRA-GGUF 头部写成了 glm5next（缺连字符），原地修补为 glm5-next 并消掉 1 字节对齐填充，保持张量偏移与文件大小不变。\n    with LORA.open(\'r+b\') as _lf:\n        _lbuf = _lf.read(32768)\n        _bad_arch = b\'\\x08\\x00\\x00\\x00\\x00\\x00\\x00\\x00glm5next\'\n        _good_arch = b\'\\x09\\x00\\x00\\x00\\x00\\x00\\x00\\x00glm5-next\'\n        if _bad_arch in _lbuf:\n            _pad_idx = _lbuf.find(b\'\\x00\' * 8, 16384)\n            assert _pad_idx > 0, \'LoRA GGUF 未找到对齐填充区\'\n            _lbuf = _lbuf.replace(_bad_arch, _good_arch, 1)\n            _lbuf = _lbuf[:_pad_idx] + _lbuf[_pad_idx + 1:]\n            _lf.seek(0)\n            _lf.write(_lbuf)\nold_tmpl_on_disk = CHAT_TMPL.read_text(encoding=\'utf-8\') if CHAT_TMPL.exists() else \'\'\nCHAT_TMPL.write_text(tmpl_str, encoding=\'utf-8\', newline=\'\')\nUI_CFG.write_text(new_ui_text, encoding=\'utf-8\', newline=\'\')\n\ndef _fp_args(args):\n    # 指纹跳过二进制路径和 --cache-ram（只改缓存上限不重启）。\n    out = []\n    i = 1 if args and Path(args[0]).name == \'llama-server\' else 0\n    while i < len(args):\n        a = args[i]\n        if not a:\n            i += 1\n            continue\n        if a == \'--cache-ram\':\n            i += 2\n            continue\n        out.append(a)\n        i += 1\n    return out\n\ndef _boot_fp(args, tmpl, ui):\n    blob = chr(0).join(_fp_args(args)) + chr(0) + tmpl + chr(0) + ui\n    return hashlib.sha1(blob.encode()).hexdigest()\n\nload_flags = [\'--lazy-mode\', \'off\'] if ple_in_tmpfs else [\'-lm\', \'none\', \'--lazy-mode\', \'off\']\nserver_args = [\'-m\', str(MODEL)]\nif MMPROJ:\n    server_args += [\'--mmproj\', str(MMPROJ)]\nif DRAFT and SPEC.get(\'mtp_n_max\'):\n    server_args += [\'--model-draft\', str(DRAFT), \'--spec-type\', \'draft-mtp\',\n                    \'--spec-draft-n-max\', str(SPEC[\'mtp_n_max\']),\n                    \'--spec-draft-p-min\', str(SPEC[\'mtp_p_min\'])]\nif LORA:\n    server_args += [\'--lora\', str(LORA)]\nif SPEC.get(\'extra_server_flags\'):\n    server_args += list(SPEC[\'extra_server_flags\'])\nserver_args += [\n    \'--slot-save-path\', str(SLOT_SAVE_DIR),\n    \'--device\', \'CUDA0\',\n    \'-ngl\', \'99\',\n    *load_flags,\n    \'--cache-ram\', str(PROMPT_CACHE_MIB),\n    \'-c\', str(TOTAL_CONTEXT_TOKENS),\n    \'--parallel\', str(PARALLEL_SLOTS),\n    \'--host\', \'127.0.0.1\',\n    \'--port\', str(PORT),\n    \'--alias\', MODEL_ALIAS,\n    \'--jinja\',\n    \'--chat-template-file\', str(CHAT_TMPL),\n    \'--ui-config-file\', str(UI_CFG),\n    \'--ui-mcp-proxy\',\n    \'--reasoning\', \'on\',\n]\nif SPEC.get(\'reasoning_effort_default\'):\n    server_args += [\'--reasoning-effort\', SPEC[\'reasoning_effort_default\']]\nserver_args += [\n    \'--temp\', SPEC[\'temp\'],\n    \'--top-p\', SPEC[\'top_p\'],\n    \'--top-k\', SPEC[\'top_k\'],\n    \'--min-p\', SPEC[\'min_p\'],\n    *SPEC.get(\'penalty_args\', []),\n    \'--flash-attn\', \'on\',\n    \'--cache-type-k\', \'q8_0\',\n    \'--cache-type-v\', \'q8_0\',\n    \'--batch-size\', \'4096\',\n    \'--ubatch-size\', str(UBATCH_SIZE),\n    \'--predict\', str(MAX_OUTPUT_TOKENS),\n]\nexpected_fp = _boot_fp(server_args, tmpl_str, new_ui_text)\n\ndef configured_server():\n    # 已在跑且启动指纹一致就跳过重启。指纹 = sha1(argv 去掉二进制和 --cache-ram + 模板 + webui-config)。\n    if not health():\n        return False\n    _pid, args = _server_proc()\n    if not args or _boot_fp(args, old_tmpl_on_disk, old_ui_on_disk) != expected_fp:\n        return False\n    if DEBUG_VERIFY:\n        props = requests.get(\'http://127.0.0.1:%d/props\' % PORT, headers=AUTH, timeout=10).json()\n        if props.get(\'default_generation_settings\', {}).get(\'n_ctx\') != SLOT_CONTEXT_TOKENS:\n            return False\n        if props.get(\'total_slots\') != PARALLEL_SLOTS:\n            return False\n        if props.get(\'chat_template_caps\', {}).get(\'reasoning_effort_levels\') != [\'low\', \'medium\', \'high\', \'xhigh\']:\n            return False\n    return True\n\nserver_just_started = False\nif health() and not configured_server():\n    print(\'检测到旧配置或不同模型服务：重启 llama-server ...\', flush=True)\n    if not released_reuse:\n        _stop_llama_servers()\n\nif configured_server():\n    print(f\'模型已在运行：{MODEL_ALIAS} / 单路 × {SLOT_CONTEXT_TOKENS} 上下文 / {(_cache_ram_of(_server_proc()[1]) or 0) / 1024:.0f} GiB Prompt Cache\', flush=True)\nelse:\n    missing_now = [_local_path(f).name for f in FILES if not _local_path(f).exists()]\n    if released_reuse or missing_now:\n        _stop_llama_servers()\n        RELEASE_STATE.unlink(missing_ok=True)\n        raise RuntimeError(\'需要重新启动服务，但缺少上次为扩大提示缓存而删掉的分片副本：\'\n                           + \', \'.join(sorted(set(missing_now) | released_reuse))\n                           + \'。旧服务已停，请再运行一次本格，会重新下载这些分片。\')\n    _stop_llama_servers()\n    if ple_name and not freshly_downloaded and not ple_in_tmpfs:\n        meminfo = Path(\'/proc/meminfo\').read_text()\n        cached_kb = next((int(l.split()[1]) for l in meminfo.splitlines() if l.startswith(\'Cached:\')), 0)\n        if cached_kb < 40 * 1024 * 1024:\n            print(\'预热 PLE 表所在分片到主机内存 Page Cache ...\', flush=True)\n            with (SHARD_DIR / ple_name).open(\'rb\') as f:\n                while f.read(32 * 1024 * 1024):\n                    pass\n\n    fut_cpp.result()\n    SERVER = next(DIST.rglob(\'llama-server\'))\n    env = os.environ.copy()\n    libdirs = {str(SERVER.parent)}\n    for pat in (\'libcudart.so.12\', \'libcublas.so.12\', \'libcublasLt.so.12\'):\n        libdirs.update(str(p.parent) for p in DIST.rglob(pat))\n    env[\'LD_LIBRARY_PATH\'] = \':\'.join(sorted(libdirs)) + \':\' + env.get(\'LD_LIBRARY_PATH\', \'\')\n    cmd = [str(SERVER), *server_args]\n    log = (ROOT / \'server.log\').open(\'a\', encoding=\'utf-8\', newline=\'\')\n    log.write(chr(10) + \'=== START %s ===\' % time.strftime(\'%Y-%m-%d %H:%M:%S\') + chr(10))\n    log.flush()\n    proc = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT, env=env, stdin=subprocess.DEVNULL, start_new_session=True)\n    qwen_server = proc\n    t0 = time.monotonic()\n    # LAUNCH_DETACHED=1（colab_cli launch）：进程已 start_new_session，不等本机 /health，交给后面的后台线程。\n    if os.environ.get(\'LAUNCH_DETACHED\') == \'1\':\n        print(\'LAUNCH_DETACHED: llama-server pid %s\' % proc.pid, flush=True)\n        server_just_started = True\n    else:\n        print(f\'加载 {MODEL_ALIAS} 到 GPU ...\', flush=True)\n        while time.monotonic() - t0 < 600:\n            if proc.poll() is not None:\n                raise RuntimeError((ROOT / \'server.log\').read_text(errors=\'replace\')[-3500:])\n            if health():\n                break\n            time.sleep(2 if DEBUG_VERIFY else 0.5)\n        else:\n            proc.terminate()\n            raise RuntimeError(f\'模型加载超时，请查看 {ROOT}/server.log\')\n        server_just_started = True\n\ndef _warmup_cuda():\n    # 第一次推理会捕获 CUDA graph，短提示通常要数秒到十几秒，不是零点几秒；放后台不挡就绪横幅。只发这一次 max_tokens=4。\n    requests.post(\'http://127.0.0.1:%d/v1/chat/completions\' % PORT, headers=AUTH, timeout=600,\n                  json={\'model\': MODEL_ALIAS,\n                        \'messages\': [{\'role\': \'user\', \'content\': \'请只回复“就绪”两个字。\'}],\n                        \'max_tokens\': 4, \'temperature\': 0, \'reasoning_effort\': \'low\'})\n\nif DEBUG_VERIFY:\n    if server_just_started:\n        print(\'服务就绪 %.0fs，核验四档思考能力并发送预热请求初始化 CUDA 图 ...\' % (time.monotonic() - t0), flush=True)\n        props_now = requests.get(\'http://127.0.0.1:%d/props\' % PORT, headers=AUTH, timeout=10).json()\n        levels_now = props_now.get(\'chat_template_caps\', {}).get(\'reasoning_effort_levels\', [])\n        if levels_now != [\'low\', \'medium\', \'high\', \'xhigh\']:\n            proc.terminate()\n            raise RuntimeError(f\'思考档位探测未达四档标准 (期望 low/medium/high/xhigh): {levels_now}\')\n        print(\'思考档位门禁通过：\', levels_now, \'| 默认档位：\', SPEC.get(\'reasoning_effort_default\', \'high\'), flush=True)\n        _warmup_cuda()\n    verify_hashes()\n    release_gpu_only_copies()\n    fut_ts.result()\n    status = ts(\'serve\', \'status\').stdout\n    if SERVICE_FQDN not in status:\n        raise RuntimeError(\'Service 未生效，serve status:\' + chr(10) + status)\nelse:\n    # 就绪横幅不等这些：预热、删内存盘副本、等 Tailscale。\n    def _bg_finish():\n        try:\n            if server_just_started:\n                if os.environ.get(\'LAUNCH_DETACHED\') == \'1\':\n                    # 主线程已返回。预热必须先等本机 /health，否则连接失败会跳过后面的内存盘释放和 Tailscale 收尾。\n                    deadline = time.monotonic() + 600\n                    while time.monotonic() < deadline:\n                        if proc.poll() is not None:\n                            raise RuntimeError((ROOT / \'server.log\').read_text(errors=\'replace\')[-3500:])\n                        if health():\n                            break\n                        time.sleep(0.5)\n                    else:\n                        raise RuntimeError(\'模型加载超时，请查看 %s/server.log\' % ROOT)\n                _warmup_cuda()\n            release_gpu_only_copies()\n            fut_cpp.result()\n            fut_ts.result()\n        except Exception as exc:\n            print(\'后台收尾失败：\', exc, flush=True)\n    threading.Thread(target=_bg_finish, daemon=True).start()\n    print(\'生产模式：预热、释放内存盘副本、Tailscale 在后台收尾，不挡住下面的就绪横幅。\', flush=True)\n\nprint(\'=\' * 64)\nprint(\'总启动耗时： %.1fs\' % (time.monotonic() - t_boot), flush=True)\nprint(\'网页版：     http://%s:443/\' % SERVICE_FQDN, flush=True)\nprint(\'接口：       http://%s:443/v1\' % SERVICE_FQDN, flush=True)\nprint(\'模型名：     %s\' % MODEL_ALIAS, flush=True)\nprint(\'=\' * 64)\n\n# 6. 就绪横幅 + WebUI 标签页 + 停机按钮\n# 浏览器只允许 window.open 的发起帧关闭该标签页，所以 WebUI 由本格输出帧打开，本格停机按钮先关它再释放运行时。\nfrom google.colab import output as colab_output, runtime\nfrom IPython.display import HTML, display\n\ndef _qwen_stop():\n    if globals().get(\'qwen_server\') and qwen_server.poll() is None:\n        qwen_server.terminate()\n    if Path(TS).exists():\n        try:\n            subprocess.run([TS, \'--socket=\' + TS_SOCK, \'down\'], timeout=5, capture_output=True)\n        except Exception:\n            pass\n    runtime.unassign()\n\ncolab_output.register_callback(\'qwen.stop\', _qwen_stop)\n\n_QWEN_CONSOLE_HTML = """\n<div style="margin:10px 0;padding:12px;border-radius:8px;background:#1e293b;color:#f8fafc;font-size:15px;line-height:1.6;">\n  🚀 <b>服务已就绪！</b>\n  <a id="qwen-open" href="__URL__" style="color:#38bdf8;font-weight:bold;text-decoration:underline;margin-left:8px;">👉 点击打开 WebUI 交互界面 (免密直连)</a>\n  <div style="margin-top:10px;">\n    <button id="qwen-stop" style="background:#f0ad4e;color:#111;border:0;border-radius:4px;padding:6px 14px;font-size:14px;cursor:pointer;">🛑 结束 G4 运行时 (省 CU)</button>\n  </div>\n  <div id="qwen-state" style="margin-top:8px;font-size:14px;"><b>当前状态：</b><span style="color:#5cb85c">🔓 GLM-5.3-Flash 320B-A18B GSQ-RCO-Q4（合入 LoRA-v2），满速运行</span> — 用完点黄色按钮立即释放 GPU，WebUI 标签页会随之自动关闭</div>\n</div>\n<script>\n(() => {\n  const URL = "__URL__";\n  const stateEl = document.getElementById("qwen-state");\n  const bootMs = Date.now();\n  let win = null;\n  // 只由本次运行的 eval_js 调用；重开笔记本时旧输出里的脚本会重跑，但不会自动弹窗\n  window.qwenOpenWebui = () => {\n    win = window.open(URL, "qwen-webui");\n    if (!win) stateEl.textContent = "浏览器拦截了自动弹窗，请点上面的链接打开";\n  };\n  document.getElementById("qwen-open").addEventListener("click", (e) => {\n    e.preventDefault();\n    window.qwenOpenWebui();\n  });\n  document.getElementById("qwen-stop").addEventListener("click", (e) => {\n    e.currentTarget.disabled = true;\n    e.currentTarget.textContent = "正在释放…";\n    if (win && !win.closed) win.close();\n    const runMin = ((Date.now() - bootMs) / 60000).toFixed(1);\n    stateEl.innerHTML = "<b>正在关闭服务并销毁运行时（本次就绪后已运行 " + runMin + " 分钟，停止计费）…</b>";\n    google.colab.kernel.invokeFunction("qwen.stop", [], {}).catch(() => {});\n  });\n})();\n</script>\n""".replace(\'__URL__\', \'http://%s:443/\' % SERVICE_FQDN)\ndisplay(HTML(_QWEN_CONSOLE_HTML))\ntry:\n    colab_output.eval_js(\'window.qwenOpenWebui()\', ignore_result=True)\nexcept Exception:\n    pass\n'}}

def _llama_pids():
    pids = []
    for procdir in Path('/proc').iterdir():
        if not procdir.name.isdigit():
            continue
        try:
            args = (procdir / 'cmdline').read_bytes().decode(errors='replace').split('\0')
        except (FileNotFoundError, PermissionError, ProcessLookupError):
            continue
        if args and Path(args[0]).name == 'llama-server':
            pids.append(int(procdir.name))
    return pids

def _switch():
    if TARGET not in SWITCH:
        raise ValueError('未知模型 %s，可选：%s' % (TARGET, ', '.join(SWITCH)))
    info = SWITCH[TARGET]
    try:
        served = [m['id'] for m in requests.get('http://127.0.0.1:8081/v1/models', timeout=5,
                  headers={'Authorization': 'Bearer ' + globals().get('KEY', '')}).json()['data']]
    except Exception:
        served = []
    if globals().get('SPEC', {}).get('slug') == TARGET and info['alias'] in served:
        print('%s 已在运行且健康，无需切换。' % info['alias'])
        return False
    gpu_mib = int(subprocess.getoutput('nvidia-smi --query-gpu=memory.total --format=csv,noheader,nounits').split()[0])
    if gpu_mib < info['min_vram_mib']:
        raise RuntimeError('当前显卡 %.0f GiB，跑不了 %s（至少 %.0f GiB），未做任何改动。'
                           % (gpu_mib / 1024, info['alias'], info['min_vram_mib'] / 1024))
    for pid in _llama_pids():
        try:
            os.kill(pid, signal.SIGTERM)
        except ProcessLookupError:
            pass
    old = globals().get('qwen_server')
    if old is not None:
        try:
            old.wait(timeout=60)
        except Exception:
            pass
    deadline = time.monotonic() + 60
    while _llama_pids() and time.monotonic() < deadline:
        time.sleep(1)
    for pid in _llama_pids():
        try:
            os.kill(pid, signal.SIGKILL)
        except ProcessLookupError:
            pass
    deadline = time.monotonic() + 20
    while _llama_pids() and time.monotonic() < deadline:
        time.sleep(1)
    if _llama_pids():
        raise RuntimeError('旧 llama-server 杀不掉，切换中止（文件未删）。请用停机按钮释放运行时后重来。')
    globals()['qwen_server'] = None
    target_root = Path(info['root_dir'])
    target_root.mkdir(parents=True, exist_ok=True)
    for slug, other in SWITCH.items():
        if slug == TARGET:
            continue
        for base in ('/dev/shm', '/var/colab'):
            shutil.rmtree(Path(base) / slug, ignore_errors=True)
        old_root = Path(other['root_dir'])
        if (old_root / 'llama-dist').is_dir() and not (target_root / 'llama-dist').exists():
            shutil.move(str(old_root / 'llama-dist'), str(target_root / 'llama-dist'))
        shutil.rmtree(old_root, ignore_errors=True)
    time.sleep(3)
    print('已停掉旧模型并清空其内存盘与本地文件，开始启动 %s ...' % info['alias'], flush=True)
    return True

if _switch():
    exec(compile(SWITCH[TARGET]['code'], '<switch:%s>' % TARGET, 'exec'), globals())
    print('切换完成：客户端里把模型名换成 %s；上方“查看配置”和“调试用全链路验收”两格会自动跟着新模型。' % SWITCH[TARGET]['alias'], flush=True)


In [ ]:
# 长程写作任务防断连护航（前台保活 + 45分钟无请求自动停机省 CU）或立即点击按钮释放 G4 GPU
import ipywidgets as widgets
import os, subprocess, threading, time
import requests
from google.colab import runtime
from IPython.display import display
from html import escape

IDLE_LIMIT_MIN = 45
stop_button = widgets.Button(description='🛑 立即结束 G4 GPU 运行时', button_style='warning')
guard_button = widgets.Button(description='🛡️ 启动长程防断连护航 (45m空闲自停)', button_style='info')
stop_status = widgets.HTML(value='长程写作/共创时可点蓝色按钮保持前台心跳防 Colab 90m 断连（连续 45 分钟无推理请求会自动释放 G4 省 CU）；用完点黄色按钮立即释放。')
_guard_running = False

def stop_runtime(_=None):
    global _guard_running
    _guard_running = False
    stop_button.disabled = True
    guard_button.disabled = True
    stop_button.description = '已请求释放'
    stop_status.value = ('正在关闭服务并释放 G4 GPU。此提示断开后不再刷新；'
                         '请在“代码执行程序 → 管理会话”确认没有活动会话。')
    for name in ('qwen_server', 'tunnel_proc'):
        process = globals().get(name)
        if process is not None and process.poll() is None:
            process.terminate()
    tsctl = '/content/tsbin/tailscale'
    if os.path.exists(tsctl):
        try:
            subprocess.run([tsctl, '--socket=/var/run/tailscale/tailscaled.sock', 'down'],
                           timeout=5, capture_output=True)
        except Exception as error:
            print('节点下线未确认，将直接释放运行时：', type(error).__name__)
    try:
        runtime.unassign()
    except Exception as error:
        stop_button.disabled = False
        stop_button.description = '重试释放 GPU'
        stop_status.value = ('<b>释放请求失败：</b>' + escape(str(error)) +
                             '<br>请重试，或使用“代码执行程序 → 断开连接并删除运行时”。')

def start_guard(_):
    global _guard_running
    if _guard_running:
        return
    _guard_running = True
    guard_button.disabled = True
    guard_button.description = '🛡️ 长程护航运行中...'
    def _loop():
        last_active = time.monotonic()
        last_tokens = -1
        while _guard_running:
            try:
                slots = requests.get('http://127.0.0.1:8081/slots', timeout=5).json()
                busy = any(s.get('is_processing') for s in slots)
                cur_tokens = sum(int(s.get('n_past', 0)) for s in slots)
                if busy or (last_tokens != -1 and cur_tokens != last_tokens):
                    last_active = time.monotonic()
                last_tokens = cur_tokens
                idle_min = (time.monotonic() - last_active) / 60.0
                if idle_min >= IDLE_LIMIT_MIN:
                    stop_status.value = f'<b>已连续空闲 {idle_min:.0f} 分钟，触发自动停机释放 G4 GPU 省 CU...</b>'
                    stop_runtime()
                    return
                stop_status.value = f'<b>🛡️ 长程护航生效中：</b>槽位状态={"🔥 推理中" if busy else "🟢 待命"} | 当前缓存={cur_tokens} tokens | 已空闲={idle_min:.1f}/{IDLE_LIMIT_MIN} 分钟'
            except Exception as e:
                stop_status.value = f'<b>护航探测异常：</b>{escape(str(e))}'
            time.sleep(30)
    threading.Thread(target=_loop, daemon=True).start()

stop_button.on_click(stop_runtime)
guard_button.on_click(start_guard)
display(widgets.VBox([widgets.HBox([guard_button, stop_button]), stop_status]))


**结束状态与用量**

释放 G4 GPU 后，右上角显示“重新连接”表示已断开；进入 **代码执行程序 → 管理会话**，显示 **没有正在进行的会话** 即表示 G4 已释放。

计费按运行时挂着的时间算，空闲一样扣。Colab G4（RTX PRO 6000 Blackwell 96GB）约 **8.71–10.05 计算单元/小时**。

本地提示：`python scripts/colab_meter.py status` 实时查 CU 余额与燃烧率；`python scripts/colab_meter.py stop --all` 强杀全部残留实例并打印双轨 CU 账单。
